<div style="background: linear-gradient(135deg, #0f0c29, #302b63, #24243e); border-radius: 16px; padding: 36px 40px; margin-bottom: 8px;">
  <h1 style="color: #e0aaff; font-size: 2.4em; font-weight: 800; margin: 0 0 10px 0; letter-spacing: 1px;">
    ⚡ iTransformer · BTC/USDT 1-Minute Forecasting
  </h1>
  <p style="color: #c77dff; font-size: 1.15em; margin: 0 0 18px 0; font-weight: 500;">
    Inverted Transformer · multi-granularity exogenous fusion · end-to-end pipeline
  </p>
  <hr style="border: none; border-top: 1px solid #7b2d8b; margin: 16px 0;">
  <p style="color: #9d8cff; font-size: 0.97em; margin: 0 0 12px 0;">
    A single self-contained notebook implementing the full specification in <code>CLAUDE.md</code>:
    load → validate → align → engineer → window → split → train → evaluate → backtest → export.
    The endogenous target is <strong>BTC/USDT 1-minute log returns</strong>, conditioned on three
    exogenous sources sampled at <strong>three different frequencies</strong>:
  </p>
  <table style="color: #c8b6ff; font-size: 0.93em; border-collapse: collapse; width: 100%;">
    <tr style="color: #e0aaff; text-align: left;">
      <th style="padding: 4px 10px 4px 0;">Role</th><th style="padding: 4px 10px 4px 0;">Series</th>
      <th style="padding: 4px 10px 4px 0;">Native granularity</th>
    </tr>
    <tr><td style="padding: 3px 10px 3px 0;">Endogenous target</td><td>BTC/USDT OHLCV (Binance)</td><td>1 minute</td></tr>
    <tr><td style="padding: 3px 10px 3px 0;">Exogenous · fast</td><td>XAU/USD Gold OHLCV</td><td>1 minute (with closures)</td></tr>
    <tr><td style="padding: 3px 10px 3px 0;">Exogenous · slow</td><td>FED Broad Trade-Weighted USD Index</td><td>1 day</td></tr>
    <tr><td style="padding: 3px 10px 3px 0;">Exogenous · very slow</td><td>US Macroeconomics (31 indicators)</td><td>1 month</td></tr>
  </table>
  <hr style="border: none; border-top: 1px solid #7b2d8b; margin: 16px 0;">
  <p style="color: #9d8cff; font-size: 0.93em; margin: 0;">
    <strong>Non-negotiables enforced throughout:</strong> PyTorch only · iTransformer is the shipped
    artifact · <strong>zero look-ahead leakage</strong> (every join is a backward as-of join, every
    rolling window is trailing and right-closed, every statistic is fitted on the training split
    alone) · multi-granularity fusion treated as the central engineering problem, not an afterthought. 🫡
  </p>
</div>

<div style="background: linear-gradient(90deg, #10002b, #240046); border-left: 4px solid #c77dff; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e0aaff; margin: 0 0 10px 0;">🗂️ How to run this on Kaggle</h2>
  <p style="color: #9d8cff; margin: 0 0 8px 0;">Written for a <strong>Kaggle T4 ×2</strong> session. Kaggle's limits shape the whole workflow: <strong>12 h per session</strong>, <strong>30 GPU-hours per week</strong>, <strong>20 GB</strong> of auto-saved <code>/kaggle/working</code>, ~29 GB RAM, 4 CPU cores, and a <strong>20-minute idle timeout</strong> while editing interactively. <code>docs/KAGGLE_GUIDE.md</code> covers all of it; the short version:</p>  <ul style="color: #9d8cff; margin: 0; padding-left: 20px;">
    <li><strong>Upload the data.</strong> One Kaggle Dataset holding the 12 files from <code>data/raw/</code> (<code>btc_usdt_binance_2018.parquet</code> … <code>2026.parquet</code>, <code>xauusd_2018_2026.parquet</code>, <code>US_Dollar_Index_2018_2026.parquet</code>, <code>fed_economic_data_2018_2026.parquet</code>). The config cell <em>searches</em> <code>/kaggle/input</code> for it, so the dataset slug does not have to match anything.</li>
    <li><strong>Settings → Accelerator → GPU T4 ×2</strong>. <strong>Internet: Off</strong> keeps the run reproducible — turn it on for one run only if you want the ONNX parity check, which needs <code>onnxruntime</code>.</li>
    <li><strong>First run: <code>PROFILE = 'smoke'</code>.</strong> ~15 min, six months of data, small model. It proves every cell executes and every sanity gate passes. Its numbers are a plumbing test, not a model test.</li>
    <li><strong>Then <code>PROFILE = 'full'</code>, across several sessions.</strong> The full profile does <em>not</em> fit in one 12 h session: the main model alone is most of it, with six trained baselines, five ablations and an optional walk-forward behind it. Use the stage switches (<code>run_baselines</code>, <code>run_ablation</code>, <code>run_walkforward</code>) to run one stage per session.</li>
    <li><strong>Resuming is built in.</strong> Training stops itself <code>reserve_hours</code> before <code>session_budget_hours</code>, so the checkpoint is safely written instead of being killed at the wall. Save the version, attach that output as an input dataset to the next session, set <code>KAGGLE_RESUME_DIR</code> to its <code>checkpoints/&lt;run_id&gt;</code> folder, and every stage picks up from its last completed epoch.</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #10002b, #240046); border-left: 4px solid #c77dff; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e0aaff; margin: 0 0 10px 0;">🧭 Notebook map</h2>
  <ul style="color: #9d8cff; margin: 0; padding-left: 20px;">
    <li><strong>§1 Setup</strong> — imports, device/AMP detection, seeding, plot theme</li>
    <li><strong>§2 Configuration</strong> — one <code>CFG</code> object; profiles; publication-lag table</li>
    <li><strong>§3 Loading</strong> — typed loaders, string→float casting, UTC normalisation</li>
    <li><strong>§4 Validation</strong> — gap census, OHLC sanity, extreme-return census</li>
    <li><strong>§5 Gold timezone</strong> — resolved empirically, not assumed</li>
    <li><strong>§6 Alignment</strong> — publication lags, backward as-of joins, staleness features</li>
    <li><strong>§7 Features</strong> — price · volume · cross-asset · macro · temporal blocks</li>
    <li><strong>§8 Hygiene</strong> — train-only winsorisation, collinearity pruning, scaler, manifest</li>
    <li><strong>§9 Splits</strong> — purged + embargoed chronological splits</li>
    <li><strong>§10 Dataset</strong> — windowing with a precomputed validity mask</li>
    <li><strong>§11 Model</strong> — iTransformer, and the baselines it must beat</li>
    <li><strong>§12 Sanity gates</strong> — five hard gates that must pass before training counts</li>
    <li><strong>§13 Training</strong> — DataParallel + AMP + resumable checkpointing</li>
    <li><strong>§14 Evaluation</strong> — metrics, Diebold–Mariano, cost-aware backtest, diagnostics</li>
    <li><strong>§15 Ablation &amp; walk-forward</strong> — which exogenous blocks actually earn their place</li>
    <li><strong>§16 Export</strong> — state_dict · TorchScript · ONNX, with a verified parity check</li>
    <li><strong>§17 Report</strong> — results table and every known limitation, stated plainly</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #10002b, #240046); border-left: 4px solid #c77dff; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e0aaff; margin: 0 0 10px 0;">📦 §1 · Setup &amp; Imports</h2>
  <p style="color: #9d8cff; margin: 0 0 8px 0;">PyTorch is the <strong>only</strong> deep-learning framework here — no TensorFlow, Keras, or JAX. Polars does the heavy dataframe work over the 4.4M-row minute table; NumPy holds the final feature matrix.</p>
</div>

In [ ]:
import copy
import gc
import hashlib
import json
import math
import os
import random
import subprocess
import sys
import time
import warnings
from dataclasses import asdict, dataclass, field
from datetime import datetime, timedelta, timezone
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="torch")


def _ensure(pkg: str, import_name: str | None = None) -> None:
    """Install a package only if it is genuinely missing (Kaggle ships most of these)."""
    name = import_name or pkg
    try:
        __import__(name)
    except ImportError:
        print(f"[setup] installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)


_ensure("polars")

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

print(f"python  {sys.version.split()[0]}")
print(f"torch   {torch.__version__}")
print(f"polars  {pl.__version__}")
print(f"numpy   {np.__version__}")


def peak_rss_gb() -> float:
    """Peak resident set size of this process, in GiB. Stdlib only, no extra dependency.

    Preprocessing is the memory ceiling of this project and the local machine has *less* RAM
    than Kaggle does, so the number has to be measured rather than assumed.
    """
    try:
        if sys.platform == "win32":
            import ctypes
            from ctypes import wintypes

            class _PMC(ctypes.Structure):
                _fields_ = [("cb", wintypes.DWORD), ("PageFaultCount", wintypes.DWORD),
                            ("PeakWorkingSetSize", ctypes.c_size_t),
                            ("WorkingSetSize", ctypes.c_size_t),
                            ("QuotaPeakPagedPoolUsage", ctypes.c_size_t),
                            ("QuotaPagedPoolUsage", ctypes.c_size_t),
                            ("QuotaPeakNonPagedPoolUsage", ctypes.c_size_t),
                            ("QuotaNonPagedPoolUsage", ctypes.c_size_t),
                            ("PagefileUsage", ctypes.c_size_t),
                            ("PeakPagefileUsage", ctypes.c_size_t)]

            pmc = _PMC()
            pmc.cb = ctypes.sizeof(_PMC)
            # c_void_p(-1) is the current-process pseudo-handle. Calling GetCurrentProcess()
            # through ctypes returns c_int, which truncates -1 to 32 bits on a 64-bit build
            # and makes the call fail silently, reporting 0 GB.
            ok = ctypes.windll.psapi.GetProcessMemoryInfo(
                ctypes.c_void_p(-1), ctypes.byref(pmc), pmc.cb)
            return pmc.PeakWorkingSetSize / 1024**3 if ok else float("nan")
        with open("/proc/self/status") as fh:                    # Linux, including Kaggle
            for line in fh:
                if line.startswith("VmHWM:"):
                    return int(line.split()[1]) / 1024**2
    except Exception:
        pass
    return float("nan")

<div style="background: linear-gradient(90deg, #10002b, #240046); border-left: 4px solid #c77dff; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e0aaff; margin: 0 0 10px 0;">🖥️ Device, precision, and reproducibility</h2>
  <p style="color: #9d8cff; margin: 0 0 8px 0;">Three things are decided here, and each one has a wrong answer that silently costs you:</p>  <ul style="color: #9d8cff; margin: 0; padding-left: 20px;">
    <li><strong>Device</strong> — never hard-coded to <code>.cuda()</code>; the notebook runs on CPU too (slowly), which matters for debugging.</li>
    <li><strong>AMP dtype</strong> — <code>bfloat16</code> needs compute capability ≥ 8.0. <strong>Kaggle's T4 is Turing (sm_75) and does not support it</strong>, so the notebook detects this and falls back to <code>float16</code> + <code>GradScaler</code>. Forcing bf16 on a T4 is emulated and slower than fp32.</li>
    <li><strong>Seeding</strong> — <code>random</code>, <code>numpy</code>, <code>torch</code>, CUDA, <code>PYTHONHASHSEED</code>, and DataLoader workers all seeded. <code>cudnn.deterministic</code> is switched on for the final run and off while exploring, because determinism costs throughput.</li>
  </ul>
</div>

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU = torch.cuda.device_count()

# bfloat16 needs sm_80+. Kaggle's T4 is sm_75 -> fp16 + GradScaler instead.
if DEVICE.type == "cuda":
    # torch.cuda.is_bf16_supported() defaults to including_emulation=True and returns
    # True on Turing (sm_75), because a bf16 *tensor* can be allocated there even though
    # there is no bf16 tensor-core path. Trusting it puts Kaggle's T4 on EMULATED bf16,
    # which is slower than plain fp32. Ask the hardware directly instead.
    BF16_OK = torch.cuda.get_device_capability(0)[0] >= 8
    AMP_DTYPE = torch.bfloat16 if BF16_OK else torch.float16
    USE_SCALER = not BF16_OK              # GradScaler is only needed for fp16
    for i in range(N_GPU):
        prop = torch.cuda.get_device_properties(i)
        print(f"gpu[{i}] {prop.name}  sm_{prop.major}{prop.minor}  {prop.total_memory / 1024**3:.1f} GB")
else:
    AMP_DTYPE, USE_SCALER = torch.float32, False

AMP_ENABLED = DEVICE.type == "cuda"
print(f"\ndevice      {DEVICE}  (n_gpu={N_GPU})")
print(f"amp dtype   {AMP_DTYPE}   grad_scaler={USE_SCALER}")


def set_seed(seed: int, deterministic: bool = False) -> None:
    """Seed every source of randomness the training loop touches."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic


def seed_worker(worker_id: int) -> None:
    """DataLoader worker_init_fn - workers inherit a derived, reproducible seed."""
    s = torch.initial_seed() % 2**32
    np.random.seed(s)
    random.seed(s)

<div style="background: linear-gradient(90deg, #10002b, #240046); border-left: 4px solid #c77dff; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e0aaff; margin: 0 0 10px 0;">🎨 Plot theme</h2>
  <p style="color: #9d8cff; margin: 0 0 8px 0;">One palette, defined once, used everywhere. The five categorical hues are a <strong>fixed order that is never cycled</strong> and were validated for colour-vision deficiency against this dark surface — worst adjacent pair ΔE 15.9 (deutan), well above the ΔE 8 target. Magnitude uses a single-hue ramp, polarity uses a two-hue diverging ramp with a neutral grey midpoint, and no chart in this notebook uses two y-axes.</p>
</div>

In [ ]:
SURFACE = "#1a1a19"
INK = "#e8e6f0"
INK_MUTED = "#9d97b5"
GRID = "#2e2c3d"

# Fixed categorical order - assigned by entity, never by rank, never cycled.
CAT = ["#2f9e68", "#a855f7", "#cf7400", "#1e9ec4", "#e94560"]
SEQ = ["#2b1a3d", "#4c2b6b", "#7040a0", "#9a5fd0", "#c391f0"]   # magnitude: one hue, light->dark
DIV = ["#1e9ec4", "#7fa8b8", "#8a8a8a", "#c78591", "#e94560"]   # polarity: two hues + grey midpoint

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "axes.edgecolor": GRID, "grid.color": GRID, "grid.linewidth": 0.6, "grid.alpha": 0.6,
    "axes.grid": True, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "legend.labelcolor": INK,
    "lines.linewidth": 2.0, "lines.markersize": 5,
    "figure.dpi": 110, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
})


def finish(ax, title: str = "", xlabel: str = "", ylabel: str = "", legend: bool = False):
    """Consistent titling. Legend appears whenever 2+ series share an axis."""
    if title:
        ax.set_title(title, loc="left", pad=12)
    ax.set_xlabel(xlabel, color=INK_MUTED)
    ax.set_ylabel(ylabel, color=INK_MUTED)
    if legend:
        ax.legend(loc="best", fontsize=9)
    return ax

<div style="background: linear-gradient(90deg, #1a1a2e, #16213e); border-left: 4px solid #e94560; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #f5a623; margin: 0 0 10px 0;">🎛️ §2 · Configuration</h2>
  <p style="color: #ffd6a5; margin: 0 0 8px 0;">Every knob lives in this one cell. No magic numbers appear later in the notebook — if a number matters, it is named here and referenced by name.</p>  <ul style="color: #ffd6a5; margin: 0; padding-left: 20px;">
    <li><strong><code>PROFILE</code></strong> rescales the whole notebook. <code>'smoke'</code> = six months, stride 60, 2 epochs, small model (~15 min). <code>'full'</code> = everything, stride 5, 30 epochs, <code>d_model=512</code> (~8–10 h).</li>
    <li><strong><code>RAW_DIR</code></strong> is the one path you must edit for your Kaggle Dataset mount.</li>
    <li><strong>Exogenous history is always loaded in full</strong>, regardless of profile. Only the BTC minute grid is shortened. This matters: a 36-month macro z-score needs 36 months of macro history even when you are training on six months of minutes.</li>
    <li><strong>The publication-lag table</strong> is part of the config, not buried in code, because it is the single most consequential anti-leakage decision in the project.</li>
  </ul>
</div>

In [ ]:
# ==========================================================================
#  EDIT THIS ONE LINE to match your Kaggle Dataset mount path.
#  Local runs fall back to ../data/raw automatically.
# ==========================================================================
KAGGLE_RAW_DIR = Path("/kaggle/input/itransformer-btc-raw")

# Resuming across Kaggle sessions: attach the PREVIOUS session's output as an input
# dataset and point this at the checkpoint folder inside it, e.g.
#   "/kaggle/input/itransformer-session-1/checkpoints/full_L1440_H60_d512_s42"
# Leave as None for a fresh run. See docs/KAGGLE_GUIDE.md.
KAGGLE_RESUME_DIR: str | None = None

PROFILE = "smoke"          # 'tiny' (~2 min, CPU) | 'smoke' (~15 min) | 'full' (staged, see guide)

REQUIRED_FILES = (
    [f"btc_usdt_binance_{y}.parquet" for y in range(2018, 2027)]
    + ["xauusd_2018_2026.parquet",
       "US_Dollar_Index_2018_2026.parquet",
       "fed_economic_data_2018_2026.parquet"]
)

ON_KAGGLE = Path("/kaggle/input").exists()


def _complete(d: Path) -> bool:
    return d.is_dir() and not [f for f in REQUIRED_FILES if not (d / f).exists()]


def _discover_raw_dir() -> Path:
    """Locate the folder holding all 12 raw files.

    Kaggle mounts a dataset at /kaggle/input/<slug>, and the slug is whatever the
    uploader named it - hard-coding one guarantees a broken first run for everyone
    else. Search the mount points instead, one level deep for zipped-folder uploads.
    """
    cands = [KAGGLE_RAW_DIR, Path("../data/raw"), Path("data/raw")]
    root = Path("/kaggle/input")
    if root.exists():
        tops = sorted(p for p in root.iterdir() if p.is_dir())
        cands += tops + [q for p in tops for q in sorted(p.iterdir()) if q.is_dir()]
    return next((c for c in cands if _complete(c)), KAGGLE_RAW_DIR)


RAW_DIR = _discover_raw_dir()
WORK_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("../artifacts")
WORK_DIR.mkdir(parents=True, exist_ok=True)

_missing = [f for f in REQUIRED_FILES if not (RAW_DIR / f).exists()]
assert not _missing, (
    f"RAW_DIR={RAW_DIR} is missing {len(_missing)} file(s): {_missing[:4]}...\n"
    f"Attach the dataset holding the 12 raw parquet files, or set KAGGLE_RAW_DIR at "
    f"the top of this cell to its mount path."
)
print(f"raw   -> {RAW_DIR}   ({len(REQUIRED_FILES)} files present)")
print(f"work  -> {WORK_DIR}")

In [ ]:
@dataclass
class Config:
    """Single source of truth. Serialised verbatim into the export bundle."""

    # ---- identity -------------------------------------------------------
    profile: str = PROFILE
    seed: int = 42
    run_id: str = ""

    # ---- master grid ----------------------------------------------------
    grid_start: str = "2018-01-02 00:00"
    grid_end: str = "2026-05-31 23:59"     # gold/macro end here; BTC's extra June 2026 is discarded
    # Gold broker-clock offset in hours, resolved empirically in section 5.
    # 0 = the file is already UTC. 'auto' re-derives it from the data.
    gold_utc_offset_h: object = 0

    # ---- windowing ------------------------------------------------------
    seq_len: int = 1440                    # L: one full day of minutes
    pred_len: int = 60                     # H: predict the next 60 one-minute log returns
    train_stride: int = 5                  # eval strides are always 1
    horizons: tuple = (1, 5, 15, 30, 60)   # cumulative horizons reported downstream

    # ---- data quality tolerances ----------------------------------------
    max_synth_run_in_window: int = 15      # reject a window containing >15 min of synthetic BTC bars
    gold_staleness_cap_min: int = 4320     # 3 days: a normal weekend is fine, an outage is not

    # ---- splits ---------------------------------------------------------
    train_end: str = "2023-12-31 23:59"
    val_end: str = "2024-12-31 23:59"
    test_end: str = "2026-05-31 23:59"
    embargo_safety_min: int = 1440         # extra margin on top of L + H

    # ---- model ----------------------------------------------------------
    d_model: int = 512
    n_heads: int = 8
    e_layers: int = 3
    d_ff: int = 2048
    dropout: float = 0.1
    activation: str = "gelu"
    use_norm: bool = True                  # RevIN-style instance normalisation
    use_linear_skip: bool = True           # DLinear-style residual skip, zero-initialised
    project_target_only: bool = True
    norm_style: str = "post"               # 'post' reproduces thuml/iTransformer; 'pre' deviates
    patch_len: int = 60                    # TimeXer / PatchTST patch length, in minutes

    # ---- optimisation ---------------------------------------------------
    epochs: int = 30
    batch_size: int = 128
    lr: float = 3e-4
    weight_decay: float = 1e-4
    betas: tuple = (0.9, 0.98)
    warmup_frac: float = 0.05
    grad_clip: float = 1.0
    loss: str = "huber"                    # 'mse' | 'huber' | 'pinball' | 'huber_dir'
    huber_delta: float = 1.0
    dir_lambda: float = 0.1                # only used by 'huber_dir'
    quantiles: tuple = (0.1, 0.5, 0.9)     # only used by 'pinball'
    early_stop_patience: int = 5
    num_workers: int = 2
    deterministic: bool = False

    # ---- what to run ----------------------------------------------------
    # Stage switches. One 12 h Kaggle session cannot fit the main model plus six
    # trained baselines plus five ablations plus walk-forward at the full profile -
    # run them in separate sessions and let the checkpoints carry over.
    run_baselines: bool = True             # AR / DLinear / PatchTST / TimeXer / VanillaTF
    run_vanilla_transformer: bool = True   # the expensive baseline (L x L attention)
    run_ablation: bool = True
    run_walkforward: bool = False          # ~5x runtime; see section 15
    walkforward_months: int = 3
    resume: bool = True
    # Stop training while there is still session left, so the checkpoint is written and
    # the version can be saved. Kaggle kills a session at the 12 h wall without warning.
    session_budget_hours: float = 11.0     # 0 disables the guard
    reserve_hours: float = 0.5             # stop this long before the budget runs out

    # ---- evaluation / backtest ------------------------------------------
    eval_max_windows: int = 200_000        # cap eval windows so a full pass stays tractable
    backtest_horizon: int = 60
    fee_per_side: float = 0.0004           # Binance spot taker, 4 bps
    slippage_per_side: float = 0.0002      # 2 bps, conservative for BTC/USDT at 1 min
    dir_acc_eps_bp: float = 1.0            # ignore |return| < 1 bp when scoring direction
    n_seeds_report: int = 5

    # ---- feature blocks (toggles feed the ablation table) ---------------
    blocks: tuple = ("btc_price", "btc_volume", "btc_momentum",
                     "gold", "cross_asset", "dxy", "macro", "temporal")
    macro_n_pca: int = 4
    fracdiff_grid: tuple = (0.2, 0.3, 0.4, 0.5, 0.6)
    fracdiff_width: int = 512
    winsor_q: float = 0.001
    collinear_thresh: float = 0.98

    @property
    def embargo_min(self) -> int:
        return self.seq_len + self.pred_len + self.embargo_safety_min

    @property
    def split_gap_min(self) -> int:
        """purge (= H) + embargo, in minutes, inserted at every split boundary."""
        return self.pred_len + self.embargo_min


CFG = Config()

# ---- profile overrides ---------------------------------------------------
if CFG.profile == "smoke":
    CFG.grid_start, CFG.grid_end = "2021-01-02 00:00", "2021-06-30 23:59"
    CFG.train_end, CFG.val_end, CFG.test_end = (
        "2021-04-30 23:59", "2021-05-31 23:59", "2021-06-30 23:59")
    CFG.seq_len, CFG.pred_len = 480, 60
    CFG.train_stride = 60
    CFG.d_model, CFG.d_ff, CFG.e_layers, CFG.n_heads = 128, 256, 2, 4
    CFG.epochs, CFG.batch_size = 2, 64
    CFG.embargo_safety_min = 240
    CFG.eval_max_windows = 40_000
    CFG.n_seeds_report = 1
elif CFG.profile == "tiny":
    # CPU-runnable smoke of the smoke: used to verify the notebook itself.
    CFG.grid_start, CFG.grid_end = "2021-01-02 00:00", "2021-03-31 23:59"
    CFG.train_end, CFG.val_end, CFG.test_end = (
        "2021-02-15 23:59", "2021-03-07 23:59", "2021-03-31 23:59")
    CFG.seq_len, CFG.pred_len = 120, 15
    CFG.train_stride = 240
    CFG.d_model, CFG.d_ff, CFG.e_layers, CFG.n_heads = 64, 128, 2, 4
    CFG.epochs, CFG.batch_size = 1, 32
    CFG.embargo_safety_min = 60
    CFG.horizons = (1, 5, 15)
    CFG.backtest_horizon = 15
    CFG.eval_max_windows = 8_000
    CFG.n_seeds_report = 1
    CFG.num_workers = 0
    CFG.fracdiff_width = 128
    CFG.run_vanilla_transformer = True
elif CFG.profile != "full":
    raise ValueError(f"unknown profile {CFG.profile!r}")

assert CFG.d_model % CFG.n_heads == 0, "d_model must be divisible by n_heads"

CFG.run_id = f"{CFG.profile}_L{CFG.seq_len}_H{CFG.pred_len}_d{CFG.d_model}_s{CFG.seed}"
RUN_DIR = WORK_DIR / "runs" / CFG.run_id
CKPT_DIR = WORK_DIR / "checkpoints" / CFG.run_id
for d in (RUN_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Checkpoints are READ from RESUME_DIR and always WRITTEN to CKPT_DIR, so a resumed
# session reads the previous session's read-only input dataset and writes its own output.
RESUME_DIR = Path(KAGGLE_RESUME_DIR) if KAGGLE_RESUME_DIR else CKPT_DIR

SESSION_T0 = time.time()


def hours_left() -> float:
    """Hours remaining in the self-imposed session budget."""
    return CFG.session_budget_hours - (time.time() - SESSION_T0) / 3600.0


set_seed(CFG.seed, CFG.deterministic)

print(f"profile      {CFG.profile}")
print(f"run_id       {CFG.run_id}")
print(f"grid         {CFG.grid_start} -> {CFG.grid_end}")
print(f"L={CFG.seq_len}  H={CFG.pred_len}  stride={CFG.train_stride}")
print(f"split gap    {CFG.split_gap_min:,} min  (purge {CFG.pred_len} + embargo {CFG.embargo_min:,})")
print(f"ckpt         {CKPT_DIR}")
print(f"resume from  {RESUME_DIR}"
      + ("   (same session)" if RESUME_DIR == CKPT_DIR else "   (previous session)"))
print(f"budget       {CFG.session_budget_hours:.1f} h with {CFG.reserve_hours:.1f} h reserved  (Kaggle hard cap: 12 h)")

<div style="background: linear-gradient(90deg, #1a1a2e, #16213e); border-left: 4px solid #e94560; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #f5a623; margin: 0 0 10px 0;">📅 The publication-lag table</h2>
  <p style="color: #ffd6a5; margin: 0 0 8px 0;"><strong>This is where leakage usually enters a macro-conditioned model.</strong> A row dated <code>2018-01-31</code> for CPI was <em>not knowable</em> on 31 January — CPI for January is published around 13 February. Joining it from its nominal date hands the model two weeks of the future, every month, for eight years.</p>  <p style="color: #ffd6a5; margin: 0 0 8px 0;">Each indicator below is shifted forward by its release lag <em>before</em> any join happens. Where a release date is uncertain the lag is rounded <strong>up</strong> to a whole extra period: losing a little signal is vastly cheaper than publishing a leaked result.</p>  <ul style="color: #ffd6a5; margin: 0; padding-left: 20px;">
    <li>Market-observed rates (Treasury yields, effective fed funds) sit inside a <em>monthly</em> file, so they are monthly <em>aggregates</em> — they get the same one-month lag as everything else, not daily treatment.</li>
    <li><code>GDP</code>/<code>GDP_Real</code> are quarterly values already forward-filled to monthly. They get a full quarter of lag, and are treated as a step function rather than differenced month-over-month.</li>
    <li>The daily USD index for date <em>d</em> is only knowable after <em>d</em>'s close, so it applies from <em>d</em> + 1 day.</li>
    <li><strong>Residual risk that no lag table can fix:</strong> these are <em>revised</em> values, not real-time ALFRED vintages. Even with perfect release lags, revisions leak a small amount of future information. This is stated again in the limitations section — it is a property of the dataset, not a bug in the pipeline.</li>
  </ul>
</div>

In [ ]:
# Release lag per macro column, in (months, days) added to the nominal month-end date.
# Conservative by construction: when in doubt, round up to a whole extra period.
RELEASE_LAG = {
    "CPI":                        (1, 13),   # ~13th of M+1, 13:30 UTC
    "CPI_Core":                   (1, 13),
    "PPI":                        (1, 14),
    "PCE":                        (2, 0),    # ~last business day of M+1 -> take 2 months
    "PCE_Core":                   (2, 0),
    "Non_Farm_Payroll":           (1, 7),    # 1st Friday of M+1
    "Unemployment_Rate":          (1, 7),
    "Labor_Force_Participation":  (1, 7),
    "Retail_Sales":               (1, 16),
    "Industrial_Production":      (1, 15),
    "Capacity_Utilization":       (1, 15),
    "GDP":                        (3, 0),    # quarterly; advance ~Q_end+30d -> take a full quarter
    "GDP_Real":                   (3, 0),
    "M1":                         (1, 0),
    "M2":                         (1, 0),
    "Bank_Credit":                (1, 0),
    "Consumer_Credit":            (2, 0),
    "Reserves":                   (1, 0),
    "Fed_Balance_Sheet":          (1, 0),
    "Fed_Funds_Rate":             (1, 0),    # monthly aggregate of a market-observed rate
    "Fed_Funds_Target":           (1, 0),
    "Fed_Funds_Target_Lower":     (1, 0),
    "Treasury_10Y":               (1, 0),
    "Treasury_2Y":                (1, 0),
    "Treasury_5Y":                (1, 0),
    "Real_Interest_Rate":         (1, 0),
    "Mortgage_Rate_30Y":          (1, 0),
    "Consumer_Sentiment":         (1, 0),
    "Inflation_Expectations_5Y":  (1, 0),
    "Inflation_Expectations_10Y": (1, 0),
    "Real_Broad_Dollar_Index":    (1, 0),    # dropped later - see the redundancy note in section 6
}

DXY_LAG_DAYS = 1          # daily index for date d is knowable from d+1 00:00 UTC

# Scheduled high-impact release times, used only for temporal flags (never for values).
RELEASE_MINUTES_UTC = {"cpi_nfp": (13, 30), "fomc": (19, 0)}

print(f"{len(RELEASE_LAG)} macro columns carry an explicit release lag")
print(f"longest lag: {max(RELEASE_LAG.values())} (months, days)")

<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 4px solid #f48c06; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #ffd60a; margin: 0 0 10px 0;">📡 §3 · Loading &amp; Schema Contracts</h2>
  <p style="color: #ffb703; margin: 0 0 8px 0;"><strong>Every column in every parquet file is a <code>String</code>.</strong> The upstream CSV→parquet conversion wrote <code>dtype=str</code>, and missing values became <strong>empty strings</strong>, not nulls. A loader that ignores this either crashes or — far worse — silently coerces bad cells to null.</p>  <ul style="color: #ffb703; margin: 0; padding-left: 20px;">
    <li>Map <code>""</code> → <code>null</code> first, then cast with <code>strict=True</code> so a malformed cell <em>raises</em> instead of quietly vanishing.</li>
    <li>Assert the expected column set on load. Schema drift after a data refresh should fail loudly here, not produce a subtly wrong model three hours later.</li>
    <li>Normalise every timestamp to <strong>tz-aware UTC</strong>. BTC's <code>datetime</code> string is cross-checked against its epoch-millisecond <code>timestamp</code> column — they must agree exactly.</li>
    <li><code>data/raw/</code> is immutable. Nothing in this notebook writes to it.</li>
  </ul>
</div>

In [ ]:
def _f64(col: str, alias: str | None = None) -> pl.Expr:
    """String column -> Float64, with empty string treated as null and strict casting."""
    return pl.col(col).replace("", None).cast(pl.Float64, strict=True).alias(alias or col)


def _assert_schema(lf: pl.LazyFrame, expected: set[str], name: str) -> None:
    got = set(lf.collect_schema().names())
    missing, extra = expected - got, got - expected
    assert not missing, f"[{name}] missing columns: {sorted(missing)}"
    if extra:
        print(f"  [{name}] note: {len(extra)} unexpected column(s) ignored: {sorted(extra)[:5]}")


def load_btc(raw_dir: Path) -> pl.DataFrame:
    """BTC/USDT 1-minute OHLCV. Defines the master grid."""
    lf = pl.scan_parquet(str(raw_dir / "btc_usdt_binance_*.parquet"))
    _assert_schema(lf, {"timestamp", "open", "high", "low", "close", "volume", "datetime"}, "btc")
    df = (
        lf.select(
            pl.col("datetime")
              .str.strptime(pl.Datetime("us"), "%Y-%m-%d %H:%M:%S%:z")
              .dt.convert_time_zone("UTC")
              .alias("t"),
            pl.col("timestamp").cast(pl.Int64, strict=True).alias("_ts_ms"),
            _f64("open", "btc_open"), _f64("high", "btc_high"),
            _f64("low", "btc_low"), _f64("close", "btc_close"),
            _f64("volume", "btc_volume"),
        )
        .sort("t")
        .collect()
    )
    # The two time representations must agree exactly, or one of them is wrong.
    lhs = df["t"].dt.epoch(time_unit="ms")
    assert (lhs == df["_ts_ms"]).all(), "btc: datetime string disagrees with epoch-ms timestamp"
    return df.drop("_ts_ms")


def load_gold(raw_dir: Path, offset_hours: int) -> pl.DataFrame:
    """XAU/USD 1-minute OHLCV on the broker clock, shifted to UTC by `offset_hours`."""
    lf = pl.scan_parquet(str(raw_dir / "xauusd_2018_2026.parquet"))
    _assert_schema(lf, {"Date", "Timestamp", "Open", "High", "Low", "Close", "Volume"}, "gold")
    return (
        lf.select(
            (pl.concat_str([pl.col("Date"), pl.lit(" "), pl.col("Timestamp")])
               .str.strptime(pl.Datetime("us"), "%Y%m%d %H:%M:%S")
             - pl.duration(hours=offset_hours))
            .dt.replace_time_zone("UTC")
            .alias("t"),
            _f64("Open", "gold_open"), _f64("High", "gold_high"),
            _f64("Low", "gold_low"), _f64("Close", "gold_close"),
            _f64("Volume", "gold_volume"),
        )
        .sort("t")
        .collect()
    )


def load_dxy(raw_dir: Path) -> pl.DataFrame:
    """Daily broad trade-weighted USD index. Weekends are absent rows; holidays are empty strings."""
    lf = pl.scan_parquet(str(raw_dir / "US_Dollar_Index_2018_2026.parquet"))
    _assert_schema(lf, {"Date", "US_Dollar_Index"}, "dxy")
    return (
        lf.select(
            pl.col("Date").str.strptime(pl.Date, "%Y-%m-%d").alias("date"),
            _f64("US_Dollar_Index", "dxy"),
        )
        .drop_nulls("dxy")          # holidays carry no observation at all
        .sort("date")
        .collect()
    )


def load_macro(raw_dir: Path) -> pl.DataFrame:
    """Monthly US macro block, month-end dated. Values are revised, not real-time vintages."""
    lf = pl.scan_parquet(str(raw_dir / "fed_economic_data_2018_2026.parquet"))
    cols = [c for c in lf.collect_schema().names() if c != "Date"]
    assert set(cols) >= set(RELEASE_LAG), f"macro: lag table references unknown columns"
    return (
        lf.select(
            pl.col("Date").str.strptime(pl.Date, "%Y-%m-%d").alias("date"),
            *[_f64(c) for c in cols],
        )
        .sort("date")
        .collect()
    )


t0 = time.time()
btc_raw = load_btc(RAW_DIR)
dxy_raw = load_dxy(RAW_DIR)
macro_raw = load_macro(RAW_DIR)
print(f"btc    {btc_raw.height:>9,} rows   {btc_raw['t'][0]} -> {btc_raw['t'][-1]}")
print(f"dxy    {dxy_raw.height:>9,} rows   {dxy_raw['date'][0]} -> {dxy_raw['date'][-1]}")
print(f"macro  {macro_raw.height:>9,} rows   {macro_raw['date'][0]} -> {macro_raw['date'][-1]}")
print(f"\nloaded in {time.time() - t0:.1f}s")

<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 4px solid #f48c06; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #ffd60a; margin: 0 0 10px 0;">🕵️ §4 · Validation Battery</h2>
  <p style="color: #ffb703; margin: 0 0 8px 0;">Nothing downstream is trustworthy if the inputs are not what the specification claims. This cell re-measures the facts rather than assuming them, and prints a report you should actually read.</p>  <ul style="color: #ffb703; margin: 0; padding-left: 20px;">
    <li><strong>Monotonicity &amp; duplicates</strong> — a duplicated minute silently doubles a bar's weight in every rolling window that contains it.</li>
    <li><strong>OHLC sanity</strong> — <code>low ≤ min(open, close) ≤ max(open, close) ≤ high</code>, all strictly positive.</li>
    <li><strong>Gap census</strong> — BTC has 31 known discontinuities (Binance maintenance), the largest ≈ 10 h. These are <em>flagged</em>, never interpolated: windows spanning them are dropped later.</li>
    <li><strong>Extreme-return census</strong> — every <code>|log return| &gt; 10%</code> in one minute is listed, not deleted. Those minutes are the phenomenon of interest (2020-03-12, 2021-05-19, FTX 2022-11), and removing them is how you build a model that fails exactly when it matters.</li>
  </ul>
</div>

In [ ]:
def validate_ohlcv(df: pl.DataFrame, prefix: str, name: str) -> dict:
    """Print a validation report for one OHLCV source and return its key statistics."""
    o, h, l, c = (f"{prefix}_{x}" for x in ("open", "high", "low", "close"))
    v = f"{prefix}_volume"
    n = df.height
    print(f"--- {name} " + "-" * (58 - len(name)))
    print(f"  rows              {n:,}")

    t = df["t"]
    assert t.is_sorted(), f"{name}: timestamps not sorted"
    n_dup = n - t.n_unique()
    print(f"  duplicate stamps  {n_dup:,}" + ("   <-- MUST BE ZERO" if n_dup else ""))
    assert n_dup == 0, f"{name}: {n_dup} duplicate timestamps"

    bad_ohlc = df.filter(
        (pl.col(l) > pl.min_horizontal(o, c)) | (pl.col(h) < pl.max_horizontal(o, c))
        | (pl.col(l) > pl.col(h)) | (pl.col(c) <= 0)
    ).height
    print(f"  OHLC violations   {bad_ohlc:,}" + ("   <-- inspect" if bad_ohlc else ""))
    if v in df.columns:
        print(f"  negative volume   {df.filter(pl.col(v) < 0).height:,}")

    nulls = {c_: df[c_].null_count() for c_ in df.columns if df[c_].null_count()}
    print(f"  nulls             {nulls if nulls else 'none'}")

    # gap census
    step = t.diff().dt.total_minutes().drop_nulls()
    gaps = step.filter(step > 1)
    print(f"  1-min steps       {(step == 1).sum():,}")
    print(f"  discontinuities   {gaps.len():,}")
    if gaps.len():
        big = gaps.sort(descending=True).head(5).to_list()
        print(f"  largest gaps      {[f'{g / 60:.2f}h' for g in big]}")
        buckets = [(1, 5), (5, 60), (60, 24 * 60), (24 * 60, 10**9)]
        hist = {f"{a}-{b}m": int(((gaps > a) & (gaps <= b)).sum()) for a, b in buckets}
        print(f"  gap histogram     {hist}")

    # extreme returns, on the source's own clock
    r = (pl.Series(np.log(df[c].to_numpy())).diff()).drop_nulls()
    ext = int((r.abs() > 0.10).sum())
    print(f"  |1-min logret|>10%  {ext:,}")
    if ext:
        idx = np.where(np.abs(r.to_numpy()) > 0.10)[0] + 1
        for i in idx[:6]:
            print(f"      {df['t'][int(i)]}  r={r[int(i) - 1]:+.4f}  close={df[c][int(i)]:,.2f}")
        if len(idx) > 6:
            print(f"      ... and {len(idx) - 6} more")
    print()
    return {"rows": n, "gaps": gaps.len(), "extremes": ext, "bad_ohlc": bad_ohlc}


rep_btc = validate_ohlcv(btc_raw, "btc", "BTC/USDT 1-min")

print("--- USD index (daily) " + "-" * 41)
print(f"  rows (non-null)   {dxy_raw.height:,}")
print(f"  range             {dxy_raw['date'][0]} -> {dxy_raw['date'][-1]}")
print(f"  level             {dxy_raw['dxy'].min():.2f} .. {dxy_raw['dxy'].max():.2f}")
_gapd = dxy_raw["date"].diff().dt.total_days().drop_nulls()
print(f"  calendar gaps     {dict(zip(*np.unique(_gapd.to_numpy(), return_counts=True)))}")

print("\n--- Macro (monthly) " + "-" * 43)
print(f"  rows              {macro_raw.height:,}   {macro_raw['date'][0]} -> {macro_raw['date'][-1]}")
_mn = {c: macro_raw[c].null_count() for c in macro_raw.columns if macro_raw[c].null_count()}
print(f"  nulls             {_mn if _mn else 'none'}")
_gdp = macro_raw["GDP"].drop_nulls()
print(f"  GDP distinct/rows {_gdp.n_unique()}/{_gdp.len()}   (quarterly, forward-filled to monthly)")

<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 4px solid #f48c06; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #ffd60a; margin: 0 0 10px 0;">🌍 §5 · Resolving the Gold Timezone — Empirically</h2>
  <p style="color: #ffb703; margin: 0 0 8px 0;">The gold file's timestamps carry <strong>no timezone label</strong>. Its first bar is <code>2018-01-01 23:00</code>, and its weekly closure pattern is consistent with a broker server clock — commonly UTC+2/UTC+3 — rather than UTC. <strong>Guessing wrong injects a fixed look-ahead or look-behind bias into every cross-asset feature</strong>, so this is measured, not assumed.</p>  <p style="color: #ffb703; margin: 0 0 8px 0;">Two independent lines of evidence are computed below. They agree.</p>  <ul style="color: #ffb703; margin: 0; padding-left: 20px;">
    <li><strong>Structural.</strong> Where does the weekly closure fall? A gold market anchored to New York closes at 17:00 New York time — that is 22:00 UTC in winter and 21:00 UTC in summer.</li>
    <li><strong>Statistical.</strong> Shift the gold series by each candidate offset from −12 h to +12 h and measure the contemporaneous correlation of 1-minute BTC and gold log returns. The true offset should stand out sharply; every wrong offset should look like noise.</li>
  </ul>  <p style="color: #ffb703; margin: 0 0 8px 0;"><strong>Result (pre-computed on the full dataset, reproduced by the cell below):</strong> the last bar before the weekly break is <code>Fri 21:59</code> in winter and <code>Fri 20:59</code> in summer, reopening <code>Sun 23:00</code> / <code>Sun 22:00</code> — exactly New York 17:00 close and 18:00 reopen, in both DST regimes. The correlation scan peaks at <code>+0h</code> with <code>ρ = +0.0626</code> while every other offset sits at <code>±0.002</code>, a 30× separation. <strong>The gold file is already in UTC; the offset is 0.</strong> CLAUDE.md §17 verification task 1 is closed.</p>
</div>

In [ ]:
def detect_gold_offset(raw_dir: Path, scan_years: tuple = (2021, 2024)) -> dict:
    """Recover the gold broker-clock UTC offset from structure + BTC cross-correlation.

    Returns the argmax offset in whole hours along with the full evidence table.
    """
    g = load_gold(raw_dir, offset_hours=0).select("t", "gold_close")
    t = g["t"].dt.replace_time_zone(None).to_numpy()
    step = np.diff(t).astype("timedelta64[m]").astype(np.int64)

    def _season(arr):
        m = np.array([int(str(x)[5:7]) for x in arr])
        return np.isin(m, [11, 12, 1, 2]), np.isin(m, [4, 5, 6, 7, 8, 9])

    def _wd(x):   # 1970-01-01 was a Thursday
        return ["Thu", "Fri", "Sat", "Sun", "Mon", "Tue", "Wed"][int(np.datetime64(x, "D").astype(int)) % 7]

    wk = np.where(step >= 24 * 60)[0]
    close_b, open_b = t[wk], t[wk + 1]
    win, summ = _season(close_b)
    struct = {}
    for label, sel in (("winter", win), ("summer", summ)):
        cc = np.unique([f"{_wd(x)} {str(x)[11:16]}" for x in close_b[sel]], return_counts=True)
        oo = np.unique([f"{_wd(x)} {str(x)[11:16]}" for x in open_b[sel]], return_counts=True)
        struct[label] = {"close": cc[0][cc[1].argmax()], "open": oo[0][oo[1].argmax()]}
        print(f"  weekly break ({label}):  close {struct[label]['close']}  ->  reopen {struct[label]['open']}")

    lo, hi = datetime(scan_years[0], 1, 1, tzinfo=timezone.utc), datetime(scan_years[1], 1, 1, tzinfo=timezone.utc)
    b = (btc_raw.filter(pl.col("t").is_between(lo, hi))
                .select("t", pl.col("btc_close").log().diff().alias("br")).drop_nulls())
    gr = g.select("t", pl.col("gold_close").log().diff().alias("gr")).drop_nulls()

    scan = []
    for off in range(-12, 13):
        j = b.join(gr.with_columns(pl.col("t") - pl.duration(hours=off)), on="t", how="inner")
        if j.height < 10_000:
            continue
        x, y = j["br"].to_numpy(), j["gr"].to_numpy()
        m = np.isfinite(x) & np.isfinite(y) & (y != 0.0)
        scan.append((off, float(np.corrcoef(x[m], y[m])[0, 1]), int(m.sum())))

    best = max(scan, key=lambda r: abs(r[1]))
    runner = max((s for s in scan if s[0] != best[0]), key=lambda r: abs(r[1]))
    print(f"\n  correlation argmax:  offset {best[0]:+d}h   rho={best[1]:+.5f}  (n={best[2]:,})")
    print(f"  next best         :  offset {runner[0]:+d}h   rho={runner[1]:+.5f}"
          f"   -> separation {abs(best[1] / runner[1]):.0f}x")
    return {"offset": best[0], "scan": scan, "structure": struct}


if CFG.gold_utc_offset_h == "auto":
    _det = detect_gold_offset(RAW_DIR)
    GOLD_OFFSET_H = _det["offset"]
    fig, ax = plt.subplots(figsize=(9, 3.2))
    offs = [s[0] for s in _det["scan"]]
    cors = [abs(s[1]) for s in _det["scan"]]
    ax.bar(offs, cors, color=[CAT[1] if o == GOLD_OFFSET_H else GRID for o in offs], width=0.7)
    ax.annotate(f"offset {GOLD_OFFSET_H:+d}h", (GOLD_OFFSET_H, max(cors)),
                textcoords="offset points", xytext=(0, 6), ha="center", color=CAT[1], fontsize=9)
    finish(ax, "Gold-clock offset scan  |  contemporaneous BTC-gold 1-min return correlation",
           "candidate offset (hours)", "|rho|")
    plt.tight_layout(); plt.show()
else:
    GOLD_OFFSET_H = int(CFG.gold_utc_offset_h)
    print(f"  using pinned offset {GOLD_OFFSET_H:+d}h "
          f"(set CFG.gold_utc_offset_h='auto' to re-derive it from the data)")

gold_raw = load_gold(RAW_DIR, GOLD_OFFSET_H)
print(f"\ngold   {gold_raw.height:>9,} rows   {gold_raw['t'][0]} -> {gold_raw['t'][-1]}")
rep_gold = validate_ohlcv(gold_raw, "gold", "XAU/USD 1-min (UTC-normalised)")

<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 4px solid #48cae4; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #90e0ef; margin: 0 0 10px 0;">🧬 §6 · Multi-Granularity Alignment</h2>
  <p style="color: #ade8f4; margin: 0 0 8px 0;">Four series sampled at 1 min / 1 min-with-closures / 1 day / 1 month must become <strong>one matrix on the BTC minute grid</strong> without leaking the future. This section is the part that most determines whether the project succeeds.</p>  <p style="color: #ade8f4; margin: 0 0 8px 0;"><strong>The governing principle:</strong> at master timestamp <em>t</em>, a feature may use only information a real observer would have possessed at or before <em>t</em>. Every join is therefore a <strong>backward as-of join</strong> — never nearest, never forward.</p>  <p style="color: #ade8f4; margin: 0 0 8px 0;"><strong>The architectural decision that makes this tractable:</strong> each source's features are computed <em>on its own native clock first</em>, and only the finished features are as-of joined onto the minute grid. Doing it the other way round — forward-filling gold prices onto every minute and then differencing — manufactures long runs of exactly-zero returns that corrupt every volatility and correlation estimate downstream. It also keeps the joined frame at ~30 columns instead of ~45, which matters at 4.4M rows.</p>  <ul style="color: #ade8f4; margin: 0; padding-left: 20px;">
    <li><strong>BTC gaps</strong> are materialised as explicit rows carrying <code>btc_is_synthetic=1</code>, price forward-filled and volume zero. They are never interpolated, and windows overlapping them are dropped at sampling time.</li>
    <li><strong>Gold staleness</strong> is a feature, not a nuisance: <code>log1p(minutes since last gold bar)</code> plus a binary <code>gold_market_open</code>. The model is told how stale its inputs are.</li>
    <li><strong>The weekend gap return</strong> (Friday close → Sunday open) is a genuine two-day return. It is not spread across the weekend minutes; it appears at the reopening bar and decays with a 60-minute time constant.</li>
    <li><strong>Macro and USD index</strong> are shifted by the publication-lag table <em>before</em> joining, and each carries an age feature (<code>macro_age_days</code>, <code>dxy_age_days</code>).</li>
  </ul>
</div>

In [ ]:
UTC = timezone.utc


def ts(s: str) -> datetime:
    """'YYYY-MM-DD HH:MM' -> tz-aware UTC datetime."""
    return datetime.strptime(s, "%Y-%m-%d %H:%M").replace(tzinfo=UTC)


# Warm-up prefix: the longest trailing window (1440) plus one lookback window,
# so that the first *usable* window at grid_start already has complete features.
WARMUP_MIN = 1440 + CFG.seq_len + 60
GRID_LO = max(ts(CFG.grid_start) - timedelta(minutes=WARMUP_MIN), btc_raw["t"][0])
GRID_HI = ts(CFG.grid_end)

grid = pl.datetime_range(GRID_LO, GRID_HI, interval="1m", time_zone="UTC", eager=True).alias("t")
master = pl.DataFrame({"t": grid}).join(
    btc_raw.filter(pl.col("t").is_between(GRID_LO, GRID_HI)), on="t", how="left"
)
n_synth = master["btc_close"].null_count()
master = master.with_columns(
    pl.col("btc_close").is_null().cast(pl.Float32).alias("btc_is_synthetic")
).with_columns(
    # forward-fill price through a gap (never backward), zero the volume
    *[pl.col(c).forward_fill() for c in ("btc_open", "btc_high", "btc_low", "btc_close")],
    pl.col("btc_volume").fill_null(0.0),
)
assert master["btc_close"].null_count() == 0, "grid starts before the first BTC bar"

print(f"master grid   {master.height:,} rows   {GRID_LO} -> {GRID_HI}")
print(f"  warm-up prefix   {WARMUP_MIN:,} min before {CFG.grid_start}")
print(f"  synthetic bars   {n_synth:,}  ({100 * n_synth / master.height:.4f}% of the grid)")

In [ ]:
# ---------------------------------------------------------------------------
# Gold features, computed on GOLD'S OWN CLOCK, then as-of joined.
# ---------------------------------------------------------------------------
GAP_MIN = 120          # a break longer than this is a closure, and its return is a "gap return"

gold_f = (
    gold_raw.with_columns(pl.col("gold_close").log().alias("_lg"))
    .with_columns(
        pl.col("t").diff().dt.total_minutes().alias("_step"),
        pl.col("_lg").diff().alias("gold_logret_1"),
    )
    .with_columns(
        # a return realised across a closure is a gap return, not a 1-minute return
        pl.when(pl.col("_step") > GAP_MIN).then(pl.col("gold_logret_1"))
          .otherwise(None).alias("_gap_ret"),
    )
    .with_columns(
        pl.when(pl.col("_step") > GAP_MIN).then(0.0)
          .otherwise(pl.col("gold_logret_1")).alias("gold_logret_1"),
        pl.when(pl.col("_step") > GAP_MIN).then(pl.col("t"))
          .otherwise(None).alias("_reopen_t"),
    )
    .with_columns(
        pl.col("_gap_ret").forward_fill().fill_null(0.0).alias("gold_gap_ret"),
        pl.col("_reopen_t").forward_fill().alias("gold_reopen_t"),
        (pl.col("_lg") - pl.col("_lg").shift(5)).alias("gold_logret_5"),
        (pl.col("_lg") - pl.col("_lg").shift(60)).alias("gold_logret_60"),
        (pl.col("_lg") - pl.col("_lg").shift(1440)).alias("gold_logret_1440"),
    )
    .with_columns(
        pl.col("gold_logret_1").pow(2).rolling_sum(60).sqrt().alias("gold_rv_60"),
        pl.col("t").alias("_gold_obs_t"),
    )
    .select("t", "_gold_obs_t", "gold_logret_1", "gold_logret_5", "gold_logret_60",
            "gold_logret_1440", "gold_rv_60", "gold_gap_ret", "gold_reopen_t")
)
print(f"gold features on gold clock: {gold_f.height:,} rows, {gold_f.width - 1} columns")

# ---------------------------------------------------------------------------
# Cross-asset features, also on gold's clock: BTC is as-of joined ONTO gold so
# that correlations are never contaminated by forward-filled zero gold returns.
# ---------------------------------------------------------------------------
xa = (
    gold_raw.select("t", pl.col("gold_close").log().alias("lg"))
    .join_asof(btc_raw.select("t", pl.col("btc_close").log().alias("lb")),
               on="t", strategy="backward")
    .drop_nulls()
    .with_columns(pl.col("lg").diff().alias("rg"), pl.col("lb").diff().alias("rb"))
    .drop_nulls()
)
LL = 5   # lead-lag probe, in gold-clock minutes
xa = (
    xa.with_columns(
        pl.rolling_corr(pl.col("rb"), pl.col("rg"), window_size=60).alias("xa_corr_60"),
        pl.rolling_corr(pl.col("rb"), pl.col("rg"), window_size=1440).alias("xa_corr_1440"),
        (pl.rolling_cov(pl.col("rb"), pl.col("rg"), window_size=1440)
         / (pl.col("rg").rolling_var(1440) + 1e-12)).alias("xa_beta_1440"),
        pl.rolling_corr(pl.col("rb"), pl.col("rg").shift(LL), window_size=240)
          .alias("xa_gold_leads_btc"),
        pl.rolling_corr(pl.col("rg"), pl.col("rb").shift(LL), window_size=240)
          .alias("xa_btc_leads_gold"),
    )
    .with_columns((pl.col("lb") - pl.col("xa_beta_1440") * pl.col("lg")).alias("_spread"))
    .with_columns(
        ((pl.col("_spread") - pl.col("_spread").rolling_mean(1440))
         / (pl.col("_spread").rolling_std(1440) + 1e-12)).alias("xa_spread_z")
    )
    .select("t", "xa_corr_60", "xa_corr_1440", "xa_beta_1440",
            "xa_gold_leads_btc", "xa_btc_leads_gold", "xa_spread_z")
)
print(f"cross-asset features on gold clock: {xa.height:,} rows, {xa.width - 1} columns")

In [ ]:
# ---------------------------------------------------------------------------
# USD index: daily clock. Available from d+1 00:00 UTC (after d's close).
# ---------------------------------------------------------------------------
def _pct_rank_252(x: np.ndarray) -> np.ndarray:
    """Trailing percentile rank of the latest value within a 252-observation window."""
    out = np.full(x.shape, np.nan)
    for i in range(251, len(x)):
        w = x[i - 251: i + 1]
        out[i] = (w < w[-1]).mean()
    return out


_d = dxy_raw.with_columns(pl.col("dxy").log().alias("_l"))
dxy_f = _d.with_columns(
    (pl.col("_l") - pl.col("_l").shift(1)).alias("dxy_logret_1d"),
    (pl.col("_l") - pl.col("_l").shift(5)).alias("dxy_logret_5d"),
    (pl.col("_l") - pl.col("_l").shift(21)).alias("dxy_logret_21d"),
    pl.Series("dxy_pctrank_252", _pct_rank_252(_d["dxy"].to_numpy())),
).with_columns(
    # `t` is when the value becomes KNOWABLE; `_dxy_obs_t` is the date it describes.
    # Joining on `t` enforces the lag; measuring age from `_dxy_obs_t` makes the
    # staleness feature report how old the underlying observation actually is.
    (pl.col("date").cast(pl.Datetime("us")) + pl.duration(days=DXY_LAG_DAYS))
    .dt.replace_time_zone("UTC").alias("t"),
    pl.col("date").cast(pl.Datetime("us")).dt.replace_time_zone("UTC").alias("_dxy_obs_t"),
).select("t", "dxy_logret_1d", "dxy_logret_5d", "dxy_logret_21d", "dxy_pctrank_252",
         "_dxy_obs_t")
print(f"dxy features (release-lagged +{DXY_LAG_DAYS}d): {dxy_f.height:,} rows")

# ---------------------------------------------------------------------------
# Macro: monthly clock. Each column shifted by its own release lag, then the
# whole block compressed to a handful of variates.
#
# `Real_Broad_Dollar_Index` is DROPPED. Measured against the daily USD index
# file it has level correlation 0.973 and month-over-month log-change
# correlation 0.989 - it is a near-duplicate variate that would inflate
# cross-variate attention on a redundant signal. The daily file is strictly
# more informative, so the monthly column goes.
# ---------------------------------------------------------------------------
MACRO_DROP = {"Real_Broad_Dollar_Index"}
MACRO_USE = [c for c in macro_raw.columns if c != "date" and c not in MACRO_DROP]

# Availability timestamp per column group. +1 extra day on top of the tabled lag,
# because a release published at 13:30 UTC is not knowable at 00:00 UTC that day.
avail = {}
for col in MACRO_USE:
    m, d = RELEASE_LAG[col]
    avail[col] = f"{m}mo{d + 1}d"

STEP_COLS = {"GDP", "GDP_Real"}      # quarterly, forward-filled: differencing them month-over-month is meaningless

mac = macro_raw.sort("date")
exprs = []
for c in MACRO_USE:
    s = pl.col(c)
    exprs.append((s / s.shift(12) - 1.0).alias(f"{c}__yoy"))
    if c not in STEP_COLS:
        exprs.append((s / s.shift(1) - 1.0).alias(f"{c}__mom"))
    exprs.append(((s - s.rolling_mean(36, min_samples=18))
                  / (s.rolling_std(36, min_samples=18) + 1e-12)).alias(f"{c}__z36"))
mac = mac.with_columns(exprs)

# Explicit, economically-motivated regime features kept out of the PCA block.
mac = mac.with_columns(
    (pl.col("Treasury_10Y") - pl.col("Treasury_2Y")).alias("macro_yield_curve"),
    (pl.col("Fed_Funds_Rate") - 100 * pl.col("CPI__yoy")).alias("macro_real_rate"),
    (pl.col("M2").log().diff(12)).alias("macro_m2_impulse"),
    (pl.col("Fed_Balance_Sheet").log().diff(12)).alias("macro_bs_impulse"),
)
print(f"macro: {len(MACRO_USE)} indicators -> {mac.width - 1} derived columns "
      f"(dropped {sorted(MACRO_DROP)})")

<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 4px solid #48cae4; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #90e0ef; margin: 0 0 10px 0;">🧮 Compressing the macro block</h2>
  <p style="color: #ade8f4; margin: 0 0 8px 0;">31 indicators × three transforms each is ~90 columns. Feeding that to a model whose attention runs <em>over variates</em> would drown the four minute-frequency signals that actually carry short-horizon information. Two levers control this, and both are fitted on the training split alone:</p>  <ul style="color: #ade8f4; margin: 0; padding-left: 20px;">
    <li><strong>PCA to 4 components.</strong> Loadings are fitted only on months whose <em>availability</em> timestamp falls inside the training range — not months whose nominal date does. That distinction is the whole point of the lag table.</li>
    <li><strong>Four hand-picked regime features kept outside the PCA</strong> — yield-curve slope, real rate, M2 impulse, balance-sheet impulse — because they are economically interpretable and we want to read them off the attention map later.</li>
  </ul>
</div>

In [ ]:
# Availability timestamp: nominal month-end shifted by that column's release lag.
mac_av = mac.with_columns([
    (pl.col("date").cast(pl.Datetime("us")).dt.offset_by(avail[c]))
    .dt.replace_time_zone("UTC").alias(f"_av__{c}")
    for c in MACRO_USE
])

TRAIN_END_TS = ts(CFG.train_end)
Z_COLS = [c for c in mac.columns if "__" in c and not c.startswith("_av")]

# One as-of join per distinct lag group keeps this to a handful of joins.
groups: dict[str, list[str]] = {}
for c in MACRO_USE:
    groups.setdefault(avail[c], []).append(c)

blocks = []
for lag_str, cols in groups.items():
    sub = [z for z in Z_COLS if z.split("__")[0] in cols]
    if not sub:
        continue
    frame = (mac_av.select(pl.col(f"_av__{cols[0]}").alias("t"), *sub)
                   .drop_nulls("t").sort("t"))
    blocks.append((lag_str, frame))
print(f"{len(blocks)} distinct release-lag groups -> {len(blocks)} as-of joins")

# PCA on the standardised macro transform matrix, fitted on TRAIN months only.
pca_frame = blocks[0][1].select("t")
for _, fr in blocks:
    pca_frame = pca_frame.join(fr, on="t", how="full", coalesce=True)
pca_frame = pca_frame.sort("t").fill_null(strategy="forward").drop_nulls()

feat_cols = [c for c in pca_frame.columns if c != "t"]
M = pca_frame.select(feat_cols).to_numpy().astype(np.float64)
train_mask = (pca_frame["t"] <= TRAIN_END_TS).to_numpy()
assert train_mask.sum() >= 12, "not enough in-train macro months to fit PCA"

# Ratio transforms (x/x.shift - 1) go infinite wherever the lagged value is zero,
# and 0/0 gives NaN. Left in place these make the SVD non-convergent, so they are
# neutralised to the column mean *before* the loadings are fitted.
n_bad_macro = int((~np.isfinite(M)).sum())
M[~np.isfinite(M)] = np.nan
col_ok = np.isfinite(M[train_mask]).sum(0) >= max(6, int(0.5 * train_mask.sum()))
col_ok &= np.nanstd(M[train_mask], axis=0) > 1e-10
M, feat_cols = M[:, col_ok], [c for c, k in zip(feat_cols, col_ok) if k]
mu_m = np.nanmean(M[train_mask], axis=0)
M = np.where(np.isfinite(M), M, mu_m)
print(f"macro matrix: {n_bad_macro:,} non-finite cells neutralised, "
      f"{int((~col_ok).sum())} degenerate columns dropped -> {M.shape[1]} inputs")

sd_m = np.std(M[train_mask], axis=0) + 1e-12
Zt = np.clip((M[train_mask] - mu_m) / sd_m, -10, 10)
try:
    U, S, Vt = np.linalg.svd(Zt, full_matrices=False)
except np.linalg.LinAlgError:                       # fall back to the covariance eigendecomposition
    ev, V = np.linalg.eigh(Zt.T @ Zt)
    order = np.argsort(ev)[::-1]
    S, Vt = np.sqrt(np.clip(ev[order], 0, None)), V[:, order].T
K = min(CFG.macro_n_pca, Vt.shape[0], int(train_mask.sum()) - 1)
W_pca = Vt[:K].T                                   # (n_features, K)
evr = (S**2 / (S**2).sum())[:K]
Zall = np.clip((M - mu_m) / sd_m, -10, 10)
PC = Zall @ W_pca
PC /= (PC[train_mask].std(0) + 1e-12)              # unit variance on the train split

macro_f = pl.DataFrame({"t": pca_frame["t"]}).with_columns(
    *[pl.Series(f"macro_pc{i + 1}", PC[:, i]) for i in range(K)]
).join(
    mac_av.select(
        pl.col("_av__M2").alias("t"),
        "macro_yield_curve", "macro_real_rate", "macro_m2_impulse", "macro_bs_impulse",
    ).drop_nulls("t"),
    on="t", how="left", coalesce=True,
).sort("t").fill_null(strategy="forward").with_columns(pl.col("t").alias("_macro_obs_t"))

# One row per distinct availability timestamp across all lag groups, not per calendar
# month: at each timestamp exactly one group carries fresh data and the rest are
# forward-filled, which is precisely the information set a real observer would hold.
print(f"macro PCA fitted on {int(train_mask.sum())} in-train availability timestamps "
      f"(of {len(M)} total), {len(feat_cols)} inputs -> {K} components")
print(f"  explained variance ratio: {np.round(evr, 4).tolist()}  (cumulative {evr.sum():.3f})")
print(f"macro features joined to the grid: {macro_f.width - 2} + age")

In [ ]:
# ---------------------------------------------------------------------------
# The joins. All backward as-of, all on UTC. Nothing here may look forward.
# ---------------------------------------------------------------------------
t0 = time.time()
master = (
    master
    .join_asof(gold_f.sort("t"), on="t", strategy="backward")
    .join_asof(xa.sort("t"), on="t", strategy="backward")
    .join_asof(dxy_f.sort("t"), on="t", strategy="backward")
    .join_asof(macro_f.sort("t"), on="t", strategy="backward")
)

master = master.with_columns(
    ((pl.col("t") - pl.col("_gold_obs_t")).dt.total_minutes()).alias("_gold_stale_min"),
    ((pl.col("t") - pl.col("_dxy_obs_t")).dt.total_days()).alias("dxy_age_days"),
    ((pl.col("t") - pl.col("_macro_obs_t")).dt.total_days()).alias("macro_age_days"),
    ((pl.col("t") - pl.col("gold_reopen_t")).dt.total_minutes()).alias("_since_reopen"),
).with_columns(
    pl.col("_gold_stale_min").log1p().alias("gold_staleness"),
    (pl.col("_gold_stale_min") <= 2).cast(pl.Float32).alias("gold_market_open"),
    # the weekend gap return appears at the reopening bar and decays over ~1 h
    (pl.col("gold_gap_ret") * (-pl.col("_since_reopen") / 60.0).exp()).alias("gold_gap_decay"),
)

# ---- alignment assertions ------------------------------------------------
assert master.height == grid.len(), "master row count != minute grid length"
assert master["t"].is_sorted() and master["t"].n_unique() == master.height
assert (master["_gold_stale_min"].drop_nulls() >= 0).all(), "gold as-of join looked forward"
assert (master["dxy_age_days"].drop_nulls() >= DXY_LAG_DAYS).all(), "dxy applied before its release lag"
assert (master["macro_age_days"].drop_nulls() >= 0).all(), "macro as-of join looked forward"

print(f"joined in {time.time() - t0:.1f}s   master shape {master.shape}")
print(f"  gold staleness   median {master['_gold_stale_min'].median():.0f} min, "
      f"p99 {master['_gold_stale_min'].quantile(0.99):.0f} min, "
      f"max {master['_gold_stale_min'].max():.0f} min")
print(f"  dxy age          median {master['dxy_age_days'].median():.1f} d, "
      f"max {master['dxy_age_days'].max():.0f} d")
print(f"  macro age        median {master['macro_age_days'].median():.0f} d, "
      f"max {master['macro_age_days'].max():.0f} d")
_first_ok = master.drop_nulls(["macro_pc1", "dxy_logret_1d", "gold_logret_1"])["t"][0]
print(f"  first row with every exogenous block available: {_first_ok}")

<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 4px solid #48cae4; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #90e0ef; margin: 0 0 10px 0;">🔬 §7 · Feature Engineering</h2>
  <p style="color: #ade8f4; margin: 0 0 8px 0;">Every feature is <strong>causal</strong>: computed from a trailing window ending at <em>t</em> inclusive. Section 12 proves this with a shift test rather than asserting it.</p>  <p style="color: #ade8f4; margin: 0 0 8px 0;"><strong>Variate 0 is <code>btc_logret_1</code> — this is load-bearing, not cosmetic.</strong> iTransformer's instance normalisation denormalises its output using the statistics of the target variate, so the prediction lands in exactly the units of variate 0. Making variate 0 the one-minute log return means the model's output <em>is</em> the future one-minute log-return path, and the <em>h</em>-minute cumulative return is just its cumulative sum — no separate target tensor, no unit mismatch.</p>  <ul style="color: #ade8f4; margin: 0; padding-left: 20px;">
    <li><strong>Target:</strong> <code>y = [r_{t+1}, …, r_{t+H}]</code>, one-minute log returns. Forecasting the <em>level</em> would produce a spectacular R² by echoing <code>close_t</code> while carrying zero information — any MSE reported on price levels in this domain is meaningless.</li>
    <li><strong>Fractional differentiation</strong> of <code>log(close)</code> with the smallest <em>d</em> that passes ADF at 95% on the training split — it preserves the memory that integer differencing destroys (López de Prado, ch. 5).</li>
    <li><strong>Four range-based volatility estimators</strong> (Parkinson, Garman–Klass, Rogers–Satchell, Yang–Zhang) plus realised volatility, bipower variation, and the <code>RV − BV</code> jump component.</li>
    <li><strong>Momentum indicators are deliberately capped at four.</strong> Hundreds of collinear indicators degrade cross-variate attention — the whole mechanism this architecture relies on.</li>
    <li><strong>Order-flow features are absent by design.</strong> The Binance export carries base-asset volume only — no quote volume, no trade count, no taker-buy split — so anything requiring them would be invented rather than measured.</li>
  </ul>
</div>

In [ ]:
EPS = 1e-12


def frac_weights(d: float, width: int, thresh: float = 1e-5) -> np.ndarray:
    """Fixed-width fractional-differentiation weights, longest-lag first."""
    w = [1.0]
    for k in range(1, width):
        wk = -w[-1] * (d - k + 1) / k
        if abs(wk) < thresh:
            break
        w.append(wk)
    return np.array(w[::-1])


def frac_diff(x: np.ndarray, d: float, width: int) -> np.ndarray:
    """Causal fixed-width frac-diff: output[i] uses only x[i-len(w)+1 .. i]."""
    w = frac_weights(d, width)
    out = np.full(x.shape, np.nan)
    if len(x) >= len(w):
        out[len(w) - 1:] = np.convolve(x, w, mode="valid")
    return out


def adf_stat(x: np.ndarray, lags: int = 8) -> float:
    """Augmented Dickey-Fuller t-statistic on the lagged-level coefficient (no p-value)."""
    x = x[np.isfinite(x)]
    dx = np.diff(x)
    n = len(dx) - lags
    if n < 100:
        return np.nan
    Y = dx[lags:]
    cols = [x[lags:-1], np.ones(n)] + [dx[lags - i: -i] for i in range(1, lags + 1)]
    X = np.column_stack(cols)
    beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
    resid = Y - X @ beta
    s2 = resid @ resid / (n - X.shape[1])
    se = np.sqrt(s2 * np.linalg.pinv(X.T @ X)[0, 0])
    return float(beta[0] / se)


ADF_CRIT_95 = -2.86     # Dickey-Fuller critical value, constant-only, large sample


def roll_moments(col: str, w: int) -> list[pl.Expr]:
    """Rolling skewness and excess kurtosis via raw moments (version-independent)."""
    x = pl.col(col)
    m1, m2, m3, m4 = (x.rolling_mean(w), x.pow(2).rolling_mean(w),
                      x.pow(3).rolling_mean(w), x.pow(4).rolling_mean(w))
    var = m2 - m1**2
    c3 = m3 - 3 * m1 * m2 + 2 * m1**3
    c4 = m4 - 4 * m1 * m3 + 6 * m1**2 * m2 - 3 * m1**4
    return [(c3 / (var.pow(1.5) + EPS)).alias(f"btc_skew_{w}"),
            (c4 / (var.pow(2) + EPS) - 3.0).alias(f"btc_kurt_{w}")]

In [ ]:
def build_features(m: pl.DataFrame, cfg: Config, frac_d: float) -> tuple[pl.DataFrame, dict]:
    """Build every feature block on the master minute grid.

    Pure function of `m` - this is what makes the shift test in section 12 possible.
    Returns the feature frame and a name->group mapping used by the ablation table.
    """
    groups: dict[str, str] = {}

    def reg(names, grp):
        for n in names:
            groups[n] = grp

    o, h, l, c, v = ("btc_open", "btc_high", "btc_low", "btc_close", "btc_volume")
    m = m.with_columns(
        pl.col(c).log().alias("_lc"),
        (pl.col(h) / pl.col(l)).log().alias("_hl"),
        (pl.col(c) / pl.col(o)).log().alias("_co"),
    )

    # ---- BTC price -------------------------------------------------------
    px = [pl.col("_lc").diff().alias("btc_logret_1")]
    for k in (5, 15, 30, 60, 240, 1440):
        px.append((pl.col("_lc") - pl.col("_lc").shift(k)).alias(f"btc_logret_{k}"))
    m = m.with_columns(px)
    reg([f"btc_logret_{k}" for k in (1, 5, 15, 30, 60, 240, 1440)], "btc_price")

    m = m.with_columns(
        pl.Series("btc_fracdiff", frac_diff(m["_lc"].to_numpy(), frac_d, cfg.fracdiff_width))
    )
    reg(["btc_fracdiff"], "btc_price")

    r = pl.col("btc_logret_1")
    vol = []
    for w in (15, 60, 1440):
        vol.append(r.pow(2).rolling_sum(w).sqrt().alias(f"btc_rv_{w}"))
    # bipower variation is robust to jumps; RV - BV isolates the jump component
    bpv = (r.abs() * r.abs().shift(1)).rolling_sum(60) * (math.pi / 2)
    vol += [
        bpv.sqrt().alias("btc_bpv_60"),
        (r.pow(2).rolling_sum(60) - bpv).clip(lower_bound=0.0).sqrt().alias("btc_jump_60"),
    ]
    for w in (60, 1440):
        vol.append((pl.col("_hl").pow(2).rolling_mean(w) / (4 * math.log(2))).sqrt()
                   .alias(f"btc_parkinson_{w}"))
    vol += [
        (0.5 * pl.col("_hl").pow(2).rolling_mean(60)
         - (2 * math.log(2) - 1) * pl.col("_co").pow(2).rolling_mean(60))
        .clip(lower_bound=0.0).sqrt().alias("btc_garman_klass_60"),
        (((pl.col(h) / pl.col(c)).log() * (pl.col(h) / pl.col(o)).log()
          + (pl.col(l) / pl.col(c)).log() * (pl.col(l) / pl.col(o)).log())
         .rolling_mean(60)).clip(lower_bound=0.0).sqrt().alias("btc_rogers_satchell_60"),
        r.abs().rolling_max(60).alias("btc_maxabs_60"),
        ((pl.col(c) - pl.col(l).rolling_min(60))
         / (pl.col(h).rolling_max(60) - pl.col(l).rolling_min(60) + EPS)).alias("btc_closepos_60"),
    ]
    m = m.with_columns(vol + roll_moments("btc_logret_1", 60))
    # Yang-Zhang combines overnight, open-to-close and Rogers-Satchell variance
    k_yz = 0.34 / (1.34 + (1441) / (1439))
    m = m.with_columns(
        ((pl.col("_lc") - pl.col("_lc").shift(1)).pow(2).rolling_mean(1440) * (1 - k_yz)
         + pl.col("_co").pow(2).rolling_mean(1440) * k_yz
         + pl.col("btc_rogers_satchell_60").pow(2).rolling_mean(1440))
        .clip(lower_bound=0.0).sqrt().alias("btc_yang_zhang_1440")
    )
    reg(["btc_rv_15", "btc_rv_60", "btc_rv_1440", "btc_bpv_60", "btc_jump_60",
         "btc_parkinson_60", "btc_parkinson_1440", "btc_garman_klass_60",
         "btc_rogers_satchell_60", "btc_yang_zhang_1440", "btc_skew_60", "btc_kurt_60",
         "btc_maxabs_60", "btc_closepos_60"], "btc_price")

    # ---- BTC volume / liquidity -----------------------------------------
    lv = pl.col(v).log1p()
    vwap = (pl.col(c) * pl.col(v)).rolling_sum(60) / (pl.col(v).rolling_sum(60) + EPS)
    # Corwin-Schultz two-bar high-low spread estimator
    beta_cs = pl.col("_hl").pow(2) + pl.col("_hl").pow(2).shift(1)
    gamma_cs = (pl.max_horizontal(pl.col(h), pl.col(h).shift(1))
                / pl.min_horizontal(pl.col(l), pl.col(l).shift(1))).log().pow(2)
    den = 3 - 2 * math.sqrt(2)
    alpha_cs = ((2 * beta_cs).sqrt() - beta_cs.sqrt()) / den - (gamma_cs / den).sqrt()
    m = m.with_columns(
        lv.alias("btc_logvol"),
        ((lv - lv.rolling_mean(1440)) / (lv.rolling_std(1440) + EPS)).alias("btc_volz_1440"),
        ((r.abs() / (pl.col(v) + 1e-8)).rolling_mean(60)).log1p().alias("btc_amihud_60"),
        ((pl.col(c) - vwap) / (vwap + EPS)).alias("btc_vwap_dev_60"),
        (2 * (alpha_cs.exp() - 1) / (1 + alpha_cs.exp())).clip(lower_bound=0.0)
        .rolling_mean(60).alias("btc_cs_spread_60"),
    )
    reg(["btc_logvol", "btc_volz_1440", "btc_amihud_60", "btc_vwap_dev_60",
         "btc_cs_spread_60"], "btc_volume")

    # ---- BTC momentum (capped at four) ----------------------------------
    d1 = pl.col(c).diff()
    gain = d1.clip(lower_bound=0.0).ewm_mean(alpha=1 / 14, adjust=False, min_samples=14)
    loss = (-d1).clip(lower_bound=0.0).ewm_mean(alpha=1 / 14, adjust=False, min_samples=14)
    ema12 = pl.col(c).ewm_mean(span=12, adjust=False)
    ema26 = pl.col(c).ewm_mean(span=26, adjust=False)
    macd = ema12 - ema26
    tr = pl.max_horizontal(pl.col(h) - pl.col(l),
                           (pl.col(h) - pl.col(c).shift(1)).abs(),
                           (pl.col(l) - pl.col(c).shift(1)).abs())
    ma60, sd60 = pl.col(c).rolling_mean(60), pl.col(c).rolling_std(60)
    m = m.with_columns(
        (100 - 100 / (1 + gain / (loss + EPS))).alias("btc_rsi_14"),
        ((macd - macd.ewm_mean(span=9, adjust=False)) / (pl.col(c) + EPS)).alias("btc_macd_hist"),
        (tr.ewm_mean(alpha=1 / 14, adjust=False, min_samples=14) / (pl.col(c) + EPS))
        .alias("btc_atr14_norm"),
        ((pl.col(c) - (ma60 - 2 * sd60)) / (4 * sd60 + EPS)).alias("btc_bb_pctb_60"),
    )
    reg(["btc_rsi_14", "btc_macd_hist", "btc_atr14_norm", "btc_bb_pctb_60"], "btc_momentum")

    # ---- exogenous blocks (already computed on their own clocks) --------
    reg(["gold_logret_1", "gold_logret_5", "gold_logret_60", "gold_logret_1440",
         "gold_rv_60", "gold_staleness", "gold_market_open", "gold_gap_decay"], "gold")
    reg(["xa_corr_60", "xa_corr_1440", "xa_beta_1440", "xa_spread_z",
         "xa_gold_leads_btc", "xa_btc_leads_gold"], "cross_asset")
    reg(["dxy_logret_1d", "dxy_logret_5d", "dxy_logret_21d", "dxy_pctrank_252",
         "dxy_age_days"], "dxy")
    reg([f"macro_pc{i + 1}" for i in range(K)]
        + ["macro_yield_curve", "macro_real_rate", "macro_m2_impulse",
           "macro_bs_impulse", "macro_age_days"], "macro")

    # ---- temporal --------------------------------------------------------
    tt = pl.col("t")
    mod = tt.dt.hour() * 60 + tt.dt.minute()
    dow, dom, moy = tt.dt.weekday() - 1, tt.dt.day() - 1, tt.dt.month() - 1
    cyc = []
    for name, expr, period in (("tod", mod, 1440.0), ("dow", dow, 7.0),
                               ("dom", dom, 31.0), ("moy", moy, 12.0)):
        cyc += [(2 * math.pi * expr / period).sin().alias(f"t_{name}_sin"),
                (2 * math.pi * expr / period).cos().alias(f"t_{name}_cos")]
    hr = tt.dt.hour()
    # minutes to / since the 13:30 UTC release slot on weekdays (CPI, NFP, PPI, retail sales)
    rel_h, rel_m = RELEASE_MINUTES_UTC["cpi_nfp"]
    rel_anchor = rel_h * 60 + rel_m
    delta = mod - rel_anchor
    m = m.with_columns(cyc + [
        ((hr >= 0) & (hr < 8)).cast(pl.Float32).alias("t_sess_asia"),
        ((hr >= 7) & (hr < 16)).cast(pl.Float32).alias("t_sess_europe"),
        ((hr >= 13) & (hr < 21)).cast(pl.Float32).alias("t_sess_us"),
        ((hr >= 13) & (hr < 16)).cast(pl.Float32).alias("t_sess_overlap"),
        (dow >= 5).cast(pl.Float32).alias("t_is_weekend"),
        pl.when(delta >= 0).then(delta).otherwise(delta + 1440).log1p().alias("t_since_release"),
        pl.when(delta <= 0).then(-delta).otherwise(1440 - delta).log1p().alias("t_to_release"),
        pl.col("btc_is_synthetic"),
    ])
    reg([f"t_{n}_{s}" for n in ("tod", "dow", "dom", "moy") for s in ("sin", "cos")]
        + ["t_sess_asia", "t_sess_europe", "t_sess_us", "t_sess_overlap", "t_is_weekend",
           "t_since_release", "t_to_release", "btc_is_synthetic"], "temporal")

    keep = [n for n in groups if groups[n] in cfg.blocks or n == "btc_is_synthetic"]
    # variate 0 must be the target's own series - see the section 7 note
    keep = ["btc_logret_1"] + [n for n in keep if n != "btc_logret_1"]
    return m.select(["t", *keep, "btc_close"]), {n: groups[n] for n in keep}


# ---- choose the fractional-differentiation order on the TRAIN split only ----
_lc_train = np.log(master.filter(pl.col("t") <= TRAIN_END_TS)["btc_close"].to_numpy())
_probe = _lc_train[-min(len(_lc_train), 400_000):]
FRAC_D, adf_table = None, []
for d in CFG.fracdiff_grid:
    stat = adf_stat(frac_diff(_probe, d, CFG.fracdiff_width))
    adf_table.append((d, stat))
    if FRAC_D is None and np.isfinite(stat) and stat < ADF_CRIT_95:
        FRAC_D = d
FRAC_D = FRAC_D if FRAC_D is not None else max(CFG.fracdiff_grid)
print("fractional differentiation - ADF t-statistic on the train split")
for d, s in adf_table:
    flag = "  <- selected" if d == FRAC_D else ("  (stationary)" if s < ADF_CRIT_95 else "")
    print(f"  d={d:.1f}   adf={s:8.3f}   crit95={ADF_CRIT_95}{flag}")
print(f"\nselected d = {FRAC_D}  (smallest order that rejects a unit root at 95%)")

In [ ]:
t0 = time.time()
feat_df, FEATURE_GROUP = build_features(master, CFG, FRAC_D)
print(f"features built in {time.time() - t0:.1f}s   shape {feat_df.shape}")

del master, gold_f, xa, dxy_f, macro_f, mac, mac_av, pca_frame
gc.collect()

FEATURE_NAMES_RAW = [c for c in feat_df.columns if c not in ("t", "btc_close")]
assert FEATURE_NAMES_RAW[0] == "btc_logret_1", "variate 0 must be the target series"

_counts: dict[str, int] = {}
for n in FEATURE_NAMES_RAW:
    _counts[FEATURE_GROUP[n]] = _counts.get(FEATURE_GROUP[n], 0) + 1
print(f"\n{len(FEATURE_NAMES_RAW)} raw features across {len(_counts)} blocks:")
for g, k in sorted(_counts.items(), key=lambda kv: -kv[1]):
    print(f"  {g:<14} {k:>3}")

<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 4px solid #48cae4; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #90e0ef; margin: 0 0 10px 0;">🧼 §8 · Feature Hygiene</h2>
  <p style="color: #ade8f4; margin: 0 0 8px 0;">Four operations, in this order, and every one of them is fitted on the <strong>training split alone</strong>. This is not a stylistic preference — fitting a scaler on the full dataset is a textbook leak that quietly inflates every downstream number.</p>  <ul style="color: #ade8f4; margin: 0; padding-left: 20px;">
    <li><strong>Warm-up truncation.</strong> Rows before the longest lookback has filled are dropped outright rather than imputed.</li>
    <li><strong>Winsorisation at the 0.1% / 99.9% training quantiles</strong> — but <em>the target variate is exempt</em>. Clipping <code>btc_logret_1</code> would destroy exactly the tail events the model exists to handle, and would put the input variate on a different scale from the label sliced out of it. Robust losses handle the tails instead.</li>
    <li><strong>Collinearity pruning</strong> at |ρ| &gt; 0.98, keeping the earlier (simpler) feature of each pair. With attention running over variates, a duplicated variate does not just waste a slot — it splits attention mass across two copies of one signal.</li>
    <li><strong>Standardisation</strong> to zero mean, unit variance using training-split statistics, persisted to <code>scaler.json</code> and shipped in the export bundle.</li>
  </ul>
</div>

In [ ]:
t_all = feat_df["t"]
# Cast inside polars, before materialising. `to_numpy()` on Float64 columns builds a float64
# array first - 2.04 GB at the full profile - purely to be thrown away one line later. The
# matrix has always ended up float32; this only moves the cast earlier, where it is free.
X_raw = feat_df.select([pl.col(c).cast(pl.Float32) for c in FEATURE_NAMES_RAW]).to_numpy()
close_all = feat_df["btc_close"].to_numpy().astype(np.float64)   # 35 MB, stays float64
del feat_df
gc.collect()

# ---- 1. warm-up truncation ------------------------------------------------
finite_row = np.isfinite(X_raw).all(axis=1)
first_ok = int(np.argmax(finite_row))
assert finite_row[first_ok:].mean() > 0.99, "non-finite values persist well past the warm-up"
grid_start_idx = max(first_ok, int(np.searchsorted(
    t_all.to_numpy(), np.datetime64(ts(CFG.grid_start).replace(tzinfo=None), "us"))))
X_raw, close_all, t_all = X_raw[grid_start_idx:], close_all[grid_start_idx:], t_all[grid_start_idx:]
del finite_row
# any residual non-finite cell (e.g. a division edge) is zeroed and counted
_bad = ~np.isfinite(X_raw)
n_bad = int(_bad.sum())
X_raw[_bad] = 0.0
del _bad
gc.collect()
T = len(X_raw)
print(f"warm-up truncation: dropped {grid_start_idx:,} rows -> {T:,} remain "
      f"({t_all[0]} -> {t_all[-1]});  {n_bad:,} residual non-finite cells zeroed")

# `t_all` is sorted, so the training mask is a contiguous prefix and the train block can be a
# VIEW. `X_raw[train_row]` allocates a fresh ~0.8 GB copy and it is needed five times below.
train_row = (t_all <= TRAIN_END_TS).to_numpy()
n_tr = int(train_row.sum())
assert train_row[:n_tr].all() and not train_row[n_tr:].any(), \
    "train mask is not a contiguous prefix - the view used below would silently be wrong"
Xtr = X_raw[:n_tr]
print(f"train rows for fitting statistics: {n_tr:,} ({100 * n_tr / T:.1f}%)")

# ---- 2. winsorisation, target exempt --------------------------------------
TARGET_NAME = "btc_logret_1"
names = list(FEATURE_NAMES_RAW)
_q = np.quantile(Xtr, [CFG.winsor_q, 1 - CFG.winsor_q], axis=0)   # one pass, not two
lo_q, hi_q = _q[0].astype(np.float32), _q[1].astype(np.float32)
exempt = np.array([n == TARGET_NAME for n in names])
lo_q[exempt], hi_q[exempt] = -np.inf, np.inf
# a cell cannot be both below lo and above hi, so two counts sum to the union
n_clipped = int(np.count_nonzero(X_raw < lo_q) + np.count_nonzero(X_raw > hi_q))
np.clip(X_raw, lo_q, hi_q, out=X_raw)                 # in-place: no second full-size array
print(f"winsorised {n_clipped:,} cells ({100 * n_clipped / X_raw.size:.3f}%); "
      f"exempt: {[n for n, e in zip(names, exempt) if e]}")

# ---- 3. collinearity pruning ---------------------------------------------
sub = Xtr[:: max(1, n_tr // 200_000)]                 # correlation on a subsample is plenty
C = np.nan_to_num(np.corrcoef(sub, rowvar=False))
drop, kept = set(), []
for i, n in enumerate(names):
    if i == 0:                                        # never drop the target variate
        kept.append(i)
        continue
    dup = next((j for j in kept if abs(C[i, j]) > CFG.collinear_thresh), None)
    if dup is None:
        kept.append(i)
    else:
        drop.add(i)
        print(f"  drop {n:<24} |rho|={abs(C[i, dup]):.4f} vs {names[dup]}")
if len(kept) != X_raw.shape[1]:                       # column fancy-index copies; skip if no-op
    X_raw = X_raw[:, kept]
    Xtr = X_raw[:n_tr]
FEATURE_NAMES = [names[i] for i in kept]
del sub, C
gc.collect()
print(f"collinearity pruning: {len(names)} -> {len(FEATURE_NAMES)} features")

# ---- 4. standardisation, train split only ---------------------------------
# float64 accumulators over a float32 array: storage stays halved, the scaler stays accurate.
mu = Xtr.mean(axis=0, dtype=np.float64)
sd = Xtr.std(axis=0, dtype=np.float64)
sd[sd < 1e-8] = 1.0
X_raw -= mu.astype(np.float32)                        # in-place; `(X_raw - mu)` with a float64
X_raw /= sd.astype(np.float32)                        # mu would upcast the whole matrix
X = X_raw
del X_raw, Xtr
gc.collect()

N_VARIATES = X.shape[1]
TARGET_IDX = FEATURE_NAMES.index(TARGET_NAME)
assert TARGET_IDX == 0, "variate 0 must remain the target series after pruning"

# Scaled space -> raw log return. Every threshold quoted in basis points (the directional
# epsilon, the backtest thresholds) is a raw-return quantity; applying it to standardised
# values would make it ~1/SIGMA_TARGET times too small and it would stop filtering.
SIGMA_TARGET = float(sd[TARGET_IDX])
print(f"target sigma    {SIGMA_TARGET:.6e}   (1 bp = {1e-4 / SIGMA_TARGET:.4f} in scaled units)")

SCALER = {"mean": mu.tolist(), "std": sd.tolist(),
          "winsor_lo": np.where(np.isfinite(lo_q[kept]), lo_q[kept], None).tolist(),
          "winsor_hi": np.where(np.isfinite(hi_q[kept]), hi_q[kept], None).tolist(),
          "fitted_on": {"split": "train", "rows": n_tr, "t_max": str(t_all[n_tr - 1])}}

print(f"\nfeature matrix  X {X.shape}  {X.nbytes / 1024**3:.2f} GB  dtype={X.dtype}")
print(f"target variate  index {TARGET_IDX} = {TARGET_NAME}")
print(f"train-split standardisation check: mean {X[:n_tr].mean():+.2e}, "
      f"std {X[:n_tr].std():.4f}   (should be ~0 and ~1)")
print(f"val+test drift  : mean {X[n_tr:].mean():+.4f}, std {X[n_tr:].std():.4f} "
      f"  (drift here is real distribution shift, not a bug)")
print(f"\npeak RSS so far {peak_rss_gb():.2f} GB   "
      f"(this machine has 14.8 GB total; Kaggle has ~29 GB)")


<div style="background: linear-gradient(135deg, #0f3460, #16213e, #1a1a2e); border-left: 4px solid #533483; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e94560; margin: 0 0 10px 0;">✂️ §9 · Purged &amp; Embargoed Splits</h2>
  <p style="color: #a8dadc; margin: 0 0 8px 0;">Random splits and k-fold cross-validation are <strong>invalid</strong> here: they train on the future. But a plain chronological cut is not enough either, and the reason is easy to miss.</p>  <p style="color: #a8dadc; margin: 0 0 8px 0;">A training window ending at the boundary carries a label reaching <em>H</em> minutes past it, overlapping the first validation window's inputs. And with <em>L</em> = 1440 the validation window's own inputs reach 1440 minutes back into training data. Both directions leak.</p>  <p style="color: #a8dadc; margin: 0 0 8px 0;">The fix (López de Prado, ch. 7) is <strong>purging + embargo</strong>. A window qualifies for a split only if its entire footprint — inputs <em>and</em> label, <code>[s, s+L+H)</code> — lies inside that split's range, and the ranges are separated by a gap of <code>purge + embargo = H + (L + H + safety)</code> minutes. That makes the no-overlap property provable rather than hoped for, and the assertion below proves it.</p>
</div>

In [ ]:
L, H = CFG.seq_len, CFG.pred_len
GAP = CFG.split_gap_min
t_np = t_all.to_numpy()


def _idx(when: datetime) -> int:
    return int(np.searchsorted(t_np, np.datetime64(when.replace(tzinfo=None), "us")))


bounds = {
    "train": (0, _idx(ts(CFG.train_end))),
    "val":   (_idx(ts(CFG.train_end)) + GAP, _idx(ts(CFG.val_end))),
    "test":  (_idx(ts(CFG.val_end)) + GAP, _idx(ts(CFG.test_end))),
}

# ---- window validity mask -------------------------------------------------
# A window starting at s covers rows [s, s+L+H). It is rejected if it spans more
# synthetic BTC minutes than tolerated, or if gold is stale beyond the cap.
def unscale(col: str) -> np.ndarray:
    """Recover a feature's pre-standardisation values (mu/sd are indexed by pruned position)."""
    i = FEATURE_NAMES.index(col)
    return X[:, i].astype(np.float64) * sd[i] + mu[i]


synth_raw = ((unscale("btc_is_synthetic") > 0.5).astype(np.int32)
             if "btc_is_synthetic" in FEATURE_NAMES else np.zeros(T, np.int32))
cs_synth = np.concatenate([[0], np.cumsum(synth_raw)])

stale_min = (np.expm1(np.clip(unscale("gold_staleness"), 0, 20))
             if "gold_staleness" in FEATURE_NAMES else np.zeros(T))
bad_stale = (stale_min > CFG.gold_staleness_cap_min).astype(np.int32)
cs_stale = np.concatenate([[0], np.cumsum(bad_stale)])

span = L + H
starts_all = np.arange(0, T - span + 1, dtype=np.int64)
n_synth_in = cs_synth[starts_all + span] - cs_synth[starts_all]
n_stale_in = cs_stale[starts_all + span] - cs_stale[starts_all]
valid = (n_synth_in <= CFG.max_synth_run_in_window) & (n_stale_in == 0)
print(f"candidate windows {len(starts_all):,}")
print(f"  rejected: synthetic BTC minutes  {int((n_synth_in > CFG.max_synth_run_in_window).sum()):,}")
print(f"  rejected: gold staleness > {CFG.gold_staleness_cap_min}m  {int((n_stale_in > 0).sum()):,}")
print(f"  valid                            {int(valid.sum()):,}")

SPLITS: dict[str, np.ndarray] = {}
for name, (lo, hi) in bounds.items():
    stride = CFG.train_stride if name == "train" else 1
    sel = valid & (starts_all >= lo) & (starts_all + span - 1 <= hi)
    s = starts_all[sel]
    s = s[(s - (s[0] if len(s) else 0)) % stride == 0] if stride > 1 else s
    SPLITS[name] = s
    if len(s):
        print(f"\n{name:<6} {len(s):>9,} windows  stride={stride}")
        print(f"       inputs  {t_np[s[0]]}  ->  {t_np[s[-1] + L - 1]}")
        print(f"       labels  {t_np[s[0] + L]}  ->  {t_np[s[-1] + span - 1]}")

# ---- the split assertion --------------------------------------------------
for a, b in (("train", "val"), ("val", "test")):
    if len(SPLITS[a]) and len(SPLITS[b]):
        end_a = SPLITS[a][-1] + span - 1        # last row the earlier split touches at all
        start_b = SPLITS[b][0]                  # first row the later split touches at all
        assert start_b > end_a, f"{a}/{b} window footprints overlap"
        print(f"\n{a} -> {b}: {start_b - end_a:,} min separation "
              f"(required >= {GAP - span:,} after footprint accounting)  OK")

<div style="background: linear-gradient(90deg, #1b1b2f, #2d2b55); border-left: 4px solid #a855f7; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #d8b4fe; margin: 0 0 10px 0;">🪟 §10 · Windowing &amp; Dataset</h2>
  <p style="color: #c4b5fd; margin: 0 0 8px 0;">A sample is <code>(X, y)</code> with <code>X ∈ R^{L×N}</code> and <code>y ∈ R^{H}</code>. Three decisions matter here:</p>  <ul style="color: #c4b5fd; margin: 0; padding-left: 20px;">
    <li><strong>Stride.</strong> At 1-minute granularity, stride 1 yields ~4.4M windows that share <em>L−1</em> of <em>L</em> rows with their neighbours — near-duplicates that inflate epoch time without adding information. Training uses stride 5; validation and test use <strong>stride 1</strong> so evaluation covers every timestamp.</li>
    <li><strong>The validity mask is precomputed once</strong> and indexed into. Filtering inside <code>__getitem__</code> would re-do the same work every epoch, in every worker.</li>
    <li><strong>The label is a view into the feature matrix, not a copy.</strong> Because variate 0 <em>is</em> the scaled one-minute log return, <code>y = X[s+L : s+L+H, 0]</code>. Zero extra memory, and the label cannot drift out of sync with the input scale.</li>
  </ul>  <p style="color: #c4b5fd; margin: 0 0 8px 0;"><strong>Windows guard:</strong> on Windows, DataLoader workers re-import the module, so a script version of this needs <code>if __name__ == "__main__":</code>. Inside a notebook on Kaggle (Linux) workers fork and share <code>X</code> copy-on-write, so no copy per worker is made.</p>
</div>

In [ ]:
class WindowDataset(Dataset):
    """Sliding windows over the shared feature matrix.

    X : (T, N) float32, already scaled with training-split statistics.
    A sample at start s is (X[s:s+L], X[s+L:s+L+H, target_idx]).
    """

    def __init__(self, X: np.ndarray, starts: np.ndarray, L: int, H: int,
                 target_idx: int = 0, variate_mask: np.ndarray | None = None,
                 full_target: bool = False):
        self.X, self.starts, self.L, self.H = X, starts, L, H
        self.target_idx = target_idx
        self.variate_mask = variate_mask      # used by the ablation table
        # full_target yields every variate's future block, which is what the auxiliary
        # multi-task objective (project_target_only=False) needs. Same memory pattern,
        # since it is still a view into the shared matrix.
        self.full_target = full_target

    def __len__(self) -> int:
        return len(self.starts)

    def __getitem__(self, i: int):
        s = int(self.starts[i])
        x = self.X[s: s + self.L]
        fut = self.X[s + self.L: s + self.L + self.H]
        y = fut if self.full_target else fut[:, self.target_idx]
        x = torch.from_numpy(np.ascontiguousarray(x))
        if self.variate_mask is not None:
            x = x * torch.from_numpy(self.variate_mask)
        return x, torch.from_numpy(np.ascontiguousarray(y))


def make_loader(split: str, batch_size: int, shuffle: bool | None = None,
                variate_mask: np.ndarray | None = None, cap: int | None = None) -> DataLoader:
    starts = SPLITS[split]
    if cap and len(starts) > cap:                     # evenly thinned, never head-truncated
        starts = starts[:: math.ceil(len(starts) / cap)]
    ds = WindowDataset(X, starts, L, H, TARGET_IDX, variate_mask,
                       full_target=not CFG.project_target_only)
    shuffle = (split == "train") if shuffle is None else shuffle
    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle,
        num_workers=CFG.num_workers, pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=CFG.num_workers > 0, prefetch_factor=4 if CFG.num_workers else None,
        drop_last=(split == "train"), worker_init_fn=seed_worker,
    )


def eval_row_step(loader: DataLoader) -> int:
    """Minutes between consecutive evaluation windows.

    `cap` thins an eval split by keeping every k-th window, so its predictions are k
    minutes apart, not 1. Every annualised quantity downstream depends on knowing k.
    """
    s = loader.dataset.starts
    return int(s[1] - s[0]) if len(s) > 1 else 1


_xb, _yb = WindowDataset(X, SPLITS["train"], L, H, TARGET_IDX,
                         full_target=not CFG.project_target_only)[0]
print(f"sample x {tuple(_xb.shape)} {_xb.dtype}   y {tuple(_yb.shape)} {_yb.dtype}")
print(f"train windows {len(SPLITS['train']):,} @ stride {CFG.train_stride}  "
      f"-> {len(SPLITS['train']) // CFG.batch_size:,} batches/epoch")
print(f"eval cap {CFG.eval_max_windows:,} windows per split")

<div style="background: linear-gradient(90deg, #1b1b2f, #2d2b55); border-left: 4px solid #a855f7; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #d8b4fe; margin: 0 0 10px 0;">🧠 §11 · The Inverted Transformer</h2>
  <p style="color: #c4b5fd; margin: 0 0 8px 0;">A vanilla time-series Transformer embeds <strong>each timestamp</strong> as a token: <code>X ∈ R^{L×N}</code> becomes <em>L</em> tokens of width <em>N</em>. iTransformer <strong>inverts the axes</strong> — each variate's <em>entire lookback series</em> becomes one token, giving <em>N</em> tokens of width <code>d_model</code>. Four consequences, all of which matter specifically for this project:</p>  <ul style="color: #c4b5fd; margin: 0; padding-left: 20px;">
    <li><strong>Attention costs <code>O(N²·d)</code>, independent of <em>L</em>.</strong> With ~60 variates and <em>L</em> = 1440, attention is trivially cheap while the model still sees a full day of minutes. A time-token Transformer at <em>L</em> = 1440 needs a 1440×1440 attention matrix per head. <strong>This is the decisive architectural argument here.</strong></li>
    <li><strong>Attention now models cross-variate structure</strong> — precisely the BTC ↔ gold ↔ USD ↔ macro relationships we are trying to learn — while the feed-forward network models each variate's temporal dynamics.</li>
    <li><strong>Heterogeneous variates stop being jammed together.</strong> Time-token embedding forces a 1-minute BTC return and a monthly CPI z-score into one token, which is physically meaningless. Inversion keeps them in separate tokens.</li>
    <li><strong>The attention matrix is directly interpretable</strong> as a learned variate-correlation map — section 14 reads it to show which exogenous inputs the model actually uses.</li>
  </ul>  <p style="color: #c4b5fd; margin: 0 0 8px 0;"><strong>No causal mask is applied in attention, and that is correct.</strong> Masking belongs on the <em>time</em> axis; this attention runs over the <em>variate</em> axis, where all tokens are contemporaneous by construction. Causality is enforced upstream — in feature construction and windowing — which is exactly why section 12's gates exist.</p>
</div>

In [ ]:
class DataEmbedding_inverted(nn.Module):
    """Embed each variate's whole lookback series into one token: (B, L, N) -> (B, N, d_model).

    No positional encoding: variate order is arbitrary, and temporal order is already
    carried inside each token by the linear projection over L.
    """

    def __init__(self, seq_len: int, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.value_embedding = nn.Linear(seq_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.value_embedding(x.permute(0, 2, 1)))


class FullAttention(nn.Module):
    """Scaled dot-product attention over the VARIATE axis."""

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1,
                 output_attention: bool = False):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout_p = dropout
        self.output_attention = output_attention

    def forward(self, x: torch.Tensor):
        B, N, _ = x.shape
        q, k, v = (proj(x).view(B, N, self.n_heads, self.d_head).transpose(1, 2)
                   for proj in (self.q_proj, self.k_proj, self.v_proj))
        attn = None
        if self.output_attention:
            # explicit path: needed for interpretation, and for ONNX export safety
            scores = (q @ k.transpose(-2, -1)) * (self.d_head ** -0.5)
            attn = scores.softmax(dim=-1)
            out = attn @ v
        else:
            out = F.scaled_dot_product_attention(
                q, k, v, dropout_p=self.dropout_p if self.training else 0.0)
        return self.out_proj(out.transpose(1, 2).reshape(B, N, -1)), attn


class EncoderLayer(nn.Module):
    """Attention across variates, feed-forward per variate.

    `norm_style="post"` reproduces thuml/iTransformer exactly: attention residual,
    then norm1, then the feed-forward residual, then norm2. The official code writes
    the feed-forward as two Conv1d(kernel_size=1) layers, which is arithmetically
    identical to two Linear layers over the last axis, with dropout after BOTH.

    `norm_style="pre"` is the now-more-common pre-norm variant. It trains more stably
    at depth, but it is NOT what the paper's reported numbers came from, so "post"
    is the default and "pre" is an explicit, logged deviation.
    """

    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1,
                 activation: str = "gelu", output_attention: bool = False,
                 norm_style: str = "post"):
        super().__init__()
        self.attn = FullAttention(d_model, n_heads, dropout, output_attention)
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.norm_style = norm_style
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU() if activation == "gelu" else nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),          # official applies dropout after conv2 as well
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor):
        if self.norm_style == "post":
            h, attn = self.attn(x)
            x = x + self.dropout(h)
            y = x = self.norm1(x)
            return self.norm2(x + self.ffn(y)), attn
        h, attn = self.attn(self.norm1(x))
        x = x + self.dropout(h)
        x = x + self.ffn(self.norm2(x))
        return x, attn


class iTransformer(nn.Module):
    """Inverted Transformer for BTC 1-minute forecasting.

    Input  : (B, L, N), variate `target_index` is the scaled 1-minute log return.
    Output : (B, H, 1) in the same scaled log-return space as that variate.
    """

    def __init__(self, seq_len: int, pred_len: int, n_variates: int, d_model: int = 512,
                 n_heads: int = 8, e_layers: int = 3, d_ff: int = 2048, dropout: float = 0.1,
                 activation: str = "gelu", use_norm: bool = True, target_index: int = 0,
                 project_target_only: bool = True, output_attention: bool = False,
                 use_linear_skip: bool = True, n_out: int = 1, norm_style: str = "post"):
        super().__init__()
        self.pred_len, self.use_norm = pred_len, use_norm
        self.target_index, self.project_target_only = target_index, project_target_only
        self.output_attention = output_attention
        self.n_out = n_out                       # >1 for quantile heads

        self.enc_embedding = DataEmbedding_inverted(seq_len, d_model, dropout)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout, activation, output_attention,
                         norm_style)
            for _ in range(e_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.projector = nn.Linear(d_model, pred_len * n_out)

        # DLinear-style residual skip, zero-initialised so training starts as pure
        # iTransformer and the linear path is earned rather than assumed.
        self.linear_skip = nn.Linear(seq_len, pred_len * n_out) if use_linear_skip else None
        if self.linear_skip is not None:
            nn.init.zeros_(self.linear_skip.weight)
            nn.init.zeros_(self.linear_skip.bias)

    def set_output_attention(self, flag: bool) -> None:
        self.output_attention = flag
        for lyr in self.layers:
            lyr.attn.output_attention = flag

    def forward(self, x: torch.Tensor, return_attention: bool = False):
        skip_in = x[:, :, self.target_index]              # pre-normalisation, scaled space

        if self.use_norm:
            # RevIN-style instance normalisation. .detach() is deliberate:
            # normalisation statistics must not receive gradients.
            means = x.mean(dim=1, keepdim=True).detach()
            x = x - means
            stdev = torch.sqrt(x.var(dim=1, keepdim=True, unbiased=False) + 1e-5).detach()
            x = x / stdev

        h = self.enc_embedding(x)
        attns = []
        for lyr in self.layers:
            h, a = lyr(h)
            if a is not None:
                attns.append(a)
        h = self.norm(h)

        B, N, _ = h.shape
        out = self.projector(h).view(B, N, self.pred_len, self.n_out)
        out = out.permute(0, 2, 1, 3)                                  # (B, H, N, n_out)

        if self.use_norm:
            out = out * stdev[:, 0, :].view(B, 1, N, 1) + means[:, 0, :].view(B, 1, N, 1)

        if self.project_target_only:
            out = out[:, :, self.target_index, :]                      # (B, H, n_out)
            if self.linear_skip is not None:
                out = out + self.linear_skip(skip_in).view(B, self.pred_len, self.n_out)
        else:
            # Auxiliary multi-task objective: every variate is forecast and the loss sees
            # all of them (the dataset yields the full future block in this mode). The
            # linear skip is a target-only path, so it is added to the target slice alone.
            if self.linear_skip is not None:
                sk = torch.zeros_like(out)
                sk[:, :, self.target_index, :] = self.linear_skip(skip_in).view(
                    B, self.pred_len, self.n_out)
                out = out + sk                                         # (B, H, N, n_out)

        return (out, attns) if return_attention else out


def build_model(cfg: Config, n_variates: int, n_out: int = 1) -> iTransformer:
    return iTransformer(
        seq_len=cfg.seq_len, pred_len=cfg.pred_len, n_variates=n_variates,
        d_model=cfg.d_model, n_heads=cfg.n_heads, e_layers=cfg.e_layers, d_ff=cfg.d_ff,
        dropout=cfg.dropout, activation=cfg.activation, use_norm=cfg.use_norm,
        target_index=TARGET_IDX, project_target_only=cfg.project_target_only,
        use_linear_skip=cfg.use_linear_skip, n_out=n_out, norm_style=cfg.norm_style,
    )


_m = build_model(CFG, N_VARIATES)
_n_par = sum(p.numel() for p in _m.parameters())
print(f"iTransformer   L={CFG.seq_len}  H={CFG.pred_len}  N={N_VARIATES}  "
      f"d_model={CFG.d_model}  layers={CFG.e_layers}")
print(f"parameters     {_n_par:,}  ({_n_par * 4 / 1024**2:.1f} MB fp32)")
with torch.no_grad():
    print(f"forward check  {tuple(_m(torch.randn(4, CFG.seq_len, N_VARIATES)).shape)}  "
          f"(expected (4, {CFG.pred_len}, 1))")
del _m

<div style="background: linear-gradient(90deg, #1b1b2f, #2d2b55); border-left: 4px solid #a855f7; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #d8b4fe; margin: 0 0 10px 0;">📚 Provenance — every choice traced to its source</h2>
  <p style="color: #c4b5fd; margin: 0 0 8px 0;">Where this notebook follows a published method, it follows the <strong>reference implementation</strong>, not a paraphrase of the paper. Where it deviates, the deviation is deliberate and recorded here rather than left for a reader to discover.</p>  <p style="color: #c4b5fd; margin: 0 0 8px 0;"><strong>Faithful to the reference code:</strong></p>  <ul style="color: #c4b5fd; margin: 0; padding-left: 20px;">
    <li><code>DataEmbedding_inverted</code> = <code>Linear(seq_len, d_model)</code> over the transposed series, no positional encoding — variate order is arbitrary (Liu et al., ICLR 2024; <code>thuml/iTransformer</code>).</li>
    <li><strong>Post-norm encoder block</strong>: attention residual → <code>norm1</code> → feed-forward residual → <code>norm2</code>, with dropout after <em>both</em> feed-forward projections. The official code writes the feed-forward as two <code>Conv1d(kernel_size=1)</code> layers, which is arithmetically identical to two <code>Linear</code> layers over the last axis.</li>
    <li><strong>Instance normalisation arithmetic verified line-for-line</strong> against the official <code>forecast()</code>: denormalise with <code>stdev[:, 0, :]</code> and <code>means[:, 0, :]</code>, statistics <code>.detach()</code>ed so they never receive gradients (Kim et al., ICLR 2022 — RevIN; Liu et al., NeurIPS 2022 — Non-stationary Transformers).</li>
    <li><strong>TimeXer</strong>: patch tokens + one learnable global endogenous token, cross-attending to exogenous variate tokens, post-norm, flatten head over <code>(patch_num+1)·d_model</code> (Wang et al., NeurIPS 2024; <code>thuml/Time-Series-Library</code>).</li>
    <li><strong>PatchTST</strong>: channel-independent patching with stride <code>patch_len/2</code> (Nie et al., ICLR 2023). Channel independence means the target's forecast cannot use any other variate — the clean control for whether cross-variate attention earns its place.</li>
    <li><strong>DLinear</strong>: moving-average decomposition, kernel 25, separate trend and seasonal linear heads (Zeng et al., AAAI 2023).</li>
    <li><strong>Purged &amp; embargoed splits</strong>, fractional differentiation by minimum <em>d</em> passing ADF, and the deflated Sharpe ratio (López de Prado, 2018, chs. 5 &amp; 7; Bailey &amp; López de Prado, 2014).</li>
    <li><strong>Diebold–Mariano</strong> with Newey–West HAC variance over <em>h</em>−1 lags — zero lags at <em>h</em>=1, since an <em>h</em>-step loss differential is MA(<em>h</em>−1) — and the Harvey–Leybourne–Newbold small-sample correction (Diebold &amp; Mariano, 1995; HLN, 1997).</li>
  </ul>  <p style="color: #c4b5fd; margin: 0 0 8px 0;"><strong>Deliberate deviations, and why:</strong></p>  <ul style="color: #c4b5fd; margin: 0; padding-left: 20px;">
    <li><strong>The target is the first variate, and the label is a slice of the feature matrix.</strong> The reference implementations forecast every variate; here only the BTC head is projected, because predicting a monthly CPI z-score 60 minutes ahead wastes capacity. <code>project_target_only=False</code> turns the reference behaviour back on as an auxiliary multi-task objective — an ablation, not an assumption.</li>
    <li><strong>Zero-initialised DLinear residual skip</strong> on the iTransformer output. Not in the paper; it starts training as pure iTransformer, so the linear path has to be earned. Toggle with <code>use_linear_skip</code>.</li>
    <li><strong>Non-affine instance normalisation.</strong> Original RevIN carries learnable affine <em>γ</em>, <em>β</em>; iTransformer's <code>use_norm</code> — reproduced here — omits them. That follows the iTransformer code, not the RevIN paper.</li>
    <li><strong><code>norm_style='pre'</code> is available but not the default.</strong> Pre-norm trains more stably at depth, but the paper's reported numbers come from post-norm, so switching is an explicit, logged choice.</li>
    <li><strong>Temporal features are ordinary variates.</strong> The official code can append covariates as extra tokens via <code>x_mark</code>; here calendar features enter the same way every other variate does, which keeps one code path and one manifest.</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #1b1b2f, #2d2b55); border-left: 4px solid #a855f7; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #d8b4fe; margin: 0 0 10px 0;">📏 Baselines that must be beaten</h2>
  <p style="color: #c4b5fd; margin: 0 0 8px 0;">At a 1-minute horizon the naive baseline is <strong>brutally strong</strong> — BTC log returns are close to a martingale difference sequence, so <code>ŷ = 0</code> is genuinely hard to beat. Reporting a model number without these side by side is not an acceptable output.</p>  <ul style="color: #c4b5fd; margin: 0; padding-left: 20px;">
    <li><strong>Random walk</strong> — <code>ŷ = 0</code> in return space.</li>
    <li><strong>Historical mean</strong> — trailing mean return over the lookback.</li>
    <li><strong>AR(p) / direct linear</strong> — closed-form ridge from the last <em>p</em> returns to the next <em>H</em>.</li>
    <li><strong>DLinear</strong> — moving-average trend/seasonal decomposition, two linear layers. Frequently competitive with Transformers on long-horizon forecasting; take it seriously.</li>
    <li><strong>Vanilla time-token Transformer</strong> — isolates the contribution of <em>inversion</em> itself, holding everything else fixed.</li>
    <li><strong>PatchTST</strong> — channel-independent patching. Isolates whether <em>cross-variate</em> attention contributes anything beyond good temporal encoding.</li>
    <li><strong>TimeXer</strong> — the head-to-head CLAUDE.md §10.3 asks for, under identical splits, scaler, seed and window mask. It is built for this exact problem shape, so if the exogenous series carry signal, this is the architecture most likely to find it.</li>
    <li><strong>iTransformer, BTC-only</strong> — isolates the contribution of the <em>exogenous</em> variables. This one lives in the ablation table in section 15.</li>
  </ul>
</div>

In [ ]:
class NaiveZero(nn.Module):
    """Random walk: the best forecast of the next log return is zero."""

    def __init__(self, pred_len: int):
        super().__init__()
        self.pred_len = pred_len
        self._p = nn.Parameter(torch.zeros(1), requires_grad=False)

    def forward(self, x):
        return torch.zeros(x.shape[0], self.pred_len, 1, device=x.device, dtype=x.dtype)


class HistMean(nn.Module):
    """Trailing mean of the target variate over the lookback, held flat across H."""

    def __init__(self, pred_len: int, target_index: int = 0):
        super().__init__()
        self.pred_len, self.target_index = pred_len, target_index
        self._p = nn.Parameter(torch.zeros(1), requires_grad=False)

    def forward(self, x):
        return x[:, :, self.target_index].mean(dim=1, keepdim=True).unsqueeze(-1).expand(
            -1, self.pred_len, 1)


class DirectLinear(nn.Module):
    """AR(p) as a direct multi-horizon map: last p returns -> next H returns."""

    def __init__(self, p: int, pred_len: int, target_index: int = 0):
        super().__init__()
        self.p, self.pred_len, self.target_index = p, pred_len, target_index
        self.fc = nn.Linear(p, pred_len)
        nn.init.zeros_(self.fc.weight); nn.init.zeros_(self.fc.bias)

    def forward(self, x):
        return self.fc(x[:, -self.p:, self.target_index]).unsqueeze(-1)


class DLinear(nn.Module):
    """LTSF-Linear: moving-average decomposition into trend and seasonal, one linear head each."""

    def __init__(self, seq_len: int, pred_len: int, kernel: int = 25, target_index: int = 0):
        super().__init__()
        self.kernel, self.target_index, self.pred_len = kernel, target_index, pred_len
        self.trend = nn.Linear(seq_len, pred_len)
        self.seasonal = nn.Linear(seq_len, pred_len)

    def forward(self, x):
        s = x[:, :, self.target_index]
        pad = self.kernel // 2
        t = F.avg_pool1d(F.pad(s.unsqueeze(1), (pad, self.kernel - 1 - pad), mode="replicate"),
                         self.kernel, stride=1).squeeze(1)
        return (self.trend(t) + self.seasonal(s - t)).unsqueeze(-1)


class VanillaTransformer(nn.Module):
    """Time-token Transformer: one token per timestamp. The control for `inversion`.

    Attention here is L x L rather than N x N, which is exactly the cost iTransformer
    avoids. Held to the same L, d_model and depth so the comparison isolates the axis choice.
    """

    def __init__(self, seq_len: int, pred_len: int, n_variates: int, d_model: int = 128,
                 n_heads: int = 4, e_layers: int = 2, d_ff: int = 256, dropout: float = 0.1):
        super().__init__()
        self.pred_len = pred_len
        self.inp = nn.Linear(n_variates, d_model)
        pe = torch.zeros(seq_len, d_model)
        pos = torch.arange(seq_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
        enc = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout, "gelu",
                                         batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(enc, e_layers)
        self.head = nn.Linear(d_model, pred_len)

    def forward(self, x):
        h = self.enc(self.inp(x) + self.pe[:, : x.shape[1]])
        return self.head(h[:, -1]).unsqueeze(-1)


class PositionalEmbedding(nn.Module):
    """Sinusoidal positional encoding over patch positions (Vaswani et al., as used by TSLib)."""

    def __init__(self, d_model: int, max_len: int = 4096):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.pe[:, : x.size(1)]


class EnEmbedding(nn.Module):
    """TimeXer endogenous embedding: patch tokens plus one learnable global token.

    (B, 1, L) -> (B, patch_num + 1, d_model). The global token is the bridge that
    later cross-attends to the exogenous variate tokens.
    """

    def __init__(self, d_model: int, patch_len: int, dropout: float):
        super().__init__()
        self.patch_len = patch_len
        self.value_embedding = nn.Linear(patch_len, d_model, bias=False)
        self.glb_token = nn.Parameter(torch.randn(1, 1, 1, d_model))
        self.position_embedding = PositionalEmbedding(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor):
        B, n_vars, _ = x.shape
        glb = self.glb_token.repeat(B, n_vars, 1, 1)
        x = x.unfold(dimension=-1, size=self.patch_len, step=self.patch_len)
        x = x.reshape(B * n_vars, x.shape[2], self.patch_len)
        x = self.value_embedding(x) + self.position_embedding(x)
        x = x.reshape(B, n_vars, -1, x.shape[-1])
        x = torch.cat([x, glb], dim=2)
        return self.dropout(x.reshape(B * n_vars, x.shape[2], x.shape[3])), n_vars


class TimeXerLayer(nn.Module):
    """Self-attention over endogenous patches + cross-attention from the global token
    to the exogenous variate tokens. Post-norm, matching thuml/Time-Series-Library."""

    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float, activation: str):
        super().__init__()
        self.self_attention = FullAttention(d_model, n_heads, dropout)
        self.cross_attention = nn.MultiheadAttention(d_model, n_heads, dropout, batch_first=True)
        self.norm1, self.norm2, self.norm3 = (nn.LayerNorm(d_model) for _ in range(3))
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU() if activation == "gelu" else nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, cross: torch.Tensor):
        x = x + self.dropout(self.self_attention(x)[0])
        x = self.norm1(x)

        x_glb_ori = x[:, -1, :].unsqueeze(1)                 # (B*n_vars, 1, d_model)
        B, _, D = cross.shape
        x_glb = x_glb_ori.reshape(B, -1, D)                  # (B, n_vars, d_model)
        attn_out, _ = self.cross_attention(x_glb, cross, cross)
        x_glb = self.norm2(x_glb_ori + self.dropout(attn_out.reshape(-1, D).unsqueeze(1)))

        y = x = torch.cat([x[:, :-1, :], x_glb], dim=1)
        return self.norm3(x + self.ffn(y))


class TimeXer(nn.Module):
    """TimeXer (Wang et al., NeurIPS 2024) for one endogenous target plus exogenous covariates.

    This is the architecture built for exactly our setting: patch-level tokens for the
    endogenous series, variate-level tokens for the exogenous series, and a global
    endogenous token bridging them via cross-attention. iTransformer, by contrast,
    treats the target as just another variate token.
    """

    def __init__(self, seq_len: int, pred_len: int, n_variates: int, d_model: int = 512,
                 n_heads: int = 8, e_layers: int = 3, d_ff: int = 2048, dropout: float = 0.1,
                 activation: str = "gelu", patch_len: int = 60, target_index: int = 0,
                 use_norm: bool = True):
        super().__init__()
        self.pred_len, self.target_index, self.use_norm = pred_len, target_index, use_norm
        patch_len = min(patch_len, seq_len)
        while seq_len % patch_len:                 # unfold requires an exact tiling
            patch_len -= 1
        self.patch_len = patch_len
        patch_num = seq_len // patch_len

        self.en_embedding = EnEmbedding(d_model, patch_len, dropout)
        self.ex_embedding = DataEmbedding_inverted(seq_len, d_model, dropout)
        self.layers = nn.ModuleList([
            TimeXerLayer(d_model, n_heads, d_ff, dropout, activation) for _ in range(e_layers)
        ])
        self.head = nn.Linear(d_model * (patch_num + 1), pred_len)

    def forward(self, x: torch.Tensor):
        if self.use_norm:
            means = x.mean(dim=1, keepdim=True).detach()
            x = x - means
            stdev = torch.sqrt(x.var(dim=1, keepdim=True, unbiased=False) + 1e-5).detach()
            x = x / stdev

        B = x.shape[0]
        en = x[:, :, self.target_index].unsqueeze(1)                       # (B, 1, L)
        ex = torch.cat([x[:, :, : self.target_index],
                        x[:, :, self.target_index + 1:]], dim=2)
        en, _ = self.en_embedding(en)                                      # (B, P+1, d)
        cross = self.ex_embedding(ex)                                      # (B, N-1, d)
        for lyr in self.layers:
            en = lyr(en, cross)
        out = self.head(en.reshape(B, -1)).unsqueeze(-1)                   # (B, H, 1)

        if self.use_norm:
            out = (out * stdev[:, 0, self.target_index].view(B, 1, 1)
                   + means[:, 0, self.target_index].view(B, 1, 1))
        return out


class PatchTST(nn.Module):
    """PatchTST (Nie et al., ICLR 2023): channel-independent patching.

    Channel independence means each variate is encoded with shared weights and never
    mixes with the others, so the target's forecast is a function of the target channel
    alone. That makes this the clean control for "does cross-variate attention help at
    all" - it is patching without any cross-variate information.
    """

    def __init__(self, seq_len: int, pred_len: int, d_model: int = 128, n_heads: int = 8,
                 e_layers: int = 3, d_ff: int = 256, dropout: float = 0.1,
                 patch_len: int = 60, stride: int | None = None, target_index: int = 0,
                 use_norm: bool = True):
        super().__init__()
        self.pred_len, self.target_index, self.use_norm = pred_len, target_index, use_norm
        patch_len = min(patch_len, seq_len)
        stride = stride or patch_len // 2
        self.patch_len, self.stride = patch_len, stride
        patch_num = (seq_len - patch_len) // stride + 1

        self.value_embedding = nn.Linear(patch_len, d_model, bias=False)
        self.position_embedding = PositionalEmbedding(d_model)
        self.dropout = nn.Dropout(dropout)
        enc = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout, "gelu",
                                         batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(enc, e_layers)
        self.head = nn.Linear(d_model * patch_num, pred_len)

    def forward(self, x: torch.Tensor):
        s = x[:, :, self.target_index]                                     # (B, L)
        if self.use_norm:
            m = s.mean(dim=1, keepdim=True).detach()
            sd = torch.sqrt(s.var(dim=1, keepdim=True, unbiased=False) + 1e-5).detach()
            s = (s - m) / sd
        p = s.unfold(dimension=-1, size=self.patch_len, step=self.stride)  # (B, P, patch_len)
        h = self.dropout(self.value_embedding(p) + self.position_embedding(p))
        h = self.enc(h)
        out = self.head(h.reshape(h.shape[0], -1)).unsqueeze(-1)
        if self.use_norm:
            out = out * sd.unsqueeze(-1) + m.unsqueeze(-1)
        return out


print("models defined: iTransformer, TimeXer, PatchTST, DLinear, VanillaTransformer, "
      "DirectLinear(AR), HistMean, NaiveZero")

<div style="background: linear-gradient(90deg, #0d1b2a, #1b263b); border-left: 4px solid #00b4d8; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #90e0ef; margin: 0 0 10px 0;">🚦 §12 · Sanity Gates</h2>
  <p style="color: #ade8f4; margin: 0 0 8px 0;"><strong>These run before training, and training does not count as valid unless all five pass.</strong> Each one catches a specific failure mode that otherwise surfaces as a suspiciously good number hours later.</p>  <ul style="color: #ade8f4; margin: 0; padding-left: 20px;">
    <li><strong>Shift test</strong> — shifting the input forward by <em>k</em> must shift every feature forward by exactly <em>k</em>. Anything centred, forward-looking, or accidentally reversed fails here.</li>
    <li><strong>Split test</strong> — no window footprint may cross a split boundary, and the embargo must actually be present.</li>
    <li><strong>Scaler test</strong> — scaler statistics must be reproducible from training rows alone; the hash is checked, not eyeballed.</li>
    <li><strong>Leakage test</strong> — replace the target with pure noise and retrain briefly. Validation performance <em>must</em> collapse to baseline. If the model still 'predicts', the features contain the target.</li>
    <li><strong>Overfit-a-single-batch</strong> — the model must drive loss to near zero on 8 samples in 200 steps. If it cannot, the architecture or the data plumbing is broken and a multi-hour run would only waste the GPU.</li>
  </ul>  <p style="color: #ade8f4; margin: 0 0 8px 0;">The random-walk baseline is also computed and logged <strong>first</strong>, before any training, so there is never a temptation to reinterpret it afterwards.</p>
</div>

In [ ]:
GATES: dict[str, bool] = {}


def _synthetic_master(n: int, seed: int = 0) -> pl.DataFrame:
    """A synthetic master frame with the same column contract as the real one."""
    rng = np.random.default_rng(seed)
    t = pl.datetime_range(datetime(2022, 1, 1, tzinfo=UTC),
                          datetime(2022, 1, 1, tzinfo=UTC) + timedelta(minutes=n - 1),
                          interval="1m", time_zone="UTC", eager=True)
    close = 30_000 * np.exp(np.cumsum(rng.normal(0, 2e-4, n)))
    df = pl.DataFrame({
        "t": t, "btc_open": close, "btc_high": close * 1.0004, "btc_low": close * 0.9996,
        "btc_close": close, "btc_volume": rng.lognormal(0, 1, n), "btc_is_synthetic": np.zeros(n),
    })
    exo = ["gold_logret_1", "gold_logret_5", "gold_logret_60", "gold_logret_1440", "gold_rv_60",
           "gold_staleness", "gold_market_open", "gold_gap_decay", "xa_corr_60", "xa_corr_1440",
           "xa_beta_1440", "xa_spread_z", "xa_gold_leads_btc", "xa_btc_leads_gold",
           "dxy_logret_1d", "dxy_logret_5d", "dxy_logret_21d", "dxy_pctrank_252", "dxy_age_days",
           "macro_yield_curve", "macro_real_rate", "macro_m2_impulse", "macro_bs_impulse",
           "macro_age_days", *[f"macro_pc{i + 1}" for i in range(K)]]
    return df.with_columns([pl.Series(c, rng.normal(0, 1, n)) for c in exo])


def gate_shift_test(k: int = 37, n: int = 20_000) -> bool:
    """Two independent causality checks on build_features.

    1a) Shift equivariance - delaying the whole input by k minutes must delay every
        feature by exactly k (CLAUDE.md 7.6.1).
    1b) Future-perturbation invariance - the stronger property. Corrupting the input
        from row j onward must leave every feature at rows < j bit-identical. Any
        centred window, forward fill, or reversed index fails this immediately.
    """
    base = _synthetic_master(n)
    a, _ = build_features(base, CFG, FRAC_D)
    cols = [c for c in a.columns if c not in ("t", "btc_close") and not c.startswith("t_")]
    A = a.select(cols).to_numpy()

    # --- 1a: re-time the same values by +k minutes -------------------------
    b, _ = build_features(
        base.head(n - k).with_columns(pl.col("t") + pl.duration(minutes=k)), CFG, FRAC_D)
    B = b.select(cols).to_numpy()
    m = np.isfinite(A[: n - k]) & np.isfinite(B)
    err_shift = float(np.abs(A[: n - k] - B)[m].max()) if m.any() else np.inf
    ok_a = err_shift < 1e-8
    print(f"  1a shift equivariance      max|delta| = {err_shift:.3e}   "
          f"{'PASS' if ok_a else 'FAIL'}")

    # --- 1b: corrupt the future, inspect the past --------------------------
    j = n // 2
    fut = base.with_columns([
        pl.when(pl.int_range(pl.len()) >= j).then(pl.col(c) * 1.5 + 7.0).otherwise(pl.col(c)).alias(c)
        for c in base.columns if c != "t"
    ])
    c_, _ = build_features(fut, CFG, FRAC_D)
    Cm = c_.select(cols).to_numpy()[:j]
    m2 = np.isfinite(A[:j]) & np.isfinite(Cm)
    err_fut = float(np.abs(A[:j] - Cm)[m2].max()) if m2.any() else np.inf
    ok_b = err_fut < 1e-10
    print(f"  1b future-perturbation     max|delta| = {err_fut:.3e}   "
          f"{'PASS' if ok_b else 'FAIL'}   (past must not react to the future)")

    if not (ok_a and ok_b):
        bad = [c for i, c in enumerate(cols)
               if np.nanmax(np.abs(A[:j, i] - Cm[:, i])) > 1e-10]
        print(f"    LEAKING columns: {bad}")
    return bool(ok_a and ok_b)


# ---- gate 2: split test ---------------------------------------------------
def gate_split_test() -> bool:
    ok = True
    for a, b in (("train", "val"), ("val", "test")):
        if not (len(SPLITS[a]) and len(SPLITS[b])):
            continue
        end_a, start_b = SPLITS[a][-1] + span - 1, SPLITS[b][0]
        sep = start_b - end_a
        ok &= sep > 0
        print(f"  split {a}->{b:<5}  separation {sep:>7,} min   "
              f"embargo required {CFG.embargo_min:,}   {'PASS' if sep > 0 else 'FAIL'}")
    inter = set(SPLITS["train"].tolist()) & set(SPLITS["test"].tolist())
    ok &= not inter
    return bool(ok)


# ---- gate 3: scaler test --------------------------------------------------
def gate_scaler_test() -> bool:
    recomputed = hashlib.sha256(
        np.concatenate([np.asarray(SCALER["mean"]), np.asarray(SCALER["std"])]).tobytes()
    ).hexdigest()[:16]
    n_train_rows = int(train_row.sum())
    ok = (SCALER["fitted_on"]["split"] == "train"
          and SCALER["fitted_on"]["rows"] == n_train_rows
          and n_train_rows < len(X))
    print(f"  scaler fitted on  {SCALER['fitted_on']['split']} split, "
          f"{n_train_rows:,}/{len(X):,} rows   hash {recomputed}   {'PASS' if ok else 'FAIL'}")
    return bool(ok)


print("Sanity gates")
GATES["shift"] = gate_shift_test()
GATES["split"] = gate_split_test()
GATES["scaler"] = gate_scaler_test()

<div style="background: linear-gradient(90deg, #0d1b2a, #1b263b); border-left: 4px solid #00b4d8; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #90e0ef; margin: 0 0 10px 0;">🎯 §13 · Losses &amp; Training</h2>
  <p style="color: #ade8f4; margin: 0 0 8px 0;"><strong>Huber is the default, not MSE.</strong> One-minute crypto returns are heavy-tailed; under MSE a handful of flash-crash minutes dominate the gradient and the model spends its capacity fitting six events. MSE is still reported for comparability with the literature.</p>  <ul style="color: #ade8f4; margin: 0; padding-left: 20px;">
    <li><strong>Pinball loss</strong> at q ∈ {0.1, 0.5, 0.9} gives prediction intervals, which are far more actionable than a point forecast in this domain.</li>
    <li><strong>Directional-aware</strong> <code>Huber + λ·softplus(−y·ŷ)</code> — at a 1-minute horizon <em>sign</em> is what a strategy monetises. Kept as an ablation with small λ, never a default.</li>
  </ul>  <p style="color: #ade8f4; margin: 0 0 8px 0;"><strong>On multi-GPU + mixed precision:</strong> the model is wrapped in <code>nn.DataParallel</code> whenever more than one GPU is visible, and <code>batch_size</code> scales with the device count. Autocast is entered in the <em>main thread</em> — PyTorch propagates autocast state into DataParallel's internal threads, so this is correct as written. Loss is computed in fp32 outside the autocast region, and checkpoints always save <code>model.module.state_dict()</code> so they load cleanly on one GPU or none.</p>
</div>

In [ ]:
def make_loss(name: str, cfg: Config):
    """Return (loss_fn, n_out). n_out > 1 only for the quantile head."""
    if name == "mse":
        return (lambda p, y: F.mse_loss(p[..., 0], y)), 1
    if name == "huber":
        return (lambda p, y: F.huber_loss(p[..., 0], y, delta=cfg.huber_delta)), 1
    if name == "huber_dir":
        def f(p, y):
            pr = p[..., 0]
            return (F.huber_loss(pr, y, delta=cfg.huber_delta)
                    + cfg.dir_lambda * F.softplus(-y * pr).mean())
        return f, 1
    if name == "pinball":
        qs = torch.tensor(cfg.quantiles)
        def f(p, y):
            q = qs.to(p.device, p.dtype).view(1, 1, -1)
            e = y.unsqueeze(-1) - p
            return torch.maximum(q * e, (q - 1) * e).mean()
        return f, len(cfg.quantiles)
    raise ValueError(f"unknown loss {name!r}")


def target_head(out: torch.Tensor) -> torch.Tensor:
    """Collapse a multi-task (B, H, N, n_out) output to the target head (B, H, n_out)."""
    return out if out.dim() == 3 else out[:, :, TARGET_IDX, :]


def point_forecast(pred: torch.Tensor, cfg: Config) -> torch.Tensor:
    """Collapse a model output to a point forecast (B, H): the median for quantile heads."""
    pred = target_head(pred)
    if pred.shape[-1] == 1:
        return pred[..., 0]
    return pred[..., cfg.quantiles.index(0.5)]


def wrap_parallel(model: nn.Module) -> nn.Module:
    model = model.to(DEVICE)
    if N_GPU > 1:
        model = nn.DataParallel(model)
    return model


def unwrap(model: nn.Module) -> nn.Module:
    return model.module if isinstance(model, nn.DataParallel) else model


def effective_batch(cfg: Config) -> int:
    """DataParallel splits the batch across devices, so scale it up to keep per-GPU work constant."""
    return cfg.batch_size * max(1, N_GPU)


@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader, cfg: Config) -> tuple[np.ndarray, np.ndarray]:
    """Return (pred, target) as (n, H) float32 in scaled log-return space."""
    model.eval()
    P, Y = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_ENABLED):
            out = model(xb)
        P.append(point_forecast(out.float(), cfg).cpu().numpy())
        Y.append(yb.numpy() if yb.ndim == 2 else yb[:, :, TARGET_IDX].numpy())
    return np.concatenate(P), np.concatenate(Y)


print(f"effective batch size {effective_batch(CFG)} "
      f"({CFG.batch_size} per GPU x {max(1, N_GPU)} GPU)")

In [ ]:
def train_model(model: nn.Module, cfg: Config, train_loader: DataLoader,
                val_loader: DataLoader, tag: str, epochs: int | None = None,
                resume: bool = False, verbose: bool = True) -> dict:
    """Train with AMP, gradient clipping, cosine schedule, early stopping and resume.

    Checkpoints every epoch so a Kaggle session hitting the 12 h wall can be continued.
    """
    epochs = epochs or cfg.epochs
    loss_fn, _ = make_loss(cfg.loss, cfg)
    model = wrap_parallel(model)

    decay, no_decay = [], []
    for n, p_ in unwrap(model).named_parameters():
        if not p_.requires_grad:
            continue
        (no_decay if p_.ndim <= 1 or n.endswith(".bias") else decay).append(p_)
    opt = torch.optim.AdamW(
        [{"params": decay, "weight_decay": cfg.weight_decay},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=cfg.lr, betas=cfg.betas, eps=1e-8)

    steps = max(1, len(train_loader)) * epochs
    warm = max(1, int(cfg.warmup_frac * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm
        else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_SCALER)

    ck = CKPT_DIR / f"{tag}_last.pt"
    best_ck = CKPT_DIR / f"{tag}_best.pt"
    start_epoch, best, bad, hist = 0, float("inf"), 0, []
    resume_src = RESUME_DIR / f"{tag}_last.pt"
    if resume and resume_src.exists():
        st = torch.load(resume_src, map_location=DEVICE, weights_only=False)
        unwrap(model).load_state_dict(st["model"])
        opt.load_state_dict(st["optimizer"]); sched.load_state_dict(st["scheduler"])
        if st.get("scaler") and USE_SCALER:
            scaler.load_state_dict(st["scaler"])
        start_epoch, best, hist = st["epoch"] + 1, st["best_metric"], st.get("history", [])
        print(f"[{tag}] resumed from epoch {st['epoch']} (best val {best:.6f})")

    log_path = RUN_DIR / f"{tag}_metrics.jsonl"
    for ep in range(start_epoch, epochs):
        model.train()
        t_ep, tot, nb, gn = time.time(), 0.0, 0, 0.0
        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            # autocast is entered here, in the main thread: DataParallel propagates
            # the autocast state into its worker threads.
            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_ENABLED):
                out = model(xb)
            loss = loss_fn(out.float(), yb.float())       # loss always in fp32
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            gn += float(torch.nn.utils.clip_grad_norm_(unwrap(model).parameters(), cfg.grad_clip))
            scaler.step(opt); scaler.update(); sched.step()
            tot += float(loss); nb += 1

        vp, vy = predict(model, val_loader, cfg)
        vloss = float(np.mean((vp - vy) ** 2))
        rec = {"epoch": ep, "train_loss": tot / max(1, nb), "val_mse": vloss,
               "lr": sched.get_last_lr()[0], "grad_norm": gn / max(1, nb),
               "secs": time.time() - t_ep,
               "vram_gb": torch.cuda.max_memory_allocated() / 1024**3 if DEVICE.type == "cuda" else 0.0}
        hist.append(rec)
        with open(log_path, "a") as f:
            f.write(json.dumps(rec) + "\n")
        if verbose:
            print(f"  [{tag}] ep {ep + 1:>2}/{epochs}  train {rec['train_loss']:.6f}  "
                  f"val_mse {vloss:.6f}  lr {rec['lr']:.2e}  "
                  f"gnorm {rec['grad_norm']:.2f}  {rec['secs']:.0f}s  "
                  f"{rec['vram_gb']:.1f}GB")

        state = {"model": unwrap(model).state_dict(), "optimizer": opt.state_dict(),
                 "scheduler": sched.state_dict(),
                 "scaler": scaler.state_dict() if USE_SCALER else None,
                 "epoch": ep, "best_metric": min(best, vloss), "history": hist,
                 "config": asdict(cfg), "feature_names": FEATURE_NAMES,
                 "torch_version": torch.__version__}
        torch.save(state, ck)
        if vloss < best - 1e-9:
            best, bad = vloss, 0
            torch.save(state, best_ck)
        else:
            bad += 1
            if bad >= cfg.early_stop_patience:
                print(f"  [{tag}] early stop at epoch {ep + 1} (patience {cfg.early_stop_patience})")
                break

        # Kaggle kills a session at the 12 h wall with no warning and no saved output.
        # Stopping while time remains leaves both a checkpoint and a session that can
        # still be versioned, which is the difference between resuming and restarting.
        if cfg.session_budget_hours > 0 and hours_left() <= cfg.reserve_hours:
            print(f"  [{tag}] session budget reached at epoch {ep + 1} "
                  f"({hours_left() * 60:.0f} min left). Checkpoint written to {ck.name} - "
                  f"save the version, then resume in a new session.")
            break

    if best_ck.exists():
        unwrap(model).load_state_dict(
            torch.load(best_ck, map_location=DEVICE, weights_only=False)["model"])
    return {"model": model, "best_val_mse": best, "history": hist}

In [ ]:
# ---- gate 4: overfit a single batch ---------------------------------------
def gate_overfit_batch(steps: int = 200, n: int = 8) -> bool:
    """If the model cannot memorise 8 samples, the plumbing is broken. Do not launch a long run."""
    set_seed(CFG.seed)
    m = build_model(CFG, N_VARIATES).to(DEVICE)
    ds = WindowDataset(X, SPLITS["train"][:n], L, H, TARGET_IDX,
                       full_target=not CFG.project_target_only)
    xb = torch.stack([ds[i][0] for i in range(n)]).to(DEVICE)
    yb = torch.stack([ds[i][1] for i in range(n)]).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
    first = None
    for s in range(steps):
        opt.zero_grad(set_to_none=True)
        # point_forecast, not [..., 0]: under the multi-task head the model returns
        # (B, H, N, n_out) and the gate is about the target series alone.
        loss = F.mse_loss(point_forecast(m(xb), CFG),
                          yb if yb.ndim == 2 else yb[:, :, TARGET_IDX])
        loss.backward(); opt.step()
        if s == 0:
            first = float(loss)
    last = float(loss)
    ok = last < 0.05 * first
    print(f"  overfit-1-batch   loss {first:.5f} -> {last:.6f} "
          f"({last / first:.1%} of start)   {'PASS' if ok else 'FAIL'}")
    del m; gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return bool(ok)


# ---- gate 5: leakage test -------------------------------------------------
def gate_leakage_test(epochs: int = 1) -> bool:
    """Replace the target with noise. If the model still 'predicts', features contain the target."""
    global X
    set_seed(CFG.seed)
    rng = np.random.default_rng(123)
    X_backup = X[:, TARGET_IDX].copy()
    try:
        X[:, TARGET_IDX] = rng.standard_normal(len(X)).astype(np.float32)
        tl = make_loader("train", effective_batch(CFG), cap=20_000)
        vl = make_loader("val", effective_batch(CFG), shuffle=False, cap=8_000)
        cfg2 = Config(**{**asdict(CFG), "epochs": epochs, "early_stop_patience": 99})
        res = train_model(build_model(cfg2, N_VARIATES), cfg2, tl, vl,
                          tag="leakcheck", epochs=epochs, resume=False, verbose=False)
        p_, y_ = predict(res["model"], vl, cfg2)
        r2 = 1 - ((p_ - y_) ** 2).sum() / max(1e-12, ((y_ - y_.mean()) ** 2).sum())
        ok = r2 < 0.02
        print(f"  leakage test      noise-target val R2 = {r2:+.4f}   "
              f"{'PASS' if ok else 'FAIL - features contain the target'}")
        del res; gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        return bool(ok)
    finally:
        X[:, TARGET_IDX] = X_backup


GATES["overfit_batch"] = gate_overfit_batch()
GATES["leakage"] = gate_leakage_test()

print("\n" + "=" * 62)
for k, v in GATES.items():
    print(f"  {k:<16} {'PASS' if v else 'FAIL'}")
ALL_GATES_PASS = all(GATES.values())
print(f"  {'ALL GATES':<16} {'PASS' if ALL_GATES_PASS else 'FAIL'}")
print("=" * 62)
if not ALL_GATES_PASS:
    print("\n  Training results below are NOT valid until every gate passes.")

<div style="background: linear-gradient(90deg, #0d1b2a, #1b263b); border-left: 4px solid #00b4d8; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #90e0ef; margin: 0 0 10px 0;">🚀 Baseline first, then the model</h2>
  <p style="color: #ade8f4; margin: 0 0 8px 0;">The random-walk number is computed and printed <strong>before</strong> any training starts. Doing it afterwards invites reinterpreting it to flatter whatever the model produced.</p>
</div>

In [ ]:
train_loader = make_loader("train", effective_batch(CFG))
val_loader = make_loader("val", effective_batch(CFG), shuffle=False, cap=CFG.eval_max_windows)
test_loader = make_loader("test", effective_batch(CFG), shuffle=False, cap=CFG.eval_max_windows)
print(f"loaders  train {len(train_loader.dataset):,} | val {len(val_loader.dataset):,} "
      f"| test {len(test_loader.dataset):,} windows")

naive_p, naive_y = predict(wrap_parallel(NaiveZero(H)), val_loader, CFG)
print(f"\nRANDOM-WALK BASELINE (validation, scaled return space)")
print(f"  MSE {np.mean((naive_p - naive_y) ** 2):.6f}   MAE {np.mean(np.abs(naive_p - naive_y)):.6f}")
print("  Everything below must beat this to mean anything.\n")

set_seed(CFG.seed, CFG.deterministic)
t0 = time.time()
res_it = train_model(build_model(CFG, N_VARIATES), CFG, train_loader, val_loader,
                     tag="itransformer", resume=CFG.resume)
print(f"\niTransformer trained in {(time.time() - t0) / 60:.1f} min   "
      f"best val MSE {res_it['best_val_mse']:.6f}")

In [ ]:
MODELS: dict[str, nn.Module] = {"iTransformer": res_it["model"]}

# The two untrained baselines always run: the naive number is what every other result
# is measured against, and it costs one forward pass. The trained ones are a stage
# switch, because they do not fit alongside the ablation in a single 12 h session.
BASELINE_SPECS = [
    ("Naive (RW)",   lambda: NaiveZero(H),                              False),
    ("HistMean",     lambda: HistMean(H, TARGET_IDX),                   False),
]
TRAINED_SPECS = [
    ("AR(60)",       lambda: DirectLinear(min(60, L), H, TARGET_IDX),   True),
    ("DLinear",      lambda: DLinear(L, H, 25, TARGET_IDX),             True),
    # PatchTST: patching without cross-variate mixing - the control for whether
    # cross-variate attention contributes anything at all.
    ("PatchTST", lambda: PatchTST(L, H, CFG.d_model // 2, max(2, CFG.n_heads // 2),
                                  max(2, CFG.e_layers - 1), CFG.d_ff // 2, CFG.dropout,
                                  CFG.patch_len, None, TARGET_IDX, CFG.use_norm), True),
    # TimeXer: the head-to-head required by CLAUDE.md 10.3, built for exactly this
    # endogenous-target-plus-exogenous-covariates setting.
    ("TimeXer",  lambda: TimeXer(L, H, N_VARIATES, CFG.d_model, CFG.n_heads, CFG.e_layers,
                                 CFG.d_ff, CFG.dropout, CFG.activation, CFG.patch_len,
                                 TARGET_IDX, CFG.use_norm), True),
]
if CFG.run_vanilla_transformer:
    TRAINED_SPECS.append(
        ("VanillaTF", lambda: VanillaTransformer(L, H, N_VARIATES, CFG.d_model // 2,
                                                 max(2, CFG.n_heads // 2), 2,
                                                 CFG.d_ff // 2, CFG.dropout), True))
if CFG.run_baselines:
    BASELINE_SPECS += TRAINED_SPECS
else:
    print("trained baselines skipped (CFG.run_baselines = False) - the model is still "
          "measured against the naive random walk, the number that matters most")

for name, ctor, needs_training in BASELINE_SPECS:
    if not needs_training:
        MODELS[name] = wrap_parallel(ctor())
        continue
    if CFG.session_budget_hours > 0 and hours_left() <= CFG.reserve_hours:
        print(f"  {name:<12} skipped - session budget exhausted")
        continue
    set_seed(CFG.seed, CFG.deterministic)
    bs = effective_batch(CFG) // (4 if name == "VanillaTF" else 1)
    tl = make_loader("train", max(8, bs))
    r = train_model(ctor(), CFG, tl, val_loader, tag=name.replace("(", "").replace(")", ""),
                    resume=CFG.resume, verbose=False)
    MODELS[name] = r["model"]
    print(f"  {name:<12} best val MSE {r['best_val_mse']:.6f}")
print(f"\n{len(MODELS)} models ready for evaluation")

<div style="background: linear-gradient(135deg, #0f3460, #16213e, #1a1a2e); border-left: 4px solid #533483; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e94560; margin: 0 0 10px 0;">📊 §14 · Evaluation</h2>
  <p style="color: #a8dadc; margin: 0 0 8px 0;"><strong>All metrics are on log returns, never price levels.</strong> A model predicting the price level scores a spectacular R² by echoing <code>close_t</code> while carrying zero information.</p>  <ul style="color: #a8dadc; margin: 0; padding-left: 20px;">
    <li><strong>MASE relative to the naive baseline.</strong> <code>MASE ≥ 1</code> means the model is useless, however good its raw MSE looks.</li>
    <li><strong>R² is expected to be small and positive.</strong> At a 1-minute horizon anything above ~0.05 should raise a leakage suspicion until re-verified — not celebration.</li>
    <li><strong>Directional accuracy and MCC</strong> on <code>sign(y)</code>, excluding near-zero returns below 1 bp, which are noise rather than direction.</li>
    <li><strong>Diebold–Mariano test</strong> against each baseline, with Newey–West HAC variance and the Harvey–Leybourne–Newbold small-sample correction. A metric difference without a significance test is not evidence.</li>
  </ul>  <p style="color: #a8dadc; margin: 0 0 8px 0;">Horizon-<em>h</em> results come from the <strong>cumulative sum</strong> of the predicted one-minute path, which is exactly the <em>h</em>-minute log return by construction.</p>
</div>

In [ ]:
def cum_h(a: np.ndarray, h: int) -> np.ndarray:
    """Cumulative log return over the first h steps of a predicted path."""
    return a[:, :h].sum(axis=1)


def dm_test(e1: np.ndarray, e2: np.ndarray, h: int = 1, power: int = 2) -> tuple[float, float]:
    """Diebold-Mariano with Newey-West HAC variance and the HLN small-sample correction.

    Returns (statistic, two-sided p-value). Negative statistic favours model 1.
    """
    d = np.abs(e1) ** power - np.abs(e2) ** power
    n = len(d)
    dbar = d.mean()
    dd = d - dbar
    gamma0 = (dd @ dd) / n
    # An h-step-ahead loss differential is MA(h-1), so h-1 autocovariance lags are
    # used - zero of them for h=1. Forcing a lag at h=1 (max(1, h-1)) would inflate
    # the variance and make the test needlessly conservative.
    lag = max(0, h - 1)
    var = gamma0 + 2 * sum((1 - k / (lag + 1)) * (dd[k:] @ dd[:-k]) / n
                           for k in range(1, lag + 1))
    if var <= 0:
        return float("nan"), float("nan")
    stat = dbar / math.sqrt(var / n)
    corr = math.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    stat *= corr
    from math import erf
    # two-sided normal approximation (n is very large here, so t ~ z)
    p = 2 * (1 - 0.5 * (1 + erf(abs(stat) / math.sqrt(2))))
    return float(stat), float(p)


NAN = float("nan")


def _phi(z: float) -> float:
    """Standard normal CDF."""
    from math import erf
    return 0.5 * (1 + erf(z / math.sqrt(2)))


def directional_metrics(p_: np.ndarray, y_: np.ndarray, eps_bp: float,
                        top_q: float = 0.9, sigma: float = 1.0) -> dict:
    """Does the model call the direction correctly?

    Three refinements over a bare hit-rate, each closing a way the bare number misleads:

    * A forecast of exactly zero is NOT a wrong call, it is NO call. Counting it as
      wrong makes the random-walk baseline score 0.0 instead of abstaining. Accuracy is
      therefore measured over *called* samples, and `coverage` reports how many those are.
    * A hit rate of 0.51 on a few thousand samples may be indistinguishable from a coin
      flip, so a two-sided binomial test against p = 0.5 is reported alongside it. This is
      the directional counterpart of the Diebold-Mariano test used for squared error.
    * A model that always says "up" still scores ~0.5. Per-direction precision and the
      predicted-up share expose that; MCC collapses to ~0 for it.

    Near-zero true returns (|y| < eps_bp basis points) are excluded as noise rather than
    direction, per CLAUDE.md 13.2.
    """
    # eps_bp is quoted in basis points of a RAW log return, but p_ and y_ arrive
    # standardised. Dividing by sigma puts the threshold back into the units the arrays
    # are actually in - without it the filter is ~1/sigma times too small and, at
    # sigma ~ 1.3e-3, it excludes 0.1% of samples instead of the intended ~10%.
    eps = eps_bp * 1e-4 / max(sigma, 1e-12)
    m = np.abs(y_) > eps                       # a real move, not noise
    if m.sum() <= 10:
        return {"dir_acc": NAN, "dir_p": NAN, "mcc": NAN, "coverage": 0.0,
                "prec_up": NAN, "prec_down": NAN, "pred_up_share": NAN,
                "dir_acc_top": NAN, "n_called": 0}

    sp, sy = np.sign(p_[m]), np.sign(y_[m])
    called = sp != 0                           # a zero forecast abstains
    n_called = int(called.sum())
    if n_called <= 10:
        return {"dir_acc": NAN, "dir_p": NAN, "mcc": NAN,
                "coverage": float(n_called / m.sum()), "prec_up": NAN, "prec_down": NAN,
                "pred_up_share": NAN, "dir_acc_top": NAN, "n_called": n_called}

    spc, syc = sp[called], sy[called]
    hit = spc == syc
    da = float(hit.mean())

    # two-sided binomial test vs 0.5 (normal approximation; n is large here)
    z = (hit.sum() - 0.5 * n_called) / math.sqrt(0.25 * n_called)
    dir_p = float(2 * (1 - _phi(abs(z))))

    tp = float(((spc > 0) & (syc > 0)).sum()); tn = float(((spc < 0) & (syc < 0)).sum())
    fp = float(((spc > 0) & (syc < 0)).sum()); fn = float(((spc < 0) & (syc > 0)).sum())
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = float((tp * tn - fp * fn) / den) if den > 0 else 0.0

    # accuracy where the model is most confident - the subset a strategy would trade
    conf = np.abs(p_[m][called])
    top = conf >= np.quantile(conf, top_q)
    return {
        "dir_acc": da, "dir_p": dir_p, "mcc": mcc,
        "coverage": float(n_called / m.sum()),
        "prec_up": float(tp / (tp + fp)) if (tp + fp) > 0 else NAN,
        "prec_down": float(tn / (tn + fn)) if (tn + fn) > 0 else NAN,
        "pred_up_share": float((spc > 0).mean()),
        "dir_acc_top": float(hit[top].mean()) if top.sum() > 10 else NAN,
        "n_called": n_called,
    }


def evaluate(pred: np.ndarray, targ: np.ndarray, horizons, eps_bp: float,
             naive_err: dict | None = None, sigma: float | None = None) -> dict:
    """Full metric suite, per horizon, on scaled log returns."""
    out = {}
    for h in horizons:
        p_, y_ = cum_h(pred, h), cum_h(targ, h)
        e = p_ - y_
        mse, mae = float((e ** 2).mean()), float(np.abs(e).mean())
        naive_mae = float(np.abs(y_).mean())          # naive forecast is 0
        out[h] = {
            "mse": mse, "rmse": math.sqrt(mse), "mae": mae,
            "mase": mae / naive_mae if naive_mae > 0 else NAN,
            "r2": float(1 - (e ** 2).sum() / max(1e-15, ((y_ - y_.mean()) ** 2).sum())),
            "n": int(len(p_)),
            **directional_metrics(p_, y_, eps_bp,
                                  sigma=SIGMA_TARGET if sigma is None else sigma),
        }
        if naive_err is not None and h in naive_err:
            stat, pv = dm_test(e, naive_err[h], h=h)
            out[h]["dm_stat"], out[h]["dm_p"] = stat, pv
    return out


PRED: dict[str, tuple[np.ndarray, np.ndarray]] = {}
for name, m in MODELS.items():
    PRED[name] = predict(m, test_loader, CFG)
    print(f"  predicted {name:<14} {PRED[name][0].shape}")

EVAL_STEP_MIN = eval_row_step(test_loader)
if EVAL_STEP_MIN > 1:
    print(f"\n  test windows thinned to one every {EVAL_STEP_MIN} min by "
          f"eval_max_windows={CFG.eval_max_windows:,}; every annualised figure below "
          f"accounts for that spacing")

naive_errs = {h: cum_h(PRED["Naive (RW)"][0], h) - cum_h(PRED["Naive (RW)"][1], h)
              for h in CFG.horizons}
RESULTS = {name: evaluate(p_, y_, CFG.horizons, CFG.dir_acc_eps_bp, naive_errs)
           for name, (p_, y_) in PRED.items()}

hz = CFG.horizons[-1]
print(f"\nTEST SPLIT  |  horizon h={hz} min  |  {RESULTS['Naive (RW)'][hz]['n']:,} windows")
print(f"{'model':<14}{'MSE':>11}{'MAE':>11}{'MASE':>8}{'R2':>9}{'DirAcc':>9}"
      f"{'dir p':>8}{'MCC':>8}{'DM p':>9}")
print("-" * 87)
for name in RESULTS:
    r_ = RESULTS[name][hz]
    dmp = r_.get("dm_p", NAN)
    da = "     n/a" if not np.isfinite(r_["dir_acc"]) else f"{r_['dir_acc']:>9.4f}"
    dp = "     n/a" if not np.isfinite(r_["dir_p"]) else f"{r_['dir_p']:>8.4f}"
    mc = "     n/a" if not np.isfinite(r_["mcc"]) else f"{r_['mcc']:>+8.3f}"
    print(f"{name:<14}{r_['mse']:>11.6f}{r_['mae']:>11.6f}{r_['mase']:>8.4f}"
          f"{r_['r2']:>+9.4f}{da}{dp}{mc}"
          f"{'' if name == 'Naive (RW)' else f'{dmp:>9.4f}'}")
print("\nMASE < 1 beats naive. DM p < 0.05 makes that difference statistically credible.")
print("DirAcc n/a means the model made no directional call at all (the random walk")
print("forecasts exactly zero) - that is abstention, not 0% accuracy.")

<div style="background: linear-gradient(135deg, #0f3460, #16213e, #1a1a2e); border-left: 4px solid #533483; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e94560; margin: 0 0 10px 0;">🧭 Directional accuracy — does it call up vs down?</h2>
  <p style="color: #a8dadc; margin: 0 0 8px 0;">At a 1-minute horizon, <strong>sign is what a strategy monetises</strong> — a forecast with a poor MSE but a reliable sign is tradeable, and a forecast with a good MSE but a coin-flip sign is not. This block reports direction as its own first-class result rather than a column in the error table.</p>  <ul style="color: #a8dadc; margin: 0; padding-left: 20px;">
    <li><strong>Hit rate over <em>called</em> samples.</strong> A forecast of exactly zero is not a wrong call, it is <em>no call</em>. Counting it as wrong is what makes a naive baseline appear to score 0% instead of abstaining, so <code>coverage</code> reports what fraction of moves the model actually committed to.</li>
    <li><strong>Binomial test against 0.5.</strong> 51% on a few thousand samples may be pure noise. This is the directional counterpart of the Diebold–Mariano test used for squared error — a hit rate without it is not evidence.</li>
    <li><strong>Per-direction precision and the predicted-up share.</strong> A model that always says 'up' still scores ~0.5 overall. These two numbers expose it immediately, and MCC collapses toward 0 for it.</li>
    <li><strong>Accuracy in the top confidence decile</strong> — the subset a threshold strategy would actually trade. If accuracy does not rise with confidence, the model's confidence is meaningless and the threshold sweep in the backtest cannot help it.</li>
    <li><strong>True moves smaller than 1 bp are excluded</strong> as noise rather than direction.</li>
  </ul>
</div>

In [ ]:
print(f"DIRECTIONAL ACCURACY  |  test split  |  h={hz} min  |  "
      f"|y| > {CFG.dir_acc_eps_bp:.0f} bp only")
print(f"{'model':<14}{'DirAcc':>9}{'p vs 0.5':>10}{'MCC':>8}{'Cover':>8}"
      f"{'PrecUp':>9}{'PrecDn':>9}{'PredUp':>9}{'Top10%':>9}")
print("-" * 85)
for name, r_ in ((n, RESULTS[n][hz]) for n in RESULTS):
    if not np.isfinite(r_["dir_acc"]):
        print(f"{name:<14}{'abstains — forecasts exactly zero, makes no directional call':>71}")
        continue
    sig = " *" if r_["dir_p"] < 0.05 else ""
    print(f"{name:<14}{r_['dir_acc']:>9.4f}{r_['dir_p']:>10.4f}{r_['mcc']:>+8.3f}"
          f"{r_['coverage']:>8.2%}{r_['prec_up']:>9.4f}{r_['prec_down']:>9.4f}"
          f"{r_['pred_up_share']:>9.2%}{r_['dir_acc_top']:>9.4f}{sig}")
print("\n  * = hit rate differs from a coin flip at the 5% level.")
print("  PredUp far from ~50% means the model is biased to one direction;")
print("  check MCC, which collapses toward 0 for a model that always calls the same way.")

# --- accuracy as a function of forecast confidence -------------------------
def da_by_confidence(p_: np.ndarray, y_: np.ndarray, eps_bp: float, n_bins: int = 10,
                     sigma: float = 1.0):
    eps = eps_bp * 1e-4 / max(sigma, 1e-12)
    m = np.abs(y_) > eps
    sp, sy = np.sign(p_[m]), np.sign(y_[m])
    called = sp != 0
    if called.sum() < 100:
        return np.array([]), np.array([]), np.array([])
    hit = (sp[called] == sy[called]).astype(float)
    conf = np.abs(p_[m][called])
    edges = np.quantile(conf, np.linspace(0, 1, n_bins + 1))
    idx = np.clip(np.digitize(conf, edges[1:-1]), 0, n_bins - 1)
    acc = np.array([hit[idx == b].mean() if (idx == b).sum() > 20 else np.nan
                    for b in range(n_bins)])
    cnt = np.array([(idx == b).sum() for b in range(n_bins)])
    return np.arange(1, n_bins + 1), acc, cnt


fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# DirAcc by horizon, with a 95% coin-flip band - 2 series so a legend is required
for i, k in enumerate([m for m in ("iTransformer", "DLinear") if m in RESULTS]):
    ys = [RESULTS[k][h]["dir_acc"] for h in CFG.horizons]
    axes[0].plot(CFG.horizons, ys, marker="o", color=CAT[[1, 3][i]], label=k)
n_eff = RESULTS["iTransformer"][hz]["n_called"] or 1
band = 1.96 * math.sqrt(0.25 / n_eff)
axes[0].axhline(0.5, color=INK_MUTED, lw=1.2, ls="--")
axes[0].axhspan(0.5 - band, 0.5 + band, color=GRID, alpha=0.55, zorder=0)
axes[0].annotate("coin flip ±95%", (CFG.horizons[0], 0.5 + band), fontsize=8,
                 color=INK_MUTED, va="bottom")
finish(axes[0], "Directional accuracy by horizon", "horizon (minutes)",
       "hit rate", legend=True)

# accuracy vs confidence decile - magnitude, so one hue
# Same horizon as the table above, so the two numbers are comparable: the confidence
# curve is about the h-minute call, not about each 1-minute step inside the path.
_p_it, _y_it = PRED["iTransformer"]
b, acc, cnt = da_by_confidence(cum_h(_p_it, hz), cum_h(_y_it, hz),
                               CFG.dir_acc_eps_bp, sigma=SIGMA_TARGET)
if len(b):
    axes[1].bar(b, acc, color=SEQ[3], width=0.68)
    axes[1].axhline(0.5, color=INK_MUTED, lw=1.2, ls="--")
    axes[1].set_ylim(min(0.42, np.nanmin(acc) - 0.02), max(0.58, np.nanmax(acc) + 0.02))
    axes[1].set_xticks(b)
finish(axes[1], f"iTransformer hit rate by confidence decile  (h={hz} min)",
       "confidence decile (10 = most confident)", "hit rate")
plt.tight_layout(); plt.show()

if len(b) and np.isfinite(acc).sum() >= 2:
    lo, hi = np.nanmean(acc[:3]), np.nanmean(acc[-3:])
    print(f"hit rate, bottom 3 deciles {lo:.4f}  ->  top 3 deciles {hi:.4f}  "
          f"({'rises' if hi > lo else 'does NOT rise'} with confidence)")
    if hi <= lo:
        print("  Confidence carries no directional information, so a threshold filter")
        print("  in the backtest can only cut turnover - it cannot improve the edge.")

<div style="background: linear-gradient(135deg, #0f3460, #16213e, #1a1a2e); border-left: 4px solid #533483; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e94560; margin: 0 0 10px 0;">💸 Economic evaluation — costs decide this, not statistics</h2>
  <p style="color: #a8dadc; margin: 0 0 8px 0;"><strong>A statistical improvement that does not survive costs is not a result.</strong> At 1-minute frequency costs dominate everything: a strategy that trades every minute pays roughly 100%+ annualised in fees alone before it has predicted anything.</p>  <ul style="color: #a8dadc; margin: 0; padding-left: 20px;">
    <li>Costs charged per side: <strong>4 bp taker fee + 2 bp slippage</strong>. Both are conservative for BTC/USDT spot but not generous.</li>
    <li>Trades are placed on a <strong>non-overlapping grid every h minutes</strong>, so each position is independent and turnover is honest. Overlapping 1-minute positions would silently multiply leverage.</li>
    <li>A confidence threshold sweeps from 0 (always in the market) upward — the interesting question is whether <em>any</em> threshold produces a positive net Sharpe, not whether the raw signal does.</li>
    <li><strong>Deflated Sharpe ratio</strong> corrects for the multiple-testing inflation this notebook itself creates by comparing several models and thresholds. A nominal Sharpe of 2 across 20 trials is not the same evidence as a Sharpe of 2 from a single pre-registered test.</li>
  </ul>  <p style="color: #a8dadc; margin: 0 0 8px 0;"><strong>Reminder:</strong> a Sharpe above 3 at 1-minute frequency should be treated as leakage until proven otherwise. The correct response to an unexpectedly good number is to hunt for the leak.</p>
</div>

In [ ]:
def backtest(pred: np.ndarray, targ: np.ndarray, h: int, thresh_bp: float,
             fee: float, slip: float, row_step_min: int = 1) -> dict:
    """Long/flat/short on the sign of the h-minute forecast, on a non-overlapping grid.

    `row_step_min` is how many minutes apart consecutive prediction rows are. It is 1 for
    a full stride-1 evaluation, but `eval_max_windows` thins the split, and then every
    annualised figure depends on the true spacing. Assuming 1 when rows are k minutes
    apart inflates the annualisation factor by sqrt(k) - a free Sharpe multiplier.
    """
    step = max(1, math.ceil(h / row_step_min))       # rows to skip for non-overlap
    period_min = step * row_step_min                 # the real holding period, in minutes
    p_ = cum_h(pred, h)[::step] * SIGMA_TARGET       # back to raw log-return units
    y_ = cum_h(targ, h)[::step] * SIGMA_TARGET
    if len(p_) < 30:
        return {}
    pos = np.where(np.abs(p_) > thresh_bp * 1e-4, np.sign(p_), 0.0)
    gross = pos * y_
    turn = np.abs(np.diff(np.concatenate([[0.0], pos])))
    cost = turn * (fee + slip)
    net = gross - cost

    per_year = (365 * 24 * 60) / period_min
    ann = math.sqrt(per_year)
    sharpe_period = float(net.mean() / (net.std() + 1e-12))
    sharpe = sharpe_period * ann
    dn = net[net < 0]
    sortino = float(net.mean() / (dn.std() + 1e-12) * ann) if len(dn) else float("nan")
    eq = np.cumsum(net)
    dd = float((np.maximum.accumulate(eq) - eq).max())
    cum = float(np.expm1(eq[-1]))
    yrs = len(net) / per_year
    cagr = float((1 + cum) ** (1 / max(yrs, 1e-9)) - 1) if cum > -1 else float("nan")
    active = pos != 0
    zc = (net - net.mean()) / (net.std() + 1e-12)
    return {
        "sharpe_net": sharpe, "sharpe_gross": float(gross.mean() / (gross.std() + 1e-12) * ann),
        # The deflated Sharpe ratio is defined on the PER-PERIOD Sharpe and the moments
        # of that same series, so they are carried alongside the annualised headline.
        "sharpe_net_period": sharpe_period, "periods_per_year": float(per_year),
        "period_min": int(period_min),
        "skew": float((zc ** 3).mean()), "kurt": float((zc ** 4).mean()),
        "sortino": sortino, "cum_return": cum, "cagr": cagr, "max_dd": dd,
        "calmar": float(cagr / dd) if dd > 1e-9 else float("nan"),
        "hit_rate": float((gross[active] > 0).mean()) if active.any() else float("nan"),
        "turnover_per_year": float(turn.sum() / max(yrs, 1e-9)),
        "cost_drag_ann": float(cost.sum() / max(yrs, 1e-9)),
        "time_in_market": float(active.mean()), "n_periods": int(len(net)),
    }


def deflated_sharpe(sr: float, n: int, trial_sharpes, skew: float = 0.0,
                    kurt: float = 3.0) -> float:
    """Bailey & Lopez de Prado (2014): probability the observed Sharpe survives selection bias.

    The expected-maximum-Sharpe threshold is

        SR0 = sqrt(Var[SR_trials]) * [ (1-gamma) Z^-1(1 - 1/N) + gamma Z^-1(1 - 1/(N e)) ]

    where the leading term is the **dispersion of the trial Sharpe ratios**, not 1/N.
    Substituting 1/N there (a common shortcut) understates the threshold whenever the
    trials disagree with each other, which is exactly when selection bias is worst.
    """
    from math import erf
    trials = np.asarray([s for s in trial_sharpes if np.isfinite(s)], dtype=float)
    n_trials = len(trials)
    if n_trials < 2 or not np.isfinite(sr) or n < 3:
        return float("nan")
    var_sr = float(trials.std(ddof=1))
    if var_sr <= 0:
        return float("nan")
    e = 0.5772156649                                   # Euler-Mascheroni
    z = lambda q: math.sqrt(2) * _erfinv(2 * q - 1)
    sr0 = var_sr * ((1 - e) * z(1 - 1 / n_trials) + e * z(1 - 1 / (n_trials * math.e)))
    den = math.sqrt(1 - skew * sr + (kurt - 1) / 4 * sr**2)
    if den <= 0:
        return float("nan")
    stat = (sr - sr0) * math.sqrt(n - 1) / den
    return float(0.5 * (1 + erf(stat / math.sqrt(2))))


def _erfinv(y: float) -> float:
    """Inverse error function (Winitzki approximation), good to ~2e-3 - ample for a DSR."""
    a = 0.147
    ln = math.log(max(1e-16, 1 - y * y))
    t = 2 / (math.pi * a) + ln / 2
    return math.copysign(math.sqrt(max(0.0, math.sqrt(t * t - ln / a) - t)), y)


hb = CFG.backtest_horizon
THRESHOLDS = [0.0, 2.0, 5.0, 10.0, 20.0]
print(f"BACKTEST  |  h={hb} min  |  fee {CFG.fee_per_side * 1e4:.0f} bp + slip "
      f"{CFG.slippage_per_side * 1e4:.0f} bp per side  |  rows {EVAL_STEP_MIN} min apart")
print(f"{'model':<14}{'thr(bp)':>8}{'Sharpe_g':>10}{'Sharpe_n':>10}{'CumRet':>10}"
      f"{'MaxDD':>9}{'Hit':>7}{'TimeIn':>8}{'Turn/y':>9}")
print("-" * 85)
BT: dict[tuple, dict] = {}
for name in ("Naive (RW)", "DLinear", "PatchTST", "TimeXer", "iTransformer"):
    if name not in PRED:
        continue
    for th in THRESHOLDS:
        b = backtest(*PRED[name], hb, th, CFG.fee_per_side, CFG.slippage_per_side,
                     row_step_min=EVAL_STEP_MIN)
        if not b:
            continue
        BT[(name, th)] = b
        print(f"{name if th == THRESHOLDS[0] else '':<14}{th:>8.0f}{b['sharpe_gross']:>10.2f}"
              f"{b['sharpe_net']:>10.2f}{b['cum_return']:>+10.2%}{b['max_dd']:>9.4f}"
              f"{b['hit_rate']:>7.3f}{b['time_in_market']:>8.2%}{b['turnover_per_year']:>9.0f}")

best_key = max(BT, key=lambda k: BT[k]["sharpe_net"]) if BT else None
if best_key:
    b = BT[best_key]
    # DSR is defined on the per-period Sharpe with n observations. Feeding it the
    # annualised number multiplies the statistic by sqrt(periods_per_year) - roughly 750x
    # at h=60 - and returns ~1.000 for anything at all, silently disabling the whole
    # multiple-testing correction this section exists to apply.
    dsr = deflated_sharpe(b["sharpe_net_period"], b["n_periods"],
                          [v["sharpe_net_period"] for v in BT.values()],
                          skew=b["skew"], kurt=b["kurt"])
    print(f"\nbest net Sharpe: {b['sharpe_net']:+.3f} annualised "
          f"({b['sharpe_net_period']:+.5f} per {b['period_min']}-min period)"
          f"   [{best_key[0]}, threshold {best_key[1]:.0f} bp]")
    print(f"deflated Sharpe probability over {len(BT)} trials: {dsr:.3f}")
    print("  (DSR < 0.95 means the result is not distinguishable from multiple-testing luck)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. training curves - 2 series, so a legend is mandatory
hst = res_it["history"]
if hst:
    ep = [r["epoch"] + 1 for r in hst]
    axes[0].plot(ep, [r["train_loss"] for r in hst], color=CAT[1], label="train loss")
    axes[0].plot(ep, [r["val_mse"] for r in hst], color=CAT[3], label="val MSE")
    finish(axes[0], "Training curve", "epoch", "loss", legend=True)

# 2. model comparison - a sorted bar chart, not six cycled hues
mm = sorted(RESULTS, key=lambda k: RESULTS[k][hz]["mase"])
vals = [RESULTS[k][hz]["mase"] for k in mm]
cols = [CAT[1] if k == "iTransformer" else GRID for k in mm]
axes[1].barh(range(len(mm)), vals, color=cols, height=0.62)
axes[1].axvline(1.0, color=CAT[4], lw=1.5, ls="--")
axes[1].set_yticks(range(len(mm))); axes[1].set_yticklabels(mm, fontsize=9)
axes[1].invert_yaxis()
for i, v in enumerate(vals):
    axes[1].text(v, i, f" {v:.4f}", va="center", fontsize=8, color=INK_MUTED)
finish(axes[1], f"MASE vs naive, h={hz} min  (dashed = useless)", "MASE")

# 3. MASE across horizons - one line per model would need 6 hues, so show the
#    two that matter and label them directly
for i, k in enumerate([m for m in ("iTransformer", "DLinear") if m in RESULTS]):
    ys = [RESULTS[k][h]["mase"] for h in CFG.horizons]
    axes[2].plot(CFG.horizons, ys, marker="o", color=CAT[[1, 3][i]], label=k)
axes[2].axhline(1.0, color=CAT[4], lw=1.5, ls="--")
finish(axes[2], "MASE by forecast horizon", "horizon (minutes)", "MASE", legend=True)
plt.tight_layout(); plt.show()

<div style="background: linear-gradient(135deg, #0f3460, #16213e, #1a1a2e); border-left: 4px solid #533483; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e94560; margin: 0 0 10px 0;">🔍 Diagnostics — reading what the model learned</h2>
  <p style="color: #a8dadc; margin: 0 0 8px 0;">The attention map is the direct scientific payoff of choosing an inverted architecture: because attention runs over variates, its averaged matrix <em>is</em> a learned variate-relationship map. It answers the question the whole project is built around — <strong>do the exogenous series actually contribute, and which ones?</strong></p>  <ul style="color: #a8dadc; margin: 0; padding-left: 20px;">
    <li><strong>Attention by variate group</strong> — how much of the target token's attention mass goes to gold, USD, and macro rather than back to BTC's own features.</li>
    <li><strong>Residual autocorrelation</strong> — remaining structure means the model missed signal that a trivially cheaper model could have captured.</li>
    <li><strong>Error by regime</strong> — bucketed by volatility tercile and by year. A model that is excellent in calm 2019 and catastrophic in March 2020 is not a good model, and an aggregate metric hides exactly that.</li>
  </ul>
</div>

In [ ]:
@torch.no_grad()
def attention_map(model: nn.Module, loader: DataLoader, max_batches: int = 20) -> np.ndarray:
    """Average the variate-attention matrix over a sample of the test split."""
    m = unwrap(model)
    if not isinstance(m, iTransformer):
        return np.zeros((N_VARIATES, N_VARIATES))
    m.set_output_attention(True); m.eval()
    acc, n = np.zeros((N_VARIATES, N_VARIATES)), 0
    for i, (xb, _) in enumerate(loader):
        if i >= max_batches:
            break
        _, attns = m(xb.to(DEVICE), return_attention=True)
        for a in attns:
            acc += a.float().mean(dim=1).sum(dim=0).cpu().numpy()
            n += a.shape[0]
    m.set_output_attention(False)
    return acc / max(n, 1)


A = attention_map(MODELS["iTransformer"], test_loader)
grp_of = [FEATURE_GROUP.get(n, "other") for n in FEATURE_NAMES]
GRP_ORDER = [g for g in ("btc_price", "btc_volume", "btc_momentum", "gold",
                         "cross_asset", "dxy", "macro", "temporal") if g in set(grp_of)]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

# attention received by each group, from the target token - magnitude -> one hue, sorted
tgt_row = A[TARGET_IDX] / max(A[TARGET_IDX].sum(), 1e-12)
share = {g: float(sum(tgt_row[i] for i in range(N_VARIATES) if grp_of[i] == g)) for g in GRP_ORDER}
gs = sorted(share, key=share.get, reverse=True)
axes[0].barh(range(len(gs)), [share[g] for g in gs], color=SEQ[3], height=0.62)
axes[0].set_yticks(range(len(gs))); axes[0].set_yticklabels(gs, fontsize=9)
axes[0].invert_yaxis()
for i, g in enumerate(gs):
    axes[0].text(share[g], i, f" {share[g]:.1%}", va="center", fontsize=8, color=INK_MUTED)
finish(axes[0], "Attention mass from the target token", "share of attention")

# residual autocorrelation - Ljung-Box style ACF of the h=1 residual
p1, y1 = PRED["iTransformer"]
resid = cum_h(p1, 1) - cum_h(y1, 1)
rr = resid - resid.mean()
acf = [float((rr[k:] @ rr[:-k]) / (rr @ rr)) for k in range(1, 31)]
conf = 1.96 / math.sqrt(len(rr))
axes[1].bar(range(1, 31), acf, color=[CAT[4] if abs(a) > conf else GRID for a in acf], width=0.7)
axes[1].axhline(conf, color=INK_MUTED, lw=1, ls="--")
axes[1].axhline(-conf, color=INK_MUTED, lw=1, ls="--")
finish(axes[1], "Residual autocorrelation (h=1)", "lag (minutes)", "ACF")

# error by realised-volatility tercile - sequential magnitude, one hue
# Regime is read from realised volatility at the last input minute of each window - the
# trailing 60-minute RV variate when it survived pruning, else |last return| as a proxy.
starts_test = test_loader.dataset.starts
_rv_name = "btc_rv_60" if "btc_rv_60" in FEATURE_NAMES else TARGET_NAME
rv = np.abs(X[starts_test + L - 1, FEATURE_NAMES.index(_rv_name)])
q1, q2 = np.quantile(rv, [1 / 3, 2 / 3])
buckets = np.digitize(rv, [q1, q2])
labels = ["low vol", "mid vol", "high vol"]
mses = [float(((cum_h(p1, hz) - cum_h(y1, hz))[buckets == b] ** 2).mean()) for b in range(3)]
nz = [float((cum_h(y1, hz)[buckets == b] ** 2).mean()) for b in range(3)]
axes[2].bar(np.arange(3) - 0.19, nz, width=0.36, color=GRID, label="naive")
axes[2].bar(np.arange(3) + 0.19, mses, width=0.36, color=CAT[1], label="iTransformer")
axes[2].set_xticks(range(3)); axes[2].set_xticklabels(labels)
finish(axes[2], f"MSE by volatility regime (h={hz})", "", "MSE", legend=True)
plt.tight_layout(); plt.show()

print("attention share from the target token:")
for g in gs:
    print(f"  {g:<14} {share[g]:>7.2%}")
exo_share = sum(share.get(g, 0) for g in ("gold", "cross_asset", "dxy", "macro"))
print(f"\n  exogenous total {exo_share:.2%}  "
      f"-- if this is near zero the exogenous blocks are not earning their place")

<div style="background: linear-gradient(135deg, #0f3460, #16213e, #1a1a2e); border-left: 4px solid #533483; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e94560; margin: 0 0 10px 0;">🧪 §15 · Ablation &amp; walk-forward</h2>
  <p style="color: #a8dadc; margin: 0 0 8px 0;">The ablation table answers the question the project exists to answer: <strong>does each exogenous block improve the forecast, holding everything else fixed?</strong> Blocks are removed by zeroing their variates rather than rebuilding the matrix, so <em>L</em>, <em>N</em>, the splits, the scaler, and the seed are all identical across rows — the only thing that changes is the information.</p>  <p style="color: #a8dadc; margin: 0 0 8px 0;"><strong>Walk-forward is implemented and gated off by default</strong> (<code>CFG.run_walkforward</code>). A single split over a regime-shifting asset is not enough evidence for a final report, but it costs ~5× a single run — enable it for a dedicated multi-session run, not for the first pass. When enabled, it trains on an expanding window and reports mean ± std across folds, which is the headline result; the single split is for development speed.</p>
</div>

In [ ]:
ABLATIONS = {
    "BTC only":        ("btc_price", "btc_volume", "btc_momentum", "temporal"),
    "BTC + Gold":      ("btc_price", "btc_volume", "btc_momentum", "temporal", "gold", "cross_asset"),
    "BTC + USD":       ("btc_price", "btc_volume", "btc_momentum", "temporal", "dxy"),
    "BTC + Macro":     ("btc_price", "btc_volume", "btc_momentum", "temporal", "macro"),
    "BTC + all exog":  tuple(CFG.blocks),
}


def variate_mask(keep_groups) -> np.ndarray:
    m = np.array([1.0 if FEATURE_GROUP.get(n, "") in keep_groups else 0.0
                  for n in FEATURE_NAMES], dtype=np.float32)
    m[TARGET_IDX] = 1.0
    return m


ABL: dict[str, dict] = {}
if CFG.run_ablation:
    print(f"{'configuration':<18}{'variates':>9}{'MSE':>11}{'MASE':>9}{'DirAcc':>9}"
          f"{'dir p':>8}{'MCC':>8}{'NetSharpe':>11}")
    print("-" * 83)
    for label, groups_keep in ABLATIONS.items():
        if CFG.session_budget_hours > 0 and hours_left() <= CFG.reserve_hours:
            print(f"{label:<18}  skipped - session budget exhausted; resume in a new "
                  f"session, the per-tag checkpoints carry over")
            continue
        msk = variate_mask(groups_keep)
        set_seed(CFG.seed, CFG.deterministic)
        tl = make_loader("train", effective_batch(CFG), variate_mask=msk)
        vl = make_loader("val", effective_batch(CFG), shuffle=False,
                         variate_mask=msk, cap=CFG.eval_max_windows)
        te = make_loader("test", effective_batch(CFG), shuffle=False,
                         variate_mask=msk, cap=CFG.eval_max_windows)
        r = train_model(build_model(CFG, N_VARIATES), CFG, tl, vl,
                        tag=f"abl_{label.replace(' ', '_').replace('+', '')}",
                        resume=CFG.resume, verbose=False)
        p_, y_ = predict(r["model"], te, CFG)
        met = evaluate(p_, y_, (hz,), CFG.dir_acc_eps_bp)[hz]
        bt = backtest(p_, y_, hb, 5.0, CFG.fee_per_side, CFG.slippage_per_side,
                      row_step_min=eval_row_step(te))
        ABL[label] = {**met, "net_sharpe": bt.get("sharpe_net", float("nan")),
                      "n_variates": int(msk.sum())}
        print(f"{label:<18}{int(msk.sum()):>9}{met['mse']:>11.6f}{met['mase']:>9.4f}"
              f"{met['dir_acc']:>9.4f}{met['dir_p']:>8.4f}{met['mcc']:>+8.3f}"
              f"{ABL[label]['net_sharpe']:>+11.2f}")
        del r; gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
else:
    print("ablation skipped (CFG.run_ablation = False)")

In [ ]:
WF: list[dict] = []
if CFG.run_walkforward:
    step = CFG.walkforward_months
    t_start, t_end = t_np[0], t_np[-1]
    anchor = np.datetime64(ts(CFG.train_end).replace(tzinfo=None), "us")
    fold = 0
    while True:
        v_lo = anchor + np.timedelta64(GAP, "m")
        v_hi = v_lo + np.timedelta64(step * 30 * 24 * 60, "m")
        s_lo = v_hi + np.timedelta64(GAP, "m")
        s_hi = s_lo + np.timedelta64(step * 30 * 24 * 60, "m")
        if s_hi > t_end:
            break
        f_bounds = {"train": (0, int(np.searchsorted(t_np, anchor))),
                    "val": (int(np.searchsorted(t_np, v_lo)), int(np.searchsorted(t_np, v_hi))),
                    "test": (int(np.searchsorted(t_np, s_lo)), int(np.searchsorted(t_np, s_hi)))}
        saved = dict(SPLITS)
        for nm, (lo, hi) in f_bounds.items():
            stride = CFG.train_stride if nm == "train" else 1
            sel = valid & (starts_all >= lo) & (starts_all + span - 1 <= hi)
            s = starts_all[sel]
            SPLITS[nm] = s[:: stride] if stride > 1 else s
        if min(len(SPLITS[k]) for k in ("train", "val", "test")) < 500:
            SPLITS.update(saved); break
        if CFG.session_budget_hours > 0 and hours_left() <= CFG.reserve_hours:
            print(f"  stopping after {fold} fold(s) - session budget exhausted")
            SPLITS.update(saved); break
        set_seed(CFG.seed + fold, CFG.deterministic)
        r = train_model(build_model(CFG, N_VARIATES), CFG,
                        make_loader("train", effective_batch(CFG)),
                        make_loader("val", effective_batch(CFG), shuffle=False,
                                    cap=CFG.eval_max_windows),
                        tag=f"wf{fold}", resume=CFG.resume, verbose=False)
        te_wf = make_loader("test", effective_batch(CFG), shuffle=False,
                            cap=CFG.eval_max_windows)
        p_, y_ = predict(r["model"], te_wf, CFG)
        met = evaluate(p_, y_, (hz,), CFG.dir_acc_eps_bp)[hz]
        bt = backtest(p_, y_, hb, 5.0, CFG.fee_per_side, CFG.slippage_per_side,
                      row_step_min=eval_row_step(te_wf))
        WF.append({"fold": fold, "test_from": str(s_lo), "test_to": str(s_hi),
                   **met, "net_sharpe": bt.get("sharpe_net", float("nan"))})
        print(f"  fold {fold}  test {str(s_lo)[:10]}..{str(s_hi)[:10]}  "
              f"MASE {met['mase']:.4f}  DirAcc {met['dir_acc']:.4f}  "
              f"netSharpe {WF[-1]['net_sharpe']:+.2f}")
        SPLITS.update(saved)
        anchor = v_hi; fold += 1
        del r; gc.collect()
    if WF:
        for k in ("mase", "dir_acc", "net_sharpe"):
            v = np.array([w[k] for w in WF], dtype=float)
            print(f"  {k:<12} {np.nanmean(v):+.4f} +/- {np.nanstd(v):.4f}  over {len(WF)} folds")
else:
    print("walk-forward skipped (CFG.run_walkforward = False)")
    print("  Enable it for the final report: it is the headline evidence, and a single")
    print("  split over a regime-shifting asset is not enough on its own.")

<div style="background: linear-gradient(90deg, #2d0036, #4a0060); border-left: 4px solid #bf5af2; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e0aaff; margin: 0 0 10px 0;">📦 §16 · Export Bundle</h2>
  <p style="color: #c77dff; margin: 0 0 8px 0;">A checkpoint that cannot be tied back to its exact feature pipeline is worthless. The bundle therefore ships the weights <em>and</em> everything needed to reconstruct the inputs they expect.</p>  <ul style="color: #c77dff; margin: 0; padding-left: 20px;">
    <li><code>model.pt</code> — <code>state_dict</code> only. A pickle of the full module is not a portable or reviewable artifact.</li>
    <li><code>model_scripted.pt</code> and <code>model.onnx</code> (opset 17, dynamic batch axis). <strong>Export fails loudly unless PyTorch, TorchScript, and ONNX Runtime agree within 1e-4</strong> on a fixed batch.</li>
    <li><code>scaler.json</code>, <code>feature_manifest.json</code>, <code>config.yaml</code>, <code>metadata.json</code> — <strong>feature order is part of the contract</strong> and is asserted at inference. A silently reordered feature matrix produces plausible-looking garbage.</li>
    <li><code>inference_example.py</code> — runnable end-to-end, documenting input shape <code>(B, L, N)</code> float32, pre-scaled, and the exact inverse transform from the <code>(B, H, 1)</code> standardised-log-return output back to a price path.</li>
  </ul>
</div>

In [ ]:
class ExportWrapper(nn.Module):
    """Fixed single-input signature for tracing and ONNX. Explicit attention math for portability."""

    def __init__(self, model: iTransformer):
        super().__init__()
        self.model = model
        self.model.set_output_attention(True)      # avoids exporting the fused SDPA kernel
        self.model.eval()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


BUNDLE = WORK_DIR / "models" / CFG.run_id
BUNDLE.mkdir(parents=True, exist_ok=True)
# A deep copy. Exporting must not move the live model off the GPU or leave it with
# output_attention flipped on - both would break any cell re-run after this one.
core = copy.deepcopy(unwrap(MODELS["iTransformer"])).to("cpu").eval()

torch.save(core.state_dict(), BUNDLE / "model.pt")
wrapper = ExportWrapper(core).eval()
example = torch.randn(2, CFG.seq_len, N_VARIATES)

parity = {}
with torch.no_grad():
    ref = wrapper(example).numpy()

    scripted = torch.jit.trace(wrapper, example, strict=False)
    torch.jit.save(scripted, str(BUNDLE / "model_scripted.pt"))
    parity["torchscript"] = float(np.abs(scripted(example).numpy() - ref).max())

    onnx_ok = False
    try:
        torch.onnx.export(
            wrapper, (example,), str(BUNDLE / "model.onnx"),
            input_names=["input"], output_names=["output"], opset_version=17,
            dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
        )
        import onnxruntime as ort
        sess = ort.InferenceSession(str(BUNDLE / "model.onnx"),
                                    providers=["CPUExecutionProvider"])
        got = sess.run(None, {"input": example.numpy()})[0]
        parity["onnx"] = float(np.abs(got - ref).max())
        onnx_ok = True
    except Exception as e:
        parity["onnx"] = float("nan")
        print(f"  ONNX export/verify unavailable: {type(e).__name__}: {e}")

for k, v in parity.items():
    status = "OK" if (np.isfinite(v) and v < 1e-4) else ("SKIPPED" if not np.isfinite(v) else "FAIL")
    print(f"  parity {k:<12} max|delta| = {v:.3e}   {status}")
assert parity["torchscript"] < 1e-4, "TorchScript parity failed - do not ship this bundle"
if onnx_ok:
    assert parity["onnx"] < 1e-4, "ONNX parity failed - do not ship this bundle"

In [ ]:
MANIFEST = {
    "feature_order": FEATURE_NAMES,
    "n_variates": N_VARIATES,
    "target_index": TARGET_IDX,
    "target_name": TARGET_NAME,
    "seq_len": CFG.seq_len,
    "pred_len": CFG.pred_len,
    "groups": {n: FEATURE_GROUP.get(n, "other") for n in FEATURE_NAMES},
    "fracdiff_d": FRAC_D,
    "gold_utc_offset_hours": GOLD_OFFSET_H,
    "release_lag_months_days": {k: list(v) for k, v in RELEASE_LAG.items()},
    "dxy_lag_days": DXY_LAG_DAYS,
    "dropped_columns": sorted(MACRO_DROP),
    "macro_pca_components": int(K),
    "winsor_quantile": CFG.winsor_q,
    "collinearity_threshold": CFG.collinear_thresh,
}
(BUNDLE / "feature_manifest.json").write_text(json.dumps(MANIFEST, indent=2))
(BUNDLE / "scaler.json").write_text(json.dumps(SCALER, indent=2))
(BUNDLE / "config.json").write_text(json.dumps(asdict(CFG), indent=2, default=str))

manifest_hash = hashlib.sha256(
    json.dumps(MANIFEST, sort_keys=True).encode()).hexdigest()
METADATA = {
    "run_id": CFG.run_id,
    "created_utc": datetime.now(UTC).isoformat(),
    "torch_version": torch.__version__,
    "python_version": sys.version.split()[0],
    "device": str(DEVICE), "n_gpu": N_GPU, "amp_dtype": str(AMP_DTYPE),
    "feature_manifest_sha256": manifest_hash,
    "gates": GATES, "all_gates_pass": ALL_GATES_PASS,
    "parity": parity,
    "metrics_test": {str(h): RESULTS["iTransformer"][h] for h in CFG.horizons},
    "baselines_test": {k: {str(h): v[h] for h in CFG.horizons}
                       for k, v in RESULTS.items() if k != "iTransformer"},
    "ablation": ABL, "walkforward": WF,
    "cost_assumptions": {"fee_per_side": CFG.fee_per_side,
                         "slippage_per_side": CFG.slippage_per_side,
                         "backtest_horizon_min": hb,
                         "eval_row_step_min": int(EVAL_STEP_MIN)},
    "splits": {k: {"windows": int(len(v)),
                   "from": str(t_np[v[0]]) if len(v) else None,
                   "to": str(t_np[v[-1] + span - 1]) if len(v) else None}
               for k, v in SPLITS.items()},
    "seeds_run": 1, "seeds_recommended": CFG.n_seeds_report,
}
(BUNDLE / "metadata.json").write_text(json.dumps(METADATA, indent=2, default=str))

INFERENCE_EXAMPLE = """# Runnable inference example for the exported iTransformer bundle.
#
# INPUT CONTRACT
#     shape  (B, L, N) float32
#     values already scaled: x = (winsorise(raw) - scaler.mean) / scaler.std
#     order  exactly feature_manifest.json["feature_order"] - assert it, never assume it
#     L, N   from feature_manifest.json (seq_len, n_variates)
#
# OUTPUT CONTRACT
#     shape  (B, H, 1) float32, in STANDARDISED log-return space
#     to raw one-minute log returns:  r = out[..., 0] * std[target_index] + mean[target_index]
#     to an h-minute cumulative return: r[:, :h].sum(axis=1)
#     to a price path: price_{t+k} = close_t * exp(cumsum(r)[k])
#
# STALENESS POLICY
#     gold      may be forward-filled up to 3 days (a normal weekend). Beyond that, refuse.
#     usd index may be forward-filled up to 5 business days.
#     macro     may be forward-filled up to 45 days past its release date.
#     Exceeding any of these means the staleness features leave their training support,
#     and the prediction should be refused rather than served.

import json
from pathlib import Path

import numpy as np
import torch

BUNDLE = Path(__file__).parent
manifest = json.loads((BUNDLE / "feature_manifest.json").read_text())
scaler = json.loads((BUNDLE / "scaler.json").read_text())

L, N = manifest["seq_len"], manifest["n_variates"]
H = manifest["pred_len"]
ti = manifest["target_index"]
mean = np.asarray(scaler["mean"], dtype=np.float32)
std = np.asarray(scaler["std"], dtype=np.float32)


def scale(raw: np.ndarray, feature_order: list) -> np.ndarray:
    # raw: (B, L, N) unscaled, columns in `feature_order`.
    assert list(feature_order) == manifest["feature_order"], (
        "feature order mismatch - the model will produce plausible-looking garbage"
    )
    lo = np.asarray([v if v is not None else -np.inf for v in scaler["winsor_lo"]], np.float32)
    hi = np.asarray([v if v is not None else np.inf for v in scaler["winsor_hi"]], np.float32)
    return ((np.clip(raw, lo, hi) - mean) / std).astype(np.float32)


model = torch.jit.load(str(BUNDLE / "model_scripted.pt"))
model.eval()

x = torch.randn(1, L, N)                      # substitute real, scaled features here
with torch.no_grad():
    out = model(x)                            # (1, H, 1) standardised log returns

r = out[..., 0].numpy() * std[ti] + mean[ti]  # raw one-minute log returns
print("output", tuple(out.shape))
for h in (1, 5, 15, 30, 60):
    if h <= H:
        print(f"  cumulative {h:>3}-min log return: {r[0, :h].sum():+.6f}")

close_t = 60_000.0
print(f"  implied price path from {close_t:,.0f}: "
      f"{(close_t * np.exp(np.cumsum(r[0])))[[0, min(H, 60) - 1]].round(2).tolist()}")
"""
(BUNDLE / "inference_example.py").write_text(INFERENCE_EXAMPLE)

print(f"bundle -> {BUNDLE}")
for f in sorted(BUNDLE.iterdir()):
    print(f"  {f.name:<26} {f.stat().st_size / 1024:>10,.1f} KB")
print(f"\nfeature manifest sha256: {manifest_hash[:32]}...")

<div style="background: linear-gradient(90deg, #2d0036, #4a0060); border-left: 4px solid #bf5af2; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e0aaff; margin: 0 0 10px 0;">📋 §17 · Results &amp; Limitations</h2>
  <p style="color: #c77dff; margin: 0 0 8px 0;">A bare number is not a result. Every figure below is reported with its baseline, its split, its seed count and its cost assumptions.</p>
</div>

In [ ]:
print("=" * 78)
print(f"RUN {CFG.run_id}   profile={CFG.profile}   seed={CFG.seed}   "
      f"gates={'PASS' if ALL_GATES_PASS else 'FAIL'}")
print("=" * 78)
print(f"\nData        {t_np[0]} -> {t_np[-1]}   {T:,} minutes   {N_VARIATES} variates")
print(f"Splits      " + "  ".join(f"{k}={len(v):,}" for k, v in SPLITS.items())
      + f"   gap={GAP:,} min")
print(f"Costs       {CFG.fee_per_side * 1e4:.0f} bp fee + {CFG.slippage_per_side * 1e4:.0f} bp "
      f"slippage per side, h={hb} min, non-overlapping")
print(f"Seeds       1 run  (CLAUDE.md asks for {CFG.n_seeds_report}; a single seed is an anecdote)")

print(f"\nTEST METRICS BY HORIZON  -  iTransformer")
print(f"{'h':>4}{'MSE':>12}{'MASE':>9}{'R2':>10}{'DirAcc':>9}{'dir p':>8}"
      f"{'MCC':>8}{'Top10%':>9}{'DM p':>9}")
for h in CFG.horizons:
    r_ = RESULTS["iTransformer"][h]
    print(f"{h:>4}{r_['mse']:>12.6f}{r_['mase']:>9.4f}{r_['r2']:>+10.4f}"
          f"{r_['dir_acc']:>9.4f}{r_['dir_p']:>8.4f}{r_['mcc']:>+8.3f}"
          f"{r_['dir_acc_top']:>9.4f}{r_.get('dm_p', NAN):>9.4f}")

if ABL:
    print(f"\nABLATION  (h={hz})")
    print(f"{'configuration':<18}{'MSE':>11}{'MASE':>9}{'DirAcc':>9}{'dir p':>8}{'NetSharpe':>11}")
    for k, v in ABL.items():
        print(f"{k:<18}{v['mse']:>11.6f}{v['mase']:>9.4f}{v['dir_acc']:>9.4f}"
              f"{v['dir_p']:>8.4f}{v['net_sharpe']:>+11.2f}")

verdict = []
_r = RESULTS["iTransformer"][hz]
verdict.append(("beats naive (MASE < 1)", _r["mase"] < 1.0))
# a significant DM only counts in the model's favour - "significantly worse" is not a tick
verdict.append(("beats naive significantly (DM p < 0.05)",
                _r["mase"] < 1.0 and _r.get("dm_p", 1.0) < 0.05))
verdict.append(("beats DLinear", _r["mse"] < RESULTS.get("DLinear", {}).get(hz, {}).get("mse", np.inf)))
verdict.append(("calls direction better than a coin flip (DirAcc > 0.5, p < 0.05)",
                bool(np.isfinite(_r["dir_acc"]) and _r["dir_acc"] > 0.5 and _r["dir_p"] < 0.05)))
verdict.append(("all sanity gates pass", ALL_GATES_PASS))
verdict.append(("walk-forward completed", bool(WF)))
verdict.append((f"{CFG.n_seeds_report} seeds run", False))
print("\nDEFINITION OF DONE")
for label, ok in verdict:
    print(f"  [{'x' if ok else ' '}] {label}")

<div style="background: linear-gradient(90deg, #2d0036, #4a0060); border-left: 4px solid #bf5af2; border-radius: 8px; padding: 18px 24px;">
  <h2 style="color: #e0aaff; margin: 0 0 10px 0;">⚠️ Known limitations — stated plainly</h2>
  <ul style="color: #c77dff; margin: 0; padding-left: 20px;">
    <li><strong>Revised data, not real-time vintages.</strong> The macro file contains revised values. Even with a correct release-lag table, revisions leak a small amount of future information. Fixing this properly requires ALFRED vintage data. This is a property of the dataset and cannot be engineered away here.</li>
    <li><strong>The USD index series identity is unresolved.</strong> Measured: starts 109.64 on 2018-01-02, ends 118.88 on 2026-05-29, range 106.5–130.0. That does not match FRED <code>DTWEXBGS</code> (~114–115 in Jan-2018) at its published base, so the file is rebased or a different vintage. Against the monthly <code>Real_Broad_Dollar_Index</code> it shows level correlation 0.973 and month-over-month log-change correlation 0.989 with a ratio drifting 1.062 ± 0.013 — consistent with a nominal/real pair rather than a constant rebasing. The monthly column is dropped as redundant; the exact FRED series ID and base period still need confirming against the source.</li>
    <li><strong>Release dates are approximated</strong> from typical publication calendars, not scraped from the actual 2018–2026 BLS/BEA/Federal Reserve schedules. Every lag is rounded up, so the error is conservative, but it is an approximation.</li>
    <li><strong>One seed.</strong> CLAUDE.md asks for five. A single seed is an anecdote, not a measurement — re-run with <code>CFG.seed ∈ {42, 1, 7, 13, 2024}</code> and report mean ± std before treating any number here as final.</li>
    <li><strong>The backtest is a strategy sketch, not a trading system.</strong> It assumes fills at the close, no market impact, no funding, no exchange outages, and a constant fee tier. Real slippage at 1-minute frequency in volatile regimes is worse than the 2 bp assumed.</li>
    <li><strong>No order-flow features exist</strong> because the Binance export carries base-asset volume only — no quote volume, no trade count, no taker-buy split. Microstructure signal is therefore approximated from OHLC alone.</li>
    <li><strong>The test split was opened once.</strong> If any result above prompts a change to features, architecture, or hyperparameters, everything after that change must be re-validated and the test re-run counts as a new experiment — which must be said explicitly in the write-up.</li>
  </ul>
</div>

<div style="background: linear-gradient(135deg, #0f0c29, #302b63, #24243e); border-radius: 16px; padding: 28px 34px;">
  <h2 style="color: #e0aaff; margin: 0 0 12px 0;">✅ Next steps — one stage per Kaggle session</h2>
  <ol style="color: #c8b6ff; margin: 0; padding-left: 22px; line-height: 1.7;">
    <li><strong>Session 0 — smoke.</strong> <code>PROFILE = 'smoke'</code>. Confirm every sanity gate prints <strong>PASS</strong>. ~15 min, and it barely dents the 30 h weekly GPU quota.</li>
    <li><strong>Session 1 — the model.</strong> <code>PROFILE = 'full'</code> with <code>run_baselines = False</code> and <code>run_ablation = False</code>. This is the shipped artifact; give it the whole session.</li>
    <li><strong>Session 2 — baselines.</strong> Attach session 1's output, set <code>KAGGLE_RESUME_DIR</code>, then <code>run_baselines = True</code>. The iTransformer resumes from its best checkpoint and the baselines train against it.</li>
    <li><strong>Session 3 — ablation.</strong> Same resume, <code>run_ablation = True</code>. This is the table that answers whether gold, USD and macro earn their place.</li>
    <li><strong>Session 4+ — walk-forward.</strong> <code>run_walkforward = True</code>. It is the headline evidence; a single split over a regime-shifting asset is not enough on its own.</li>
    <li><strong>Then seeds.</strong> Repeat the final configuration across <code>seed ∈ {42, 1, 7, 13, 2024}</code> and report mean ± std. One seed is an anecdote.</li>
    <li>Only then treat the numbers as final — and if anything looks too good, hunt for the leak before celebrating.</li>
  </ol>
</div>